In [1]:
from sympy import symbols, pprint, collect, factor, expand, preorder_traversal, Symbol, Mul, Add
from sympy import *

import numpy as np
import re
from qiskit import QuantumCircuit, QuantumRegister, Aer, execute
from qiskit.opflow import PauliOp, SummedOp, I
from qiskit.quantum_info import Pauli
from qiskit.circuit import ParameterVector
from qiskit.algorithms import QAOA
from qiskit.algorithms.optimizers import COBYLA
x = []
name = "x"
for i in range(0,21):
    v = symbols(name+str(i))
    x.append(v)


In [2]:
def substitute_with_global_binary_symbols(expr, bit_length, base_name="b"):
    """
    Replace each variable in expr with a symbolic binary representation.
    x_i → b_{3i+1} + 2*b_{3i+2} + 4*b_{3i+3}, etc.
    """
    vars = sorted(expr.free_symbols, key=lambda x: str(x))  # consistent ordering
    subs = {}

    for idx, v in enumerate(vars):
        bin_vars = [symbols(f"{base_name}_{bit_index}") for bit_index in range(idx * bit_length + 1, (idx + 1) * bit_length + 1)]
        binary_expr = sum((2**i) * bin_vars[i] for i in range(bit_length))
        subs[v] = binary_expr

    return expand(expr.subs(subs))


def remove_variable_exponents(expr):
    """
    Replace all instances of x**n (n > 1) with x in a symbolic expression.
    """
    replacements = {}

    for subexpr in preorder_traversal(expr):
        if subexpr.is_Pow:
            base, exp = subexpr.args
            if isinstance(base, Symbol) and exp.is_Number and exp > 1:
                replacements[subexpr] = base

    return expr.xreplace(replacements)

def substitute_with_spin_variables(expr, prefix_original='b', prefix_new='z'):
    """
    Substitute each variable x_i in expr with (1 - z_i)/2, where z_i is a new symbolic variable.
    """
    subs = {}

    for var in expr.free_symbols:
        name = str(var)
        if name.startswith(prefix_original):
            index = name[len(prefix_original):]
            z = symbols(f"{prefix_new}{index}")
            subs[var] = (1 - z) / 2

    return simplify(expand(expr.subs(subs)))

In [17]:
def parse_hamiltonian_expr(expr):
    terms = sympify(expr, evaluate=False).as_ordered_terms()
    parsed = []

    for term in terms:
        coeff = term
        indices = []

        if isinstance(term, Symbol):
            coeff = 1
            indices = [int(str(term).split("_")[1]) - 1]
        else:
            factors = term.as_ordered_factors()
            coeff = 1
            indices = []
            for factor in factors:
                if factor.is_Number:
                    coeff *= float(factor)
                elif isinstance(factor, Symbol) and str(factor).startswith("z_"):
                    qubit_idx = int(str(factor).split("_")[1]) - 1
                    indices.append(qubit_idx)
                else:
                    raise ValueError(f"Unsupported factor: {factor}")

        # If no qubit indices, assign [0] instead of []
        if len(indices) == 0:
            indices = [0]

        parsed.append((int(coeff), sorted(indices)))

    return parsed

In [3]:
from sympy import IndexedBase, Symbol
import re

def convert_symbols_to_indexed(expr, prefix="z"):
    """
    Converts symbols like z_1, z_2, ... in a SymPy expression to indexed form z[1], z[2], ...

    Args:
        expr (sympy expression): The SymPy expression with symbols like z_1.
        prefix (str): The variable name prefix (e.g., 'z').

    Returns:
        sympy expression: The expression with replaced indexed variables.
    """
    indexed_array = IndexedBase(prefix)
    replacements = {}

    for sym in expr.free_symbols:
        if isinstance(sym, Symbol):
            match = re.fullmatch(rf"{prefix}_(\d+)", sym.name)
            if match:
                index = int(match.group(1))
                replacements[sym] = indexed_array[index]

    return expr.subs(replacements)



In [4]:
def bitstring_cost(bitstring, hamiltonian):
    """
    Calculate the cost (energy) of a bitstring using the Hamiltonian.
    bitstring: str, e.g. '010011001' (Qiskit bit order, rightmost is qubit 0)
    """
    z = np.array([1 if b == '0' else -1 for b in bitstring[::-1]])  # reverse to qubit order
    cost = 0
    for term in hamiltonian.oplist:
        coeff = term.coeff.real
        label = term.primitive.to_label()
        indices = [i for i, p in enumerate(label) if p == 'Z']
        product = np.prod(z[indices]) if indices else 1
        cost += coeff * product
    return cost

def find_best_bitstrings(circuit, hamiltonian, shots=2048, top_k=5):
    backend = Aer.get_backend('qasm_simulator')
    job = execute(circuit, backend=backend, shots=shots)
    counts = job.result().get_counts()

    scored_bitstrings = []
    for bitstring, count in counts.items():
        cost = bitstring_cost(bitstring, hamiltonian)
        scored_bitstrings.append((bitstring, cost, count))

    # Sort by cost (lowest first), then by count (highest first)
    scored_bitstrings.sort(key=lambda x: (x[1], -x[2]))

    print(f"Top {top_k} bitstrings:")
    for bs, cost, count in scored_bitstrings[:top_k]:
        print(f"Bitstring: {bs}, Cost: {cost:.4f}, Count: {count}")

    return scored_bitstrings[:top_k]


In [47]:
def group_and_convert(bitstring, group_size):
    Decimals = []
    for i in range(len(bitstring)):
        reversed_bits = bitstring[i][::-1]
        #print(reversed_bits)
        chunks = [reversed_bits[i:i+group_size][::1] for i in range(0, len(reversed_bits), group_size)]
        #print(chunks)
        decimals = [int(chunk, 2) for chunk in chunks]
        Decimals.append(decimals)
    return(Decimals)

def evaluate_Diophantine(entrada,poly):
    Cost = []
    for i in range(len(entrada)):
        Cost.append(poly(entrada[i][0],entrada[i][1],entrada[i][2]))
    return(Cost)
    

In [5]:
# Build Hamiltonian from your terms (extend the list as needed)
def build_cost_hamiltonian_1(num_qubits,terms):

    zero_op = PauliOp(Pauli('I' * num_qubits)) * 0
    hamiltonian = SummedOp([zero_op])

    for coeff, qubits_idx in terms:
        pauli_label = ['I'] * num_qubits
        for i in qubits_idx:
            pauli_label[i] = 'Z'
        pauli_str = ''.join(pauli_label)
        hamiltonian += PauliOp(Pauli(pauli_str)) * coeff

    return hamiltonian


# Multi-controlled RZ helper
def apply_multi_controlled_rz(qc, angle, controls, target):
    qc.h(target)
    from qiskit.circuit.library import MCXGate
    mcx = MCXGate(len(controls))
    qc.append(mcx, controls + [target])
    qc.rz(angle, target)
    qc.append(mcx, controls + [target])
    qc.h(target)

# Apply cost unitary e^{-i gamma H}
def apply_cost_unitary(qc, hamiltonian, gamma):
    for term in hamiltonian.oplist:
        coeff = term.coeff.real
        label = term.primitive.to_label()
        qubits_in_term = [i for i, p in enumerate(label) if p == 'Z']

        if len(qubits_in_term) == 0:
            continue
        if len(qubits_in_term) == 1:
            qc.rz(2 * gamma * coeff, qubits_in_term[0])
            continue

        target = qubits_in_term[-1]
        controls = qubits_in_term[:-1]
        apply_multi_controlled_rz(qc, 2 * gamma * coeff, controls, target)

# Apply mixer unitary (RX rotations)
def apply_mixer_unitary(qc, beta):
    for q in range(qc.num_qubits):
        qc.rx(2 * beta, q)

# Build full QAOA circuit with p layers
def build_qaoa_circuit(num_qubits, hamiltonian, p, gammas, betas):
    qr = QuantumRegister(num_qubits)
    qc = QuantumCircuit(qr)

    # Initialize to uniform superposition
    qc.h(range(num_qubits))

    for layer in range(p):
        apply_cost_unitary(qc, hamiltonian, gammas[layer])
        apply_mixer_unitary(qc, betas[layer])

    qc.measure_all()
    return qc

# Expectation evaluation function
def expectation_from_counts(counts, hamiltonian):
    """
    Compute expectation <H> from measurement counts.
    """
    expect = 0
    shots = sum(counts.values())
    for bitstring, count in counts.items():
        z = np.array([1 if b=='0' else -1 for b in bitstring[::-1]])  # Qiskit reverses bit order
        val = 0
        for term in hamiltonian.oplist:
            coeff = term.coeff.real
            label = term.primitive.to_label()
            indices = [i for i, p in enumerate(label) if p == 'Z']
            product = np.prod(z[indices]) if indices else 1
            val += coeff * product
        expect += val * count / shots
    return expect

# Main QAOA optimization
def qaoa(num_qubits, hamiltonian, p=1, shots=1024):

    # Parameters
    gammas = ParameterVector('gamma', p)
    betas = ParameterVector('beta', p)

    backend = Aer.get_backend('qasm_simulator')

    def objective(x):
        # Build circuit with current parameters
        qc = build_qaoa_circuit(num_qubits, hamiltonian, p, x[:p], x[p:])
        job = execute(qc, backend=backend, shots=shots)
        counts = job.result().get_counts()
        return expectation_from_counts(counts, hamiltonian)

    # Initial guess
    x0 = np.array([0.1]*(2*p))

    optimizer = COBYLA(maxiter=100)
    opt_result = optimizer.optimize(num_vars=2*p, objective_function=objective, initial_point=x0)

    print(f"Optimal parameters (gammas, betas): {opt_result[0]}")
    print(f"Minimum expectation value: {opt_result[1]}")

    # Final circuit with optimized params
    final_qc = build_qaoa_circuit(num_qubits, hamiltonian, p, opt_result[0][:p], opt_result[0][p:])
    return final_qc, opt_result

In [6]:
def bitstring_to_pm1(LIST,H):
    Z = []
    for i in range(len(LIST)):
        bitstring = LIST[i][0]
        reversed_bits = bitstring[::-1]
        z = []
        z.append(0)
        for j, bit in enumerate(reversed_bits):
            #print(bit)
            z.append(1 - 2*int(bit))
        Z.append(z)
    #print(Z)

#def Evaluations(Z,H):
    Cost = []
    for i in range(len(Z)):
        Cost.append(f"Bitstring: {LIST[0]}, Evaluated cost: {H(Z[i])}")
    return(Cost)


# Catalan's $x^a-y^b=1$ for $a, b > 1, x, y > 0$ Case $a=2, b=3$ has a solution which is $x = 3, y= 2.$

In [7]:
def D_Cat(x,y):
    return(x**2-y**3-1)

For numbers between 0 and 15, which has binary length of 4:

In [8]:
paso1_D_Cat_less_15 = substitute_with_global_binary_symbols(D_Cat(x[1],x[2])**2, 4, base_name="b")
paso2_D_Cat_less_15 = remove_variable_exponents(paso1_D_Cat_less_15)
paso3_D_Cat_less_15 = substitute_with_spin_variables(paso2_D_Cat_less_15)
#print(paso3_D_Cat_less_15)

In [9]:
def evaluate_hamiltonian_less_15(z):
    return(96*z[1]*z[2]*z[3]*z[4] - 180*z[1]*z[2]*z[3] - 360*z[1]*z[2]*z[4] + 12*z[1]*z[2]*z[5]*z[6]*z[7] + 24*z[1]*z[2]*z[5]*z[6]*z[8] - 45*z[1]*z[2]*z[5]*z[6] + 48*z[1]*z[2]*z[5]*z[7]*z[8] - 90*z[1]*z[2]*z[5]*z[7] - 180*z[1]*z[2]*z[5]*z[8] + 232*z[1]*z[2]*z[5] + 96*z[1]*z[2]*z[6]*z[7]*z[8] - 180*z[1]*z[2]*z[6]*z[7] - 360*z[1]*z[2]*z[6]*z[8] + 461*z[1]*z[2]*z[6] - 720*z[1]*z[2]*z[7]*z[8] + 898*z[1]*z[2]*z[7] + 1604*z[1]*z[2]*z[8] - 1342*z[1]*z[2] - 720*z[1]*z[3]*z[4] + 24*z[1]*z[3]*z[5]*z[6]*z[7] + 48*z[1]*z[3]*z[5]*z[6]*z[8] - 90*z[1]*z[3]*z[5]*z[6] + 96*z[1]*z[3]*z[5]*z[7]*z[8] - 180*z[1]*z[3]*z[5]*z[7] - 360*z[1]*z[3]*z[5]*z[8] + 464*z[1]*z[3]*z[5] + 192*z[1]*z[3]*z[6]*z[7]*z[8] - 360*z[1]*z[3]*z[6]*z[7] - 720*z[1]*z[3]*z[6]*z[8] + 922*z[1]*z[3]*z[6] - 1440*z[1]*z[3]*z[7]*z[8] + 1796*z[1]*z[3]*z[7] + 3208*z[1]*z[3]*z[8] - 2708*z[1]*z[3] + 48*z[1]*z[4]*z[5]*z[6]*z[7] + 96*z[1]*z[4]*z[5]*z[6]*z[8] - 180*z[1]*z[4]*z[5]*z[6] + 192*z[1]*z[4]*z[5]*z[7]*z[8] - 360*z[1]*z[4]*z[5]*z[7] - 720*z[1]*z[4]*z[5]*z[8] + 928*z[1]*z[4]*z[5] + 384*z[1]*z[4]*z[6]*z[7]*z[8] - 720*z[1]*z[4]*z[6]*z[7] - 1440*z[1]*z[4]*z[6]*z[8] + 1844*z[1]*z[4]*z[6] - 2880*z[1]*z[4]*z[7]*z[8] + 3592*z[1]*z[4]*z[7] + 6416*z[1]*z[4]*z[8] - 5608*z[1]*z[4] - 90*z[1]*z[5]*z[6]*z[7] - 180*z[1]*z[5]*z[6]*z[8] + 675*z[1]*z[5]*z[6]/2 - 360*z[1]*z[5]*z[7]*z[8] + 675*z[1]*z[5]*z[7] + 1350*z[1]*z[5]*z[8] - 1740*z[1]*z[5] - 720*z[1]*z[6]*z[7]*z[8] + 1350*z[1]*z[6]*z[7] + 2700*z[1]*z[6]*z[8] - 6915*z[1]*z[6]/2 + 5400*z[1]*z[7]*z[8] - 6735*z[1]*z[7] - 12030*z[1]*z[8] + 23445*z[1]/2 - 1440*z[2]*z[3]*z[4] + 48*z[2]*z[3]*z[5]*z[6]*z[7] + 96*z[2]*z[3]*z[5]*z[6]*z[8] - 180*z[2]*z[3]*z[5]*z[6] + 192*z[2]*z[3]*z[5]*z[7]*z[8] - 360*z[2]*z[3]*z[5]*z[7] - 720*z[2]*z[3]*z[5]*z[8] + 928*z[2]*z[3]*z[5] + 384*z[2]*z[3]*z[6]*z[7]*z[8] - 720*z[2]*z[3]*z[6]*z[7] - 1440*z[2]*z[3]*z[6]*z[8] + 1844*z[2]*z[3]*z[6] - 2880*z[2]*z[3]*z[7]*z[8] + 3592*z[2]*z[3]*z[7] + 6416*z[2]*z[3]*z[8] - 5428*z[2]*z[3] + 96*z[2]*z[4]*z[5]*z[6]*z[7] + 192*z[2]*z[4]*z[5]*z[6]*z[8] - 360*z[2]*z[4]*z[5]*z[6] + 384*z[2]*z[4]*z[5]*z[7]*z[8] - 720*z[2]*z[4]*z[5]*z[7] - 1440*z[2]*z[4]*z[5]*z[8] + 1856*z[2]*z[4]*z[5] + 768*z[2]*z[4]*z[6]*z[7]*z[8] - 1440*z[2]*z[4]*z[6]*z[7] - 2880*z[2]*z[4]*z[6]*z[8] + 3688*z[2]*z[4]*z[6] - 5760*z[2]*z[4]*z[7]*z[8] + 7184*z[2]*z[4]*z[7] + 12832*z[2]*z[4]*z[8] - 11240*z[2]*z[4] - 180*z[2]*z[5]*z[6]*z[7] - 360*z[2]*z[5]*z[6]*z[8] + 675*z[2]*z[5]*z[6] - 720*z[2]*z[5]*z[7]*z[8] + 1350*z[2]*z[5]*z[7] + 2700*z[2]*z[5]*z[8] - 3480*z[2]*z[5] - 1440*z[2]*z[6]*z[7]*z[8] + 2700*z[2]*z[6]*z[7] + 5400*z[2]*z[6]*z[8] - 6915*z[2]*z[6] + 10800*z[2]*z[7]*z[8] - 13470*z[2]*z[7] - 24060*z[2]*z[8] + 23490*z[2] + 192*z[3]*z[4]*z[5]*z[6]*z[7] + 384*z[3]*z[4]*z[5]*z[6]*z[8] - 720*z[3]*z[4]*z[5]*z[6] + 768*z[3]*z[4]*z[5]*z[7]*z[8] - 1440*z[3]*z[4]*z[5]*z[7] - 2880*z[3]*z[4]*z[5]*z[8] + 3712*z[3]*z[4]*z[5] + 1536*z[3]*z[4]*z[6]*z[7]*z[8] - 2880*z[3]*z[4]*z[6]*z[7] - 5760*z[3]*z[4]*z[6]*z[8] + 7376*z[3]*z[4]*z[6] - 11520*z[3]*z[4]*z[7]*z[8] + 14368*z[3]*z[4]*z[7] + 25664*z[3]*z[4]*z[8] - 22672*z[3]*z[4] - 360*z[3]*z[5]*z[6]*z[7] - 720*z[3]*z[5]*z[6]*z[8] + 1350*z[3]*z[5]*z[6] - 1440*z[3]*z[5]*z[7]*z[8] + 2700*z[3]*z[5]*z[7] + 5400*z[3]*z[5]*z[8] - 6960*z[3]*z[5] - 2880*z[3]*z[6]*z[7]*z[8] + 5400*z[3]*z[6]*z[7] + 10800*z[3]*z[6]*z[8] - 13830*z[3]*z[6] + 21600*z[3]*z[7]*z[8] - 26940*z[3]*z[7] - 48120*z[3]*z[8] + 47340*z[3] - 720*z[4]*z[5]*z[6]*z[7] - 1440*z[4]*z[5]*z[6]*z[8] + 2700*z[4]*z[5]*z[6] - 2880*z[4]*z[5]*z[7]*z[8] + 5400*z[4]*z[5]*z[7] + 10800*z[4]*z[5]*z[8] - 13920*z[4]*z[5] - 5760*z[4]*z[6]*z[7]*z[8] + 10800*z[4]*z[6]*z[7] + 21600*z[4]*z[6]*z[8] - 27660*z[4]*z[6] + 43200*z[4]*z[7]*z[8] - 53880*z[4]*z[7] - 96240*z[4]*z[8] + 97560*z[4] + 91200*z[5]*z[6]*z[7]*z[8] - 97632*z[5]*z[6]*z[7] - 152064*z[5]*z[6]*z[8] + 315947*z[5]*z[6]/2 - 282528*z[5]*z[7]*z[8] + 289547*z[5]*z[7] + 402454*z[5]*z[8] - 817749*z[5]/2 - 554256*z[6]*z[7]*z[8] + 565804*z[6]*z[7] + 781208*z[6]*z[8] - 1583307*z[6]/2 + 1381456*z[7]*z[8] - 1390743*z[7] - 1759374*z[8] + 1778473)

In [18]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 8  # 
p = 1  # QAOA depth

hamiltonian_less_15 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_Cat_less_15))
final_circuit_less_15, result_less_15 = qaoa(num_qubits, hamiltonian_less_15, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [1.33093956 1.0040299 ]
Minimum expectation value: -516481.298828125


In [19]:
top_solutions_less_15 = find_best_bitstrings(final_circuit_less_15, hamiltonian_less_15)

Top 5 bitstrings:
Bitstring: 00010001, Cost: -3556945.0000, Count: 21
Bitstring: 00000001, Cost: -3556945.0000, Count: 9
Bitstring: 00110101, Cost: -3556937.0000, Count: 7
Bitstring: 01011011, Cost: -3556921.0000, Count: 8
Bitstring: 00010011, Cost: -3556897.0000, Count: 13


In [20]:
bitstring_to_pm1(top_solutions_less_15,evaluate_hamiltonian_less_15)

["Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 1.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 0.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 9.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 25.0",
 "Bitstring: ('00010001', -3556945.0, 21), Evaluated cost: 49.0"]

# Diophantine equation $x^2+y^2=3$ which has no integer solutions 

In [12]:
def D_equation(x,y):
    return(x**2+y**2-3)

For numbers between 0 and 15, which has binary length of 4:

In [13]:
paso1_D_equation_less_15_2 = substitute_with_global_binary_symbols(D_equation(x[1],x[2])**2, 4, base_name="b")
paso2_D_equation_less_15_2 = remove_variable_exponents(paso1_D_equation_less_15_2)
paso3_D_equation_less_15_2 = substitute_with_spin_variables(paso2_D_equation_less_15_2)

In [14]:
def evaluate_hamiltonian_less_15_2(z):
    return(96*z[1]*z[2]*z[3]*z[4] - 180*z[1]*z[2]*z[3] - 360*z[1]*z[2]*z[4] + 2*z[1]*z[2]*z[5]*z[6] + 4*z[1]*z[2]*z[5]*z[7] + 8*z[1]*z[2]*z[5]*z[8] - 15*z[1]*z[2]*z[5] + 8*z[1]*z[2]*z[6]*z[7] + 16*z[1]*z[2]*z[6]*z[8] - 30*z[1]*z[2]*z[6] + 32*z[1]*z[2]*z[7]*z[8] - 60*z[1]*z[2]*z[7] - 120*z[1]*z[2]*z[8] + 609*z[1]*z[2] - 720*z[1]*z[3]*z[4] + 4*z[1]*z[3]*z[5]*z[6] + 8*z[1]*z[3]*z[5]*z[7] + 16*z[1]*z[3]*z[5]*z[8] - 30*z[1]*z[3]*z[5] + 16*z[1]*z[3]*z[6]*z[7] + 32*z[1]*z[3]*z[6]*z[8] - 60*z[1]*z[3]*z[6] + 64*z[1]*z[3]*z[7]*z[8] - 120*z[1]*z[3]*z[7] - 240*z[1]*z[3]*z[8] + 1194*z[1]*z[3] + 8*z[1]*z[4]*z[5]*z[6] + 16*z[1]*z[4]*z[5]*z[7] + 32*z[1]*z[4]*z[5]*z[8] - 60*z[1]*z[4]*z[5] + 32*z[1]*z[4]*z[6]*z[7] + 64*z[1]*z[4]*z[6]*z[8] - 120*z[1]*z[4]*z[6] + 128*z[1]*z[4]*z[7]*z[8] - 240*z[1]*z[4]*z[7] - 480*z[1]*z[4]*z[8] + 2196*z[1]*z[4] - 15*z[1]*z[5]*z[6] - 30*z[1]*z[5]*z[7] - 60*z[1]*z[5]*z[8] + 225*z[1]*z[5]/2 - 60*z[1]*z[6]*z[7] - 120*z[1]*z[6]*z[8] + 225*z[1]*z[6] - 240*z[1]*z[7]*z[8] + 450*z[1]*z[7] + 900*z[1]*z[8] - 2910*z[1] - 1440*z[2]*z[3]*z[4] + 8*z[2]*z[3]*z[5]*z[6] + 16*z[2]*z[3]*z[5]*z[7] + 32*z[2]*z[3]*z[5]*z[8] - 60*z[2]*z[3]*z[5] + 32*z[2]*z[3]*z[6]*z[7] + 64*z[2]*z[3]*z[6]*z[8] - 120*z[2]*z[3]*z[6] + 128*z[2]*z[3]*z[7]*z[8] - 240*z[2]*z[3]*z[7] - 480*z[2]*z[3]*z[8] + 2376*z[2]*z[3] + 16*z[2]*z[4]*z[5]*z[6] + 32*z[2]*z[4]*z[5]*z[7] + 64*z[2]*z[4]*z[5]*z[8] - 120*z[2]*z[4]*z[5] + 64*z[2]*z[4]*z[6]*z[7] + 128*z[2]*z[4]*z[6]*z[8] - 240*z[2]*z[4]*z[6] + 256*z[2]*z[4]*z[7]*z[8] - 480*z[2]*z[4]*z[7] - 960*z[2]*z[4]*z[8] + 4368*z[2]*z[4] - 30*z[2]*z[5]*z[6] - 60*z[2]*z[5]*z[7] - 120*z[2]*z[5]*z[8] + 225*z[2]*z[5] - 120*z[2]*z[6]*z[7] - 240*z[2]*z[6]*z[8] + 450*z[2]*z[6] - 480*z[2]*z[7]*z[8] + 900*z[2]*z[7] + 1800*z[2]*z[8] - 5775*z[2] + 32*z[3]*z[4]*z[5]*z[6] + 64*z[3]*z[4]*z[5]*z[7] + 128*z[3]*z[4]*z[5]*z[8] - 240*z[3]*z[4]*z[5] + 128*z[3]*z[4]*z[6]*z[7] + 256*z[3]*z[4]*z[6]*z[8] - 480*z[3]*z[4]*z[6] + 512*z[3]*z[4]*z[7]*z[8] - 960*z[3]*z[4]*z[7] - 1920*z[3]*z[4]*z[8] + 8544*z[3]*z[4] - 60*z[3]*z[5]*z[6] - 120*z[3]*z[5]*z[7] - 240*z[3]*z[5]*z[8] + 450*z[3]*z[5] - 240*z[3]*z[6]*z[7] - 480*z[3]*z[6]*z[8] + 900*z[3]*z[6] - 960*z[3]*z[7]*z[8] + 1800*z[3]*z[7] + 3600*z[3]*z[8] - 11190*z[3] - 120*z[4]*z[5]*z[6] - 240*z[4]*z[5]*z[7] - 480*z[4]*z[5]*z[8] + 900*z[4]*z[5] - 480*z[4]*z[6]*z[7] - 960*z[4]*z[6]*z[8] + 1800*z[4]*z[6] - 1920*z[4]*z[7]*z[8] + 3600*z[4]*z[7] + 7200*z[4]*z[8] - 19500*z[4] + 96*z[5]*z[6]*z[7]*z[8] - 180*z[5]*z[6]*z[7] - 360*z[5]*z[6]*z[8] + 609*z[5]*z[6] - 720*z[5]*z[7]*z[8] + 1194*z[5]*z[7] + 2196*z[5]*z[8] - 2910*z[5] - 1440*z[6]*z[7]*z[8] + 2376*z[6]*z[7] + 4368*z[6]*z[8] - 5775*z[6] + 8544*z[7]*z[8] - 11190*z[7] - 19500*z[8] + 66761/2)

In [21]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 8  # 
p = 1  # QAOA depth

hamiltonian_less_15_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_equation_less_15_2))
final_circuit_less_15_2, result_less_15_2 = qaoa(num_qubits, hamiltonian_less_15_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.10251549 1.08459815]
Minimum expectation value: 22842.87109375


In [22]:
top_solutions_less_15_2 = find_best_bitstrings(final_circuit_less_15_2, hamiltonian_less_15_2)

Top 5 bitstrings:
Bitstring: 00010001, Cost: -66760.0000, Count: 1
Bitstring: 00100001, Cost: -66756.0000, Count: 1
Bitstring: 00000011, Cost: -66724.0000, Count: 4
Bitstring: 00110001, Cost: -66712.0000, Count: 1
Bitstring: 00100011, Cost: -66660.0000, Count: 2


In [23]:
bitstring_to_pm1(top_solutions_less_15_2, evaluate_hamiltonian_less_15_2)

["Bitstring: ('00010001', -66760.0, 1), Evaluated cost: 1.0",
 "Bitstring: ('00010001', -66760.0, 1), Evaluated cost: 4.0",
 "Bitstring: ('00010001', -66760.0, 1), Evaluated cost: 36.0",
 "Bitstring: ('00010001', -66760.0, 1), Evaluated cost: 49.0",
 "Bitstring: ('00010001', -66760.0, 1), Evaluated cost: 100.0"]

For numbers between 0 and 127, which has binary length of 7

In [24]:
paso1_D_equation_less_127_2 = substitute_with_global_binary_symbols(D_equation(x[1],x[2])**2, 7, base_name="b")
paso2_D_equation_less_127_2 = remove_variable_exponents(paso1_D_equation_less_127_2)
paso3_D_equation_less_127_2 = substitute_with_spin_variables(paso2_D_equation_less_127_2)

In [25]:
def evaluate_hamiltonian_less_127_2(z):
    return(32*z[1]*z[10]*z[11]*z[2] + 64*z[1]*z[10]*z[11]*z[3] + 128*z[1]*z[10]*z[11]*z[4] + 256*z[1]*z[10]*z[11]*z[5] + 512*z[1]*z[10]*z[11]*z[6] + 1024*z[1]*z[10]*z[11]*z[7] - 2032*z[1]*z[10]*z[11] + 64*z[1]*z[10]*z[12]*z[2] + 128*z[1]*z[10]*z[12]*z[3] + 256*z[1]*z[10]*z[12]*z[4] + 512*z[1]*z[10]*z[12]*z[5] + 1024*z[1]*z[10]*z[12]*z[6] + 2048*z[1]*z[10]*z[12]*z[7] - 4064*z[1]*z[10]*z[12] + 128*z[1]*z[10]*z[13]*z[2] + 256*z[1]*z[10]*z[13]*z[3] + 512*z[1]*z[10]*z[13]*z[4] + 1024*z[1]*z[10]*z[13]*z[5] + 2048*z[1]*z[10]*z[13]*z[6] + 4096*z[1]*z[10]*z[13]*z[7] - 8128*z[1]*z[10]*z[13] + 256*z[1]*z[10]*z[14]*z[2] + 512*z[1]*z[10]*z[14]*z[3] + 1024*z[1]*z[10]*z[14]*z[4] + 2048*z[1]*z[10]*z[14]*z[5] + 4096*z[1]*z[10]*z[14]*z[6] + 8192*z[1]*z[10]*z[14]*z[7] - 16256*z[1]*z[10]*z[14] + 4*z[1]*z[10]*z[2]*z[8] + 8*z[1]*z[10]*z[2]*z[9] - 508*z[1]*z[10]*z[2] + 8*z[1]*z[10]*z[3]*z[8] + 16*z[1]*z[10]*z[3]*z[9] - 1016*z[1]*z[10]*z[3] + 16*z[1]*z[10]*z[4]*z[8] + 32*z[1]*z[10]*z[4]*z[9] - 2032*z[1]*z[10]*z[4] + 32*z[1]*z[10]*z[5]*z[8] + 64*z[1]*z[10]*z[5]*z[9] - 4064*z[1]*z[10]*z[5] + 64*z[1]*z[10]*z[6]*z[8] + 128*z[1]*z[10]*z[6]*z[9] - 8128*z[1]*z[10]*z[6] + 128*z[1]*z[10]*z[7]*z[8] + 256*z[1]*z[10]*z[7]*z[9] - 16256*z[1]*z[10]*z[7] - 254*z[1]*z[10]*z[8] - 508*z[1]*z[10]*z[9] + 32258*z[1]*z[10] + 128*z[1]*z[11]*z[12]*z[2] + 256*z[1]*z[11]*z[12]*z[3] + 512*z[1]*z[11]*z[12]*z[4] + 1024*z[1]*z[11]*z[12]*z[5] + 2048*z[1]*z[11]*z[12]*z[6] + 4096*z[1]*z[11]*z[12]*z[7] - 8128*z[1]*z[11]*z[12] + 256*z[1]*z[11]*z[13]*z[2] + 512*z[1]*z[11]*z[13]*z[3] + 1024*z[1]*z[11]*z[13]*z[4] + 2048*z[1]*z[11]*z[13]*z[5] + 4096*z[1]*z[11]*z[13]*z[6] + 8192*z[1]*z[11]*z[13]*z[7] - 16256*z[1]*z[11]*z[13] + 512*z[1]*z[11]*z[14]*z[2] + 1024*z[1]*z[11]*z[14]*z[3] + 2048*z[1]*z[11]*z[14]*z[4] + 4096*z[1]*z[11]*z[14]*z[5] + 8192*z[1]*z[11]*z[14]*z[6] + 16384*z[1]*z[11]*z[14]*z[7] - 32512*z[1]*z[11]*z[14] + 8*z[1]*z[11]*z[2]*z[8] + 16*z[1]*z[11]*z[2]*z[9] - 1016*z[1]*z[11]*z[2] + 16*z[1]*z[11]*z[3]*z[8] + 32*z[1]*z[11]*z[3]*z[9] - 2032*z[1]*z[11]*z[3] + 32*z[1]*z[11]*z[4]*z[8] + 64*z[1]*z[11]*z[4]*z[9] - 4064*z[1]*z[11]*z[4] + 64*z[1]*z[11]*z[5]*z[8] + 128*z[1]*z[11]*z[5]*z[9] - 8128*z[1]*z[11]*z[5] + 128*z[1]*z[11]*z[6]*z[8] + 256*z[1]*z[11]*z[6]*z[9] - 16256*z[1]*z[11]*z[6] + 256*z[1]*z[11]*z[7]*z[8] + 512*z[1]*z[11]*z[7]*z[9] - 32512*z[1]*z[11]*z[7] - 508*z[1]*z[11]*z[8] - 1016*z[1]*z[11]*z[9] + 64516*z[1]*z[11] + 512*z[1]*z[12]*z[13]*z[2] + 1024*z[1]*z[12]*z[13]*z[3] + 2048*z[1]*z[12]*z[13]*z[4] + 4096*z[1]*z[12]*z[13]*z[5] + 8192*z[1]*z[12]*z[13]*z[6] + 16384*z[1]*z[12]*z[13]*z[7] - 32512*z[1]*z[12]*z[13] + 1024*z[1]*z[12]*z[14]*z[2] + 2048*z[1]*z[12]*z[14]*z[3] + 4096*z[1]*z[12]*z[14]*z[4] + 8192*z[1]*z[12]*z[14]*z[5] + 16384*z[1]*z[12]*z[14]*z[6] + 32768*z[1]*z[12]*z[14]*z[7] - 65024*z[1]*z[12]*z[14] + 16*z[1]*z[12]*z[2]*z[8] + 32*z[1]*z[12]*z[2]*z[9] - 2032*z[1]*z[12]*z[2] + 32*z[1]*z[12]*z[3]*z[8] + 64*z[1]*z[12]*z[3]*z[9] - 4064*z[1]*z[12]*z[3] + 64*z[1]*z[12]*z[4]*z[8] + 128*z[1]*z[12]*z[4]*z[9] - 8128*z[1]*z[12]*z[4] + 128*z[1]*z[12]*z[5]*z[8] + 256*z[1]*z[12]*z[5]*z[9] - 16256*z[1]*z[12]*z[5] + 256*z[1]*z[12]*z[6]*z[8] + 512*z[1]*z[12]*z[6]*z[9] - 32512*z[1]*z[12]*z[6] + 512*z[1]*z[12]*z[7]*z[8] + 1024*z[1]*z[12]*z[7]*z[9] - 65024*z[1]*z[12]*z[7] - 1016*z[1]*z[12]*z[8] - 2032*z[1]*z[12]*z[9] + 129032*z[1]*z[12] + 2048*z[1]*z[13]*z[14]*z[2] + 4096*z[1]*z[13]*z[14]*z[3] + 8192*z[1]*z[13]*z[14]*z[4] + 16384*z[1]*z[13]*z[14]*z[5] + 32768*z[1]*z[13]*z[14]*z[6] + 65536*z[1]*z[13]*z[14]*z[7] - 130048*z[1]*z[13]*z[14] + 32*z[1]*z[13]*z[2]*z[8] + 64*z[1]*z[13]*z[2]*z[9] - 4064*z[1]*z[13]*z[2] + 64*z[1]*z[13]*z[3]*z[8] + 128*z[1]*z[13]*z[3]*z[9] - 8128*z[1]*z[13]*z[3] + 128*z[1]*z[13]*z[4]*z[8] + 256*z[1]*z[13]*z[4]*z[9] - 16256*z[1]*z[13]*z[4] + 256*z[1]*z[13]*z[5]*z[8] + 512*z[1]*z[13]*z[5]*z[9] - 32512*z[1]*z[13]*z[5] + 512*z[1]*z[13]*z[6]*z[8] + 1024*z[1]*z[13]*z[6]*z[9] - 65024*z[1]*z[13]*z[6] + 1024*z[1]*z[13]*z[7]*z[8] + 2048*z[1]*z[13]*z[7]*z[9] - 130048*z[1]*z[13]*z[7] - 2032*z[1]*z[13]*z[8] - 4064*z[1]*z[13]*z[9] + 258064*z[1]*z[13] + 64*z[1]*z[14]*z[2]*z[8] + 128*z[1]*z[14]*z[2]*z[9] - 8128*z[1]*z[14]*z[2] + 128*z[1]*z[14]*z[3]*z[8] + 256*z[1]*z[14]*z[3]*z[9] - 16256*z[1]*z[14]*z[3] + 256*z[1]*z[14]*z[4]*z[8] + 512*z[1]*z[14]*z[4]*z[9] - 32512*z[1]*z[14]*z[4] + 512*z[1]*z[14]*z[5]*z[8] + 1024*z[1]*z[14]*z[5]*z[9] - 65024*z[1]*z[14]*z[5] + 1024*z[1]*z[14]*z[6]*z[8] + 2048*z[1]*z[14]*z[6]*z[9] - 130048*z[1]*z[14]*z[6] + 2048*z[1]*z[14]*z[7]*z[8] + 4096*z[1]*z[14]*z[7]*z[9] - 260096*z[1]*z[14]*z[7] - 4064*z[1]*z[14]*z[8] - 8128*z[1]*z[14]*z[9] + 516128*z[1]*z[14] + 96*z[1]*z[2]*z[3]*z[4] + 192*z[1]*z[2]*z[3]*z[5] + 384*z[1]*z[2]*z[3]*z[6] + 768*z[1]*z[2]*z[3]*z[7] - 1524*z[1]*z[2]*z[3] + 384*z[1]*z[2]*z[4]*z[5] + 768*z[1]*z[2]*z[4]*z[6] + 1536*z[1]*z[2]*z[4]*z[7] - 3048*z[1]*z[2]*z[4] + 1536*z[1]*z[2]*z[5]*z[6] + 3072*z[1]*z[2]*z[5]*z[7] - 6096*z[1]*z[2]*z[5] + 6144*z[1]*z[2]*z[6]*z[7] - 12192*z[1]*z[2]*z[6] - 24384*z[1]*z[2]*z[7] + 2*z[1]*z[2]*z[8]*z[9] - 127*z[1]*z[2]*z[8] - 254*z[1]*z[2]*z[9] + 43169*z[1]*z[2] + 768*z[1]*z[3]*z[4]*z[5] + 1536*z[1]*z[3]*z[4]*z[6] + 3072*z[1]*z[3]*z[4]*z[7] - 6096*z[1]*z[3]*z[4] + 3072*z[1]*z[3]*z[5]*z[6] + 6144*z[1]*z[3]*z[5]*z[7] - 12192*z[1]*z[3]*z[5] + 12288*z[1]*z[3]*z[6]*z[7] - 24384*z[1]*z[3]*z[6] - 48768*z[1]*z[3]*z[7] + 4*z[1]*z[3]*z[8]*z[9] - 254*z[1]*z[3]*z[8] - 508*z[1]*z[3]*z[9] + 86314*z[1]*z[3] + 6144*z[1]*z[4]*z[5]*z[6] + 12288*z[1]*z[4]*z[5]*z[7] - 24384*z[1]*z[4]*z[5] + 24576*z[1]*z[4]*z[6]*z[7] - 48768*z[1]*z[4]*z[6] - 97536*z[1]*z[4]*z[7] + 8*z[1]*z[4]*z[8]*z[9] - 508*z[1]*z[4]*z[8] - 1016*z[1]*z[4]*z[9] + 172436*z[1]*z[4] + 49152*z[1]*z[5]*z[6]*z[7] - 97536*z[1]*z[5]*z[6] - 195072*z[1]*z[5]*z[7] + 16*z[1]*z[5]*z[8]*z[9] - 1016*z[1]*z[5]*z[8] - 2032*z[1]*z[5]*z[9] + 343336*z[1]*z[5] - 390144*z[1]*z[6]*z[7] + 32*z[1]*z[6]*z[8]*z[9] - 2032*z[1]*z[6]*z[8] - 4064*z[1]*z[6]*z[9] + 674384*z[1]*z[6] + 64*z[1]*z[7]*z[8]*z[9] - 4064*z[1]*z[7]*z[8] - 8128*z[1]*z[7]*z[9] + 1250464*z[1]*z[7] - 127*z[1]*z[8]*z[9] + 16129*z[1]*z[8]/2 + 16129*z[1]*z[9] - 1717294*z[1] + 24576*z[10]*z[11]*z[12]*z[13] + 49152*z[10]*z[11]*z[12]*z[14] + 768*z[10]*z[11]*z[12]*z[8] + 1536*z[10]*z[11]*z[12]*z[9] - 97536*z[10]*z[11]*z[12] + 98304*z[10]*z[11]*z[13]*z[14] + 1536*z[10]*z[11]*z[13]*z[8] + 3072*z[10]*z[11]*z[13]*z[9] - 195072*z[10]*z[11]*z[13] + 3072*z[10]*z[11]*z[14]*z[8] + 6144*z[10]*z[11]*z[14]*z[9] - 390144*z[10]*z[11]*z[14] + 128*z[10]*z[11]*z[2]*z[3] + 256*z[10]*z[11]*z[2]*z[4] + 512*z[10]*z[11]*z[2]*z[5] + 1024*z[10]*z[11]*z[2]*z[6] + 2048*z[10]*z[11]*z[2]*z[7] - 4064*z[10]*z[11]*z[2] + 512*z[10]*z[11]*z[3]*z[4] + 1024*z[10]*z[11]*z[3]*z[5] + 2048*z[10]*z[11]*z[3]*z[6] + 4096*z[10]*z[11]*z[3]*z[7] - 8128*z[10]*z[11]*z[3] + 2048*z[10]*z[11]*z[4]*z[5] + 4096*z[10]*z[11]*z[4]*z[6] + 8192*z[10]*z[11]*z[4]*z[7] - 16256*z[10]*z[11]*z[4] + 8192*z[10]*z[11]*z[5]*z[6] + 16384*z[10]*z[11]*z[5]*z[7] - 32512*z[10]*z[11]*z[5] + 32768*z[10]*z[11]*z[6]*z[7] - 65024*z[10]*z[11]*z[6] - 130048*z[10]*z[11]*z[7] + 96*z[10]*z[11]*z[8]*z[9] - 6096*z[10]*z[11]*z[8] - 12192*z[10]*z[11]*z[9] + 689504*z[10]*z[11] + 196608*z[10]*z[12]*z[13]*z[14] + 3072*z[10]*z[12]*z[13]*z[8] + 6144*z[10]*z[12]*z[13]*z[9] - 390144*z[10]*z[12]*z[13] + 6144*z[10]*z[12]*z[14]*z[8] + 12288*z[10]*z[12]*z[14]*z[9] - 780288*z[10]*z[12]*z[14] + 256*z[10]*z[12]*z[2]*z[3] + 512*z[10]*z[12]*z[2]*z[4] + 1024*z[10]*z[12]*z[2]*z[5] + 2048*z[10]*z[12]*z[2]*z[6] + 4096*z[10]*z[12]*z[2]*z[7] - 8128*z[10]*z[12]*z[2] + 1024*z[10]*z[12]*z[3]*z[4] + 2048*z[10]*z[12]*z[3]*z[5] + 4096*z[10]*z[12]*z[3]*z[6] + 8192*z[10]*z[12]*z[3]*z[7] - 16256*z[10]*z[12]*z[3] + 4096*z[10]*z[12]*z[4]*z[5] + 8192*z[10]*z[12]*z[4]*z[6] + 16384*z[10]*z[12]*z[4]*z[7] - 32512*z[10]*z[12]*z[4] + 16384*z[10]*z[12]*z[5]*z[6] + 32768*z[10]*z[12]*z[5]*z[7] - 65024*z[10]*z[12]*z[5] + 65536*z[10]*z[12]*z[6]*z[7] - 130048*z[10]*z[12]*z[6] - 260096*z[10]*z[12]*z[7] + 192*z[10]*z[12]*z[8]*z[9] - 12192*z[10]*z[12]*z[8] - 24384*z[10]*z[12]*z[9] + 1372864*z[10]*z[12] + 12288*z[10]*z[13]*z[14]*z[8] + 24576*z[10]*z[13]*z[14]*z[9] - 1560576*z[10]*z[13]*z[14] + 512*z[10]*z[13]*z[2]*z[3] + 1024*z[10]*z[13]*z[2]*z[4] + 2048*z[10]*z[13]*z[2]*z[5] + 4096*z[10]*z[13]*z[2]*z[6] + 8192*z[10]*z[13]*z[2]*z[7] - 16256*z[10]*z[13]*z[2] + 2048*z[10]*z[13]*z[3]*z[4] + 4096*z[10]*z[13]*z[3]*z[5] + 8192*z[10]*z[13]*z[3]*z[6] + 16384*z[10]*z[13]*z[3]*z[7] - 32512*z[10]*z[13]*z[3] + 8192*z[10]*z[13]*z[4]*z[5] + 16384*z[10]*z[13]*z[4]*z[6] + 32768*z[10]*z[13]*z[4]*z[7] - 65024*z[10]*z[13]*z[4] + 32768*z[10]*z[13]*z[5]*z[6] + 65536*z[10]*z[13]*z[5]*z[7] - 130048*z[10]*z[13]*z[5] + 131072*z[10]*z[13]*z[6]*z[7] - 260096*z[10]*z[13]*z[6] - 520192*z[10]*z[13]*z[7] + 384*z[10]*z[13]*z[8]*z[9] - 24384*z[10]*z[13]*z[8] - 48768*z[10]*z[13]*z[9] + 2696576*z[10]*z[13] + 1024*z[10]*z[14]*z[2]*z[3] + 2048*z[10]*z[14]*z[2]*z[4] + 4096*z[10]*z[14]*z[2]*z[5] + 8192*z[10]*z[14]*z[2]*z[6] + 16384*z[10]*z[14]*z[2]*z[7] - 32512*z[10]*z[14]*z[2] + 4096*z[10]*z[14]*z[3]*z[4] + 8192*z[10]*z[14]*z[3]*z[5] + 16384*z[10]*z[14]*z[3]*z[6] + 32768*z[10]*z[14]*z[3]*z[7] - 65024*z[10]*z[14]*z[3] + 16384*z[10]*z[14]*z[4]*z[5] + 32768*z[10]*z[14]*z[4]*z[6] + 65536*z[10]*z[14]*z[4]*z[7] - 130048*z[10]*z[14]*z[4] + 65536*z[10]*z[14]*z[5]*z[6] + 131072*z[10]*z[14]*z[5]*z[7] - 260096*z[10]*z[14]*z[5] + 262144*z[10]*z[14]*z[6]*z[7] - 520192*z[10]*z[14]*z[6] - 1040384*z[10]*z[14]*z[7] + 768*z[10]*z[14]*z[8]*z[9] - 48768*z[10]*z[14]*z[8] - 97536*z[10]*z[14]*z[9] + 4999936*z[10]*z[14] + 16*z[10]*z[2]*z[3]*z[8] + 32*z[10]*z[2]*z[3]*z[9] - 2032*z[10]*z[2]*z[3] + 32*z[10]*z[2]*z[4]*z[8] + 64*z[10]*z[2]*z[4]*z[9] - 4064*z[10]*z[2]*z[4] + 64*z[10]*z[2]*z[5]*z[8] + 128*z[10]*z[2]*z[5]*z[9] - 8128*z[10]*z[2]*z[5] + 128*z[10]*z[2]*z[6]*z[8] + 256*z[10]*z[2]*z[6]*z[9] - 16256*z[10]*z[2]*z[6] + 256*z[10]*z[2]*z[7]*z[8] + 512*z[10]*z[2]*z[7]*z[9] - 32512*z[10]*z[2]*z[7] - 508*z[10]*z[2]*z[8] - 1016*z[10]*z[2]*z[9] + 64516*z[10]*z[2] + 64*z[10]*z[3]*z[4]*z[8] + 128*z[10]*z[3]*z[4]*z[9] - 8128*z[10]*z[3]*z[4] + 128*z[10]*z[3]*z[5]*z[8] + 256*z[10]*z[3]*z[5]*z[9] - 16256*z[10]*z[3]*z[5] + 256*z[10]*z[3]*z[6]*z[8] + 512*z[10]*z[3]*z[6]*z[9] - 32512*z[10]*z[3]*z[6] + 512*z[10]*z[3]*z[7]*z[8] + 1024*z[10]*z[3]*z[7]*z[9] - 65024*z[10]*z[3]*z[7] - 1016*z[10]*z[3]*z[8] - 2032*z[10]*z[3]*z[9] + 129032*z[10]*z[3] + 256*z[10]*z[4]*z[5]*z[8] + 512*z[10]*z[4]*z[5]*z[9] - 32512*z[10]*z[4]*z[5] + 512*z[10]*z[4]*z[6]*z[8] + 1024*z[10]*z[4]*z[6]*z[9] - 65024*z[10]*z[4]*z[6] + 1024*z[10]*z[4]*z[7]*z[8] + 2048*z[10]*z[4]*z[7]*z[9] - 130048*z[10]*z[4]*z[7] - 2032*z[10]*z[4]*z[8] - 4064*z[10]*z[4]*z[9] + 258064*z[10]*z[4] + 1024*z[10]*z[5]*z[6]*z[8] + 2048*z[10]*z[5]*z[6]*z[9] - 130048*z[10]*z[5]*z[6] + 2048*z[10]*z[5]*z[7]*z[8] + 4096*z[10]*z[5]*z[7]*z[9] - 260096*z[10]*z[5]*z[7] - 4064*z[10]*z[5]*z[8] - 8128*z[10]*z[5]*z[9] + 516128*z[10]*z[5] + 4096*z[10]*z[6]*z[7]*z[8] + 8192*z[10]*z[6]*z[7]*z[9] - 520192*z[10]*z[6]*z[7] - 8128*z[10]*z[6]*z[8] - 16256*z[10]*z[6]*z[9] + 1032256*z[10]*z[6] - 16256*z[10]*z[7]*z[8] - 32512*z[10]*z[7]*z[9] + 2064512*z[10]*z[7] - 1524*z[10]*z[8]*z[9] + 86314*z[10]*z[8] + 172616*z[10]*z[9] - 6865366*z[10] + 393216*z[11]*z[12]*z[13]*z[14] + 6144*z[11]*z[12]*z[13]*z[8] + 12288*z[11]*z[12]*z[13]*z[9] - 780288*z[11]*z[12]*z[13] + 12288*z[11]*z[12]*z[14]*z[8] + 24576*z[11]*z[12]*z[14]*z[9] - 1560576*z[11]*z[12]*z[14] + 512*z[11]*z[12]*z[2]*z[3] + 1024*z[11]*z[12]*z[2]*z[4] + 2048*z[11]*z[12]*z[2]*z[5] + 4096*z[11]*z[12]*z[2]*z[6] + 8192*z[11]*z[12]*z[2]*z[7] - 16256*z[11]*z[12]*z[2] + 2048*z[11]*z[12]*z[3]*z[4] + 4096*z[11]*z[12]*z[3]*z[5] + 8192*z[11]*z[12]*z[3]*z[6] + 16384*z[11]*z[12]*z[3]*z[7] - 32512*z[11]*z[12]*z[3] + 8192*z[11]*z[12]*z[4]*z[5] + 16384*z[11]*z[12]*z[4]*z[6] + 32768*z[11]*z[12]*z[4]*z[7] - 65024*z[11]*z[12]*z[4] + 32768*z[11]*z[12]*z[5]*z[6] + 65536*z[11]*z[12]*z[5]*z[7] - 130048*z[11]*z[12]*z[5] + 131072*z[11]*z[12]*z[6]*z[7] - 260096*z[11]*z[12]*z[6] - 520192*z[11]*z[12]*z[7] + 384*z[11]*z[12]*z[8]*z[9] - 24384*z[11]*z[12]*z[8] - 48768*z[11]*z[12]*z[9] + 2742656*z[11]*z[12] + 24576*z[11]*z[13]*z[14]*z[8] + 49152*z[11]*z[13]*z[14]*z[9] - 3121152*z[11]*z[13]*z[14] + 1024*z[11]*z[13]*z[2]*z[3] + 2048*z[11]*z[13]*z[2]*z[4] + 4096*z[11]*z[13]*z[2]*z[5] + 8192*z[11]*z[13]*z[2]*z[6] + 16384*z[11]*z[13]*z[2]*z[7] - 32512*z[11]*z[13]*z[2] + 4096*z[11]*z[13]*z[3]*z[4] + 8192*z[11]*z[13]*z[3]*z[5] + 16384*z[11]*z[13]*z[3]*z[6] + 32768*z[11]*z[13]*z[3]*z[7] - 65024*z[11]*z[13]*z[3] + 16384*z[11]*z[13]*z[4]*z[5] + 32768*z[11]*z[13]*z[4]*z[6] + 65536*z[11]*z[13]*z[4]*z[7] - 130048*z[11]*z[13]*z[4] + 65536*z[11]*z[13]*z[5]*z[6] + 131072*z[11]*z[13]*z[5]*z[7] - 260096*z[11]*z[13]*z[5] + 262144*z[11]*z[13]*z[6]*z[7] - 520192*z[11]*z[13]*z[6] - 1040384*z[11]*z[13]*z[7] + 768*z[11]*z[13]*z[8]*z[9] - 48768*z[11]*z[13]*z[8] - 97536*z[11]*z[13]*z[9] + 5387008*z[11]*z[13] + 2048*z[11]*z[14]*z[2]*z[3] + 4096*z[11]*z[14]*z[2]*z[4] + 8192*z[11]*z[14]*z[2]*z[5] + 16384*z[11]*z[14]*z[2]*z[6] + 32768*z[11]*z[14]*z[2]*z[7] - 65024*z[11]*z[14]*z[2] + 8192*z[11]*z[14]*z[3]*z[4] + 16384*z[11]*z[14]*z[3]*z[5] + 32768*z[11]*z[14]*z[3]*z[6] + 65536*z[11]*z[14]*z[3]*z[7] - 130048*z[11]*z[14]*z[3] + 32768*z[11]*z[14]*z[4]*z[5] + 65536*z[11]*z[14]*z[4]*z[6] + 131072*z[11]*z[14]*z[4]*z[7] - 260096*z[11]*z[14]*z[4] + 131072*z[11]*z[14]*z[5]*z[6] + 262144*z[11]*z[14]*z[5]*z[7] - 520192*z[11]*z[14]*z[5] + 524288*z[11]*z[14]*z[6]*z[7] - 1040384*z[11]*z[14]*z[6] - 2080768*z[11]*z[14]*z[7] + 1536*z[11]*z[14]*z[8]*z[9] - 97536*z[11]*z[14]*z[8] - 195072*z[11]*z[14]*z[9] + 9987584*z[11]*z[14] + 32*z[11]*z[2]*z[3]*z[8] + 64*z[11]*z[2]*z[3]*z[9] - 4064*z[11]*z[2]*z[3] + 64*z[11]*z[2]*z[4]*z[8] + 128*z[11]*z[2]*z[4]*z[9] - 8128*z[11]*z[2]*z[4] + 128*z[11]*z[2]*z[5]*z[8] + 256*z[11]*z[2]*z[5]*z[9] - 16256*z[11]*z[2]*z[5] + 256*z[11]*z[2]*z[6]*z[8] + 512*z[11]*z[2]*z[6]*z[9] - 32512*z[11]*z[2]*z[6] + 512*z[11]*z[2]*z[7]*z[8] + 1024*z[11]*z[2]*z[7]*z[9] - 65024*z[11]*z[2]*z[7] - 1016*z[11]*z[2]*z[8] - 2032*z[11]*z[2]*z[9] + 129032*z[11]*z[2] + 128*z[11]*z[3]*z[4]*z[8] + 256*z[11]*z[3]*z[4]*z[9] - 16256*z[11]*z[3]*z[4] + 256*z[11]*z[3]*z[5]*z[8] + 512*z[11]*z[3]*z[5]*z[9] - 32512*z[11]*z[3]*z[5] + 512*z[11]*z[3]*z[6]*z[8] + 1024*z[11]*z[3]*z[6]*z[9] - 65024*z[11]*z[3]*z[6] + 1024*z[11]*z[3]*z[7]*z[8] + 2048*z[11]*z[3]*z[7]*z[9] - 130048*z[11]*z[3]*z[7] - 2032*z[11]*z[3]*z[8] - 4064*z[11]*z[3]*z[9] + 258064*z[11]*z[3] + 512*z[11]*z[4]*z[5]*z[8] + 1024*z[11]*z[4]*z[5]*z[9] - 65024*z[11]*z[4]*z[5] + 1024*z[11]*z[4]*z[6]*z[8] + 2048*z[11]*z[4]*z[6]*z[9] - 130048*z[11]*z[4]*z[6] + 2048*z[11]*z[4]*z[7]*z[8] + 4096*z[11]*z[4]*z[7]*z[9] - 260096*z[11]*z[4]*z[7] - 4064*z[11]*z[4]*z[8] - 8128*z[11]*z[4]*z[9] + 516128*z[11]*z[4] + 2048*z[11]*z[5]*z[6]*z[8] + 4096*z[11]*z[5]*z[6]*z[9] - 260096*z[11]*z[5]*z[6] + 4096*z[11]*z[5]*z[7]*z[8] + 8192*z[11]*z[5]*z[7]*z[9] - 520192*z[11]*z[5]*z[7] - 8128*z[11]*z[5]*z[8] - 16256*z[11]*z[5]*z[9] + 1032256*z[11]*z[5] + 8192*z[11]*z[6]*z[7]*z[8] + 16384*z[11]*z[6]*z[7]*z[9] - 1040384*z[11]*z[6]*z[7] - 16256*z[11]*z[6]*z[8] - 32512*z[11]*z[6]*z[9] + 2064512*z[11]*z[6] - 32512*z[11]*z[7]*z[8] - 65024*z[11]*z[7]*z[9] + 4129024*z[11]*z[7] - 3048*z[11]*z[8]*z[9] + 172436*z[11]*z[8] + 344848*z[11]*z[9] - 13706348*z[11] + 49152*z[12]*z[13]*z[14]*z[8] + 98304*z[12]*z[13]*z[14]*z[9] - 6242304*z[12]*z[13]*z[14] + 2048*z[12]*z[13]*z[2]*z[3] + 4096*z[12]*z[13]*z[2]*z[4] + 8192*z[12]*z[13]*z[2]*z[5] + 16384*z[12]*z[13]*z[2]*z[6] + 32768*z[12]*z[13]*z[2]*z[7] - 65024*z[12]*z[13]*z[2] + 8192*z[12]*z[13]*z[3]*z[4] + 16384*z[12]*z[13]*z[3]*z[5] + 32768*z[12]*z[13]*z[3]*z[6] + 65536*z[12]*z[13]*z[3]*z[7] - 130048*z[12]*z[13]*z[3] + 32768*z[12]*z[13]*z[4]*z[5] + 65536*z[12]*z[13]*z[4]*z[6] + 131072*z[12]*z[13]*z[4]*z[7] - 260096*z[12]*z[13]*z[4] + 131072*z[12]*z[13]*z[5]*z[6] + 262144*z[12]*z[13]*z[5]*z[7] - 520192*z[12]*z[13]*z[5] + 524288*z[12]*z[13]*z[6]*z[7] - 1040384*z[12]*z[13]*z[6] - 2080768*z[12]*z[13]*z[7] + 1536*z[12]*z[13]*z[8]*z[9] - 97536*z[12]*z[13]*z[8] - 195072*z[12]*z[13]*z[9] + 10724864*z[12]*z[13] + 4096*z[12]*z[14]*z[2]*z[3] + 8192*z[12]*z[14]*z[2]*z[4] + 16384*z[12]*z[14]*z[2]*z[5] + 32768*z[12]*z[14]*z[2]*z[6] + 65536*z[12]*z[14]*z[2]*z[7] - 130048*z[12]*z[14]*z[2] + 16384*z[12]*z[14]*z[3]*z[4] + 32768*z[12]*z[14]*z[3]*z[5] + 65536*z[12]*z[14]*z[3]*z[6] + 131072*z[12]*z[14]*z[3]*z[7] - 260096*z[12]*z[14]*z[3] + 65536*z[12]*z[14]*z[4]*z[5] + 131072*z[12]*z[14]*z[4]*z[6] + 262144*z[12]*z[14]*z[4]*z[7] - 520192*z[12]*z[14]*z[4] + 262144*z[12]*z[14]*z[5]*z[6] + 524288*z[12]*z[14]*z[5]*z[7] - 1040384*z[12]*z[14]*z[5] + 1048576*z[12]*z[14]*z[6]*z[7] - 2080768*z[12]*z[14]*z[6] - 4161536*z[12]*z[14]*z[7] + 3072*z[12]*z[14]*z[8]*z[9] - 195072*z[12]*z[14]*z[8] - 390144*z[12]*z[14]*z[9] + 19876864*z[12]*z[14] + 64*z[12]*z[2]*z[3]*z[8] + 128*z[12]*z[2]*z[3]*z[9] - 8128*z[12]*z[2]*z[3] + 128*z[12]*z[2]*z[4]*z[8] + 256*z[12]*z[2]*z[4]*z[9] - 16256*z[12]*z[2]*z[4] + 256*z[12]*z[2]*z[5]*z[8] + 512*z[12]*z[2]*z[5]*z[9] - 32512*z[12]*z[2]*z[5] + 512*z[12]*z[2]*z[6]*z[8] + 1024*z[12]*z[2]*z[6]*z[9] - 65024*z[12]*z[2]*z[6] + 1024*z[12]*z[2]*z[7]*z[8] + 2048*z[12]*z[2]*z[7]*z[9] - 130048*z[12]*z[2]*z[7] - 2032*z[12]*z[2]*z[8] - 4064*z[12]*z[2]*z[9] + 258064*z[12]*z[2] + 256*z[12]*z[3]*z[4]*z[8] + 512*z[12]*z[3]*z[4]*z[9] - 32512*z[12]*z[3]*z[4] + 512*z[12]*z[3]*z[5]*z[8] + 1024*z[12]*z[3]*z[5]*z[9] - 65024*z[12]*z[3]*z[5] + 1024*z[12]*z[3]*z[6]*z[8] + 2048*z[12]*z[3]*z[6]*z[9] - 130048*z[12]*z[3]*z[6] + 2048*z[12]*z[3]*z[7]*z[8] + 4096*z[12]*z[3]*z[7]*z[9] - 260096*z[12]*z[3]*z[7] - 4064*z[12]*z[3]*z[8] - 8128*z[12]*z[3]*z[9] + 516128*z[12]*z[3] + 1024*z[12]*z[4]*z[5]*z[8] + 2048*z[12]*z[4]*z[5]*z[9] - 130048*z[12]*z[4]*z[5] + 2048*z[12]*z[4]*z[6]*z[8] + 4096*z[12]*z[4]*z[6]*z[9] - 260096*z[12]*z[4]*z[6] + 4096*z[12]*z[4]*z[7]*z[8] + 8192*z[12]*z[4]*z[7]*z[9] - 520192*z[12]*z[4]*z[7] - 8128*z[12]*z[4]*z[8] - 16256*z[12]*z[4]*z[9] + 1032256*z[12]*z[4] + 4096*z[12]*z[5]*z[6]*z[8] + 8192*z[12]*z[5]*z[6]*z[9] - 520192*z[12]*z[5]*z[6] + 8192*z[12]*z[5]*z[7]*z[8] + 16384*z[12]*z[5]*z[7]*z[9] - 1040384*z[12]*z[5]*z[7] - 16256*z[12]*z[5]*z[8] - 32512*z[12]*z[5]*z[9] + 2064512*z[12]*z[5] + 16384*z[12]*z[6]*z[7]*z[8] + 32768*z[12]*z[6]*z[7]*z[9] - 2080768*z[12]*z[6]*z[7] - 32512*z[12]*z[6]*z[8] - 65024*z[12]*z[6]*z[9] + 4129024*z[12]*z[6] - 65024*z[12]*z[7]*z[8] - 130048*z[12]*z[7]*z[9] + 8258048*z[12]*z[7] - 6096*z[12]*z[8]*z[9] + 343336*z[12]*z[8] + 686624*z[12]*z[9] - 27217624*z[12] + 8192*z[13]*z[14]*z[2]*z[3] + 16384*z[13]*z[14]*z[2]*z[4] + 32768*z[13]*z[14]*z[2]*z[5] + 65536*z[13]*z[14]*z[2]*z[6] + 131072*z[13]*z[14]*z[2]*z[7] - 260096*z[13]*z[14]*z[2] + 32768*z[13]*z[14]*z[3]*z[4] + 65536*z[13]*z[14]*z[3]*z[5] + 131072*z[13]*z[14]*z[3]*z[6] + 262144*z[13]*z[14]*z[3]*z[7] - 520192*z[13]*z[14]*z[3] + 131072*z[13]*z[14]*z[4]*z[5] + 262144*z[13]*z[14]*z[4]*z[6] + 524288*z[13]*z[14]*z[4]*z[7] - 1040384*z[13]*z[14]*z[4] + 524288*z[13]*z[14]*z[5]*z[6] + 1048576*z[13]*z[14]*z[5]*z[7] - 2080768*z[13]*z[14]*z[5] + 2097152*z[13]*z[14]*z[6]*z[7] - 4161536*z[13]*z[14]*z[6] - 8323072*z[13]*z[14]*z[7] + 6144*z[13]*z[14]*z[8]*z[9] - 390144*z[13]*z[14]*z[8] - 780288*z[13]*z[14]*z[9] + 38967296*z[13]*z[14] + 128*z[13]*z[2]*z[3]*z[8] + 256*z[13]*z[2]*z[3]*z[9] - 16256*z[13]*z[2]*z[3] + 256*z[13]*z[2]*z[4]*z[8] + 512*z[13]*z[2]*z[4]*z[9] - 32512*z[13]*z[2]*z[4] + 512*z[13]*z[2]*z[5]*z[8] + 1024*z[13]*z[2]*z[5]*z[9] - 65024*z[13]*z[2]*z[5] + 1024*z[13]*z[2]*z[6]*z[8] + 2048*z[13]*z[2]*z[6]*z[9] - 130048*z[13]*z[2]*z[6] + 2048*z[13]*z[2]*z[7]*z[8] + 4096*z[13]*z[2]*z[7]*z[9] - 260096*z[13]*z[2]*z[7] - 4064*z[13]*z[2]*z[8] - 8128*z[13]*z[2]*z[9] + 516128*z[13]*z[2] + 512*z[13]*z[3]*z[4]*z[8] + 1024*z[13]*z[3]*z[4]*z[9] - 65024*z[13]*z[3]*z[4] + 1024*z[13]*z[3]*z[5]*z[8] + 2048*z[13]*z[3]*z[5]*z[9] - 130048*z[13]*z[3]*z[5] + 2048*z[13]*z[3]*z[6]*z[8] + 4096*z[13]*z[3]*z[6]*z[9] - 260096*z[13]*z[3]*z[6] + 4096*z[13]*z[3]*z[7]*z[8] + 8192*z[13]*z[3]*z[7]*z[9] - 520192*z[13]*z[3]*z[7] - 8128*z[13]*z[3]*z[8] - 16256*z[13]*z[3]*z[9] + 1032256*z[13]*z[3] + 2048*z[13]*z[4]*z[5]*z[8] + 4096*z[13]*z[4]*z[5]*z[9] - 260096*z[13]*z[4]*z[5] + 4096*z[13]*z[4]*z[6]*z[8] + 8192*z[13]*z[4]*z[6]*z[9] - 520192*z[13]*z[4]*z[6] + 8192*z[13]*z[4]*z[7]*z[8] + 16384*z[13]*z[4]*z[7]*z[9] - 1040384*z[13]*z[4]*z[7] - 16256*z[13]*z[4]*z[8] - 32512*z[13]*z[4]*z[9] + 2064512*z[13]*z[4] + 8192*z[13]*z[5]*z[6]*z[8] + 16384*z[13]*z[5]*z[6]*z[9] - 1040384*z[13]*z[5]*z[6] + 16384*z[13]*z[5]*z[7]*z[8] + 32768*z[13]*z[5]*z[7]*z[9] - 2080768*z[13]*z[5]*z[7] - 32512*z[13]*z[5]*z[8] - 65024*z[13]*z[5]*z[9] + 4129024*z[13]*z[5] + 32768*z[13]*z[6]*z[7]*z[8] + 65536*z[13]*z[6]*z[7]*z[9] - 4161536*z[13]*z[6]*z[7] - 65024*z[13]*z[6]*z[8] - 130048*z[13]*z[6]*z[9] + 8258048*z[13]*z[6] - 130048*z[13]*z[7]*z[8] - 260096*z[13]*z[7]*z[9] + 16516096*z[13]*z[7] - 12192*z[13]*z[8]*z[9] + 674384*z[13]*z[8] + 1348672*z[13]*z[9] - 52874672*z[13] + 256*z[14]*z[2]*z[3]*z[8] + 512*z[14]*z[2]*z[3]*z[9] - 32512*z[14]*z[2]*z[3] + 512*z[14]*z[2]*z[4]*z[8] + 1024*z[14]*z[2]*z[4]*z[9] - 65024*z[14]*z[2]*z[4] + 1024*z[14]*z[2]*z[5]*z[8] + 2048*z[14]*z[2]*z[5]*z[9] - 130048*z[14]*z[2]*z[5] + 2048*z[14]*z[2]*z[6]*z[8] + 4096*z[14]*z[2]*z[6]*z[9] - 260096*z[14]*z[2]*z[6] + 4096*z[14]*z[2]*z[7]*z[8] + 8192*z[14]*z[2]*z[7]*z[9] - 520192*z[14]*z[2]*z[7] - 8128*z[14]*z[2]*z[8] - 16256*z[14]*z[2]*z[9] + 1032256*z[14]*z[2] + 1024*z[14]*z[3]*z[4]*z[8] + 2048*z[14]*z[3]*z[4]*z[9] - 130048*z[14]*z[3]*z[4] + 2048*z[14]*z[3]*z[5]*z[8] + 4096*z[14]*z[3]*z[5]*z[9] - 260096*z[14]*z[3]*z[5] + 4096*z[14]*z[3]*z[6]*z[8] + 8192*z[14]*z[3]*z[6]*z[9] - 520192*z[14]*z[3]*z[6] + 8192*z[14]*z[3]*z[7]*z[8] + 16384*z[14]*z[3]*z[7]*z[9] - 1040384*z[14]*z[3]*z[7] - 16256*z[14]*z[3]*z[8] - 32512*z[14]*z[3]*z[9] + 2064512*z[14]*z[3] + 4096*z[14]*z[4]*z[5]*z[8] + 8192*z[14]*z[4]*z[5]*z[9] - 520192*z[14]*z[4]*z[5] + 8192*z[14]*z[4]*z[6]*z[8] + 16384*z[14]*z[4]*z[6]*z[9] - 1040384*z[14]*z[4]*z[6] + 16384*z[14]*z[4]*z[7]*z[8] + 32768*z[14]*z[4]*z[7]*z[9] - 2080768*z[14]*z[4]*z[7] - 32512*z[14]*z[4]*z[8] - 65024*z[14]*z[4]*z[9] + 4129024*z[14]*z[4] + 16384*z[14]*z[5]*z[6]*z[8] + 32768*z[14]*z[5]*z[6]*z[9] - 2080768*z[14]*z[5]*z[6] + 32768*z[14]*z[5]*z[7]*z[8] + 65536*z[14]*z[5]*z[7]*z[9] - 4161536*z[14]*z[5]*z[7] - 65024*z[14]*z[5]*z[8] - 130048*z[14]*z[5]*z[9] + 8258048*z[14]*z[5] + 65536*z[14]*z[6]*z[7]*z[8] + 131072*z[14]*z[6]*z[7]*z[9] - 8323072*z[14]*z[6]*z[7] - 130048*z[14]*z[6]*z[8] - 260096*z[14]*z[6]*z[9] + 16516096*z[14]*z[6] - 260096*z[14]*z[7]*z[8] - 520192*z[14]*z[7]*z[9] + 33032192*z[14]*z[7] - 24384*z[14]*z[8]*z[9] + 1250464*z[14]*z[8] + 2500736*z[14]*z[9] - 93264736*z[14] + 1536*z[2]*z[3]*z[4]*z[5] + 3072*z[2]*z[3]*z[4]*z[6] + 6144*z[2]*z[3]*z[4]*z[7] - 12192*z[2]*z[3]*z[4] + 6144*z[2]*z[3]*z[5]*z[6] + 12288*z[2]*z[3]*z[5]*z[7] - 24384*z[2]*z[3]*z[5] + 24576*z[2]*z[3]*z[6]*z[7] - 48768*z[2]*z[3]*z[6] - 97536*z[2]*z[3]*z[7] + 8*z[2]*z[3]*z[8]*z[9] - 508*z[2]*z[3]*z[8] - 1016*z[2]*z[3]*z[9] + 172616*z[2]*z[3] + 12288*z[2]*z[4]*z[5]*z[6] + 24576*z[2]*z[4]*z[5]*z[7] - 48768*z[2]*z[4]*z[5] + 49152*z[2]*z[4]*z[6]*z[7] - 97536*z[2]*z[4]*z[6] - 195072*z[2]*z[4]*z[7] + 16*z[2]*z[4]*z[8]*z[9] - 1016*z[2]*z[4]*z[8] - 2032*z[2]*z[4]*z[9] + 344848*z[2]*z[4] + 98304*z[2]*z[5]*z[6]*z[7] - 195072*z[2]*z[5]*z[6] - 390144*z[2]*z[5]*z[7] + 32*z[2]*z[5]*z[8]*z[9] - 2032*z[2]*z[5]*z[8] - 4064*z[2]*z[5]*z[9] + 686624*z[2]*z[5] - 780288*z[2]*z[6]*z[7] + 64*z[2]*z[6]*z[8]*z[9] - 4064*z[2]*z[6]*z[8] - 8128*z[2]*z[6]*z[9] + 1348672*z[2]*z[6] + 128*z[2]*z[7]*z[8]*z[9] - 8128*z[2]*z[7]*z[8] - 16256*z[2]*z[7]*z[9] + 2500736*z[2]*z[7] - 254*z[2]*z[8]*z[9] + 16129*z[2]*z[8] + 32258*z[2]*z[9] - 3434207*z[2] + 24576*z[3]*z[4]*z[5]*z[6] + 49152*z[3]*z[4]*z[5]*z[7] - 97536*z[3]*z[4]*z[5] + 98304*z[3]*z[4]*z[6]*z[7] - 195072*z[3]*z[4]*z[6] - 390144*z[3]*z[4]*z[7] + 32*z[3]*z[4]*z[8]*z[9] - 2032*z[3]*z[4]*z[8] - 4064*z[3]*z[4]*z[9] + 689504*z[3]*z[4] + 196608*z[3]*z[5]*z[6]*z[7] - 390144*z[3]*z[5]*z[6] - 780288*z[3]*z[5]*z[7] + 64*z[3]*z[5]*z[8]*z[9] - 4064*z[3]*z[5]*z[8] - 8128*z[3]*z[5]*z[9] + 1372864*z[3]*z[5] - 1560576*z[3]*z[6]*z[7] + 128*z[3]*z[6]*z[8]*z[9] - 8128*z[3]*z[6]*z[8] - 16256*z[3]*z[6]*z[9] + 2696576*z[3]*z[6] + 256*z[3]*z[7]*z[8]*z[9] - 16256*z[3]*z[7]*z[8] - 32512*z[3]*z[7]*z[9] + 4999936*z[3]*z[7] - 508*z[3]*z[8]*z[9] + 32258*z[3]*z[8] + 64516*z[3]*z[9] - 6865366*z[3] + 393216*z[4]*z[5]*z[6]*z[7] - 780288*z[4]*z[5]*z[6] - 1560576*z[4]*z[5]*z[7] + 128*z[4]*z[5]*z[8]*z[9] - 8128*z[4]*z[5]*z[8] - 16256*z[4]*z[5]*z[9] + 2742656*z[4]*z[5] - 3121152*z[4]*z[6]*z[7] + 256*z[4]*z[6]*z[8]*z[9] - 16256*z[4]*z[6]*z[8] - 32512*z[4]*z[6]*z[9] + 5387008*z[4]*z[6] + 512*z[4]*z[7]*z[8]*z[9] - 32512*z[4]*z[7]*z[8] - 65024*z[4]*z[7]*z[9] + 9987584*z[4]*z[7] - 1016*z[4]*z[8]*z[9] + 64516*z[4]*z[8] + 129032*z[4]*z[9] - 13706348*z[4] - 6242304*z[5]*z[6]*z[7] + 512*z[5]*z[6]*z[8]*z[9] - 32512*z[5]*z[6]*z[8] - 65024*z[5]*z[6]*z[9] + 10724864*z[5]*z[6] + 1024*z[5]*z[7]*z[8]*z[9] - 65024*z[5]*z[7]*z[8] - 130048*z[5]*z[7]*z[9] + 19876864*z[5]*z[7] - 2032*z[5]*z[8]*z[9] + 129032*z[5]*z[8] + 258064*z[5]*z[9] - 27217624*z[5] + 2048*z[6]*z[7]*z[8]*z[9] - 130048*z[6]*z[7]*z[8] - 260096*z[6]*z[7]*z[9] + 38967296*z[6]*z[7] - 4064*z[6]*z[8]*z[9] + 258064*z[6]*z[8] + 516128*z[6]*z[9] - 52874672*z[6] - 8128*z[7]*z[8]*z[9] + 516128*z[7]*z[8] + 1032256*z[7]*z[9] - 93264736*z[7] + 43169*z[8]*z[9] - 1717294*z[8] - 3434207*z[9] + 326978409/2)

In [26]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 14  # 
p = 1  # QAOA depth

hamiltonian_less_127_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_equation_less_127_2))
final_circuit_less_127_2, result_less_127_2 = qaoa(num_qubits, hamiltonian_less_127_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.1000853  1.09994781]
Minimum expectation value: 3647064.2734375


In [27]:
top_solutions_less_127_2 = find_best_bitstrings(final_circuit_less_127_2, hamiltonian_less_127_2)

Top 5 bitstrings:
Bitstring: 00000100000101, Cost: -326977732.0000, Count: 1
Bitstring: 00001000001001, Cost: -326969572.0000, Count: 1
Bitstring: 00010110000011, Cost: -326962280.0000, Count: 1
Bitstring: 00011000000001, Cost: -326958244.0000, Count: 1
Bitstring: 00011100000111, Cost: -326919844.0000, Count: 1


In [28]:
bitstring_to_pm1(top_solutions_less_127_2, evaluate_hamiltonian_less_127_2)

["Bitstring: ('00000100000101', -326977732.0, 1), Evaluated cost: 676.0",
 "Bitstring: ('00000100000101', -326977732.0, 1), Evaluated cost: 8836.0",
 "Bitstring: ('00000100000101', -326977732.0, 1), Evaluated cost: 16129.0",
 "Bitstring: ('00000100000101', -326977732.0, 1), Evaluated cost: 20164.0",
 "Bitstring: ('00000100000101', -326977732.0, 1), Evaluated cost: 58564.0"]

For numbers between 0 and 255, which has binary length of 8

In [29]:
paso1_D_equation_less_255_2 = substitute_with_global_binary_symbols(D_equation(x[1],x[2])**2, 8, base_name="b")
paso2_D_equation_less_255_2 = remove_variable_exponents(paso1_D_equation_less_255_2)
paso3_D_equation_less_255_2 = substitute_with_spin_variables(paso2_D_equation_less_255_2)

In [34]:
def evaluate_hamiltonian_less_255_2(z):
    return(8*z[1]*z[10]*z[11]*z[2] + 16*z[1]*z[10]*z[11]*z[3] + 32*z[1]*z[10]*z[11]*z[4] + 64*z[1]*z[10]*z[11]*z[5] + 128*z[1]*z[10]*z[11]*z[6] + 256*z[1]*z[10]*z[11]*z[7] + 512*z[1]*z[10]*z[11]*z[8] - 1020*z[1]*z[10]*z[11] + 16*z[1]*z[10]*z[12]*z[2] + 32*z[1]*z[10]*z[12]*z[3] + 64*z[1]*z[10]*z[12]*z[4] + 128*z[1]*z[10]*z[12]*z[5] + 256*z[1]*z[10]*z[12]*z[6] + 512*z[1]*z[10]*z[12]*z[7] + 1024*z[1]*z[10]*z[12]*z[8] - 2040*z[1]*z[10]*z[12] + 32*z[1]*z[10]*z[13]*z[2] + 64*z[1]*z[10]*z[13]*z[3] + 128*z[1]*z[10]*z[13]*z[4] + 256*z[1]*z[10]*z[13]*z[5] + 512*z[1]*z[10]*z[13]*z[6] + 1024*z[1]*z[10]*z[13]*z[7] + 2048*z[1]*z[10]*z[13]*z[8] - 4080*z[1]*z[10]*z[13] + 64*z[1]*z[10]*z[14]*z[2] + 128*z[1]*z[10]*z[14]*z[3] + 256*z[1]*z[10]*z[14]*z[4] + 512*z[1]*z[10]*z[14]*z[5] + 1024*z[1]*z[10]*z[14]*z[6] + 2048*z[1]*z[10]*z[14]*z[7] + 4096*z[1]*z[10]*z[14]*z[8] - 8160*z[1]*z[10]*z[14] + 128*z[1]*z[10]*z[15]*z[2] + 256*z[1]*z[10]*z[15]*z[3] + 512*z[1]*z[10]*z[15]*z[4] + 1024*z[1]*z[10]*z[15]*z[5] + 2048*z[1]*z[10]*z[15]*z[6] + 4096*z[1]*z[10]*z[15]*z[7] + 8192*z[1]*z[10]*z[15]*z[8] - 16320*z[1]*z[10]*z[15] + 256*z[1]*z[10]*z[16]*z[2] + 512*z[1]*z[10]*z[16]*z[3] + 1024*z[1]*z[10]*z[16]*z[4] + 2048*z[1]*z[10]*z[16]*z[5] + 4096*z[1]*z[10]*z[16]*z[6] + 8192*z[1]*z[10]*z[16]*z[7] + 16384*z[1]*z[10]*z[16]*z[8] - 32640*z[1]*z[10]*z[16] + 2*z[1]*z[10]*z[2]*z[9] - 510*z[1]*z[10]*z[2] + 4*z[1]*z[10]*z[3]*z[9] - 1020*z[1]*z[10]*z[3] + 8*z[1]*z[10]*z[4]*z[9] - 2040*z[1]*z[10]*z[4] + 16*z[1]*z[10]*z[5]*z[9] - 4080*z[1]*z[10]*z[5] + 32*z[1]*z[10]*z[6]*z[9] - 8160*z[1]*z[10]*z[6] + 64*z[1]*z[10]*z[7]*z[9] - 16320*z[1]*z[10]*z[7] + 128*z[1]*z[10]*z[8]*z[9] - 32640*z[1]*z[10]*z[8] - 255*z[1]*z[10]*z[9] + 65025*z[1]*z[10] + 32*z[1]*z[11]*z[12]*z[2] + 64*z[1]*z[11]*z[12]*z[3] + 128*z[1]*z[11]*z[12]*z[4] + 256*z[1]*z[11]*z[12]*z[5] + 512*z[1]*z[11]*z[12]*z[6] + 1024*z[1]*z[11]*z[12]*z[7] + 2048*z[1]*z[11]*z[12]*z[8] - 4080*z[1]*z[11]*z[12] + 64*z[1]*z[11]*z[13]*z[2] + 128*z[1]*z[11]*z[13]*z[3] + 256*z[1]*z[11]*z[13]*z[4] + 512*z[1]*z[11]*z[13]*z[5] + 1024*z[1]*z[11]*z[13]*z[6] + 2048*z[1]*z[11]*z[13]*z[7] + 4096*z[1]*z[11]*z[13]*z[8] - 8160*z[1]*z[11]*z[13] + 128*z[1]*z[11]*z[14]*z[2] + 256*z[1]*z[11]*z[14]*z[3] + 512*z[1]*z[11]*z[14]*z[4] + 1024*z[1]*z[11]*z[14]*z[5] + 2048*z[1]*z[11]*z[14]*z[6] + 4096*z[1]*z[11]*z[14]*z[7] + 8192*z[1]*z[11]*z[14]*z[8] - 16320*z[1]*z[11]*z[14] + 256*z[1]*z[11]*z[15]*z[2] + 512*z[1]*z[11]*z[15]*z[3] + 1024*z[1]*z[11]*z[15]*z[4] + 2048*z[1]*z[11]*z[15]*z[5] + 4096*z[1]*z[11]*z[15]*z[6] + 8192*z[1]*z[11]*z[15]*z[7] + 16384*z[1]*z[11]*z[15]*z[8] - 32640*z[1]*z[11]*z[15] + 512*z[1]*z[11]*z[16]*z[2] + 1024*z[1]*z[11]*z[16]*z[3] + 2048*z[1]*z[11]*z[16]*z[4] + 4096*z[1]*z[11]*z[16]*z[5] + 8192*z[1]*z[11]*z[16]*z[6] + 16384*z[1]*z[11]*z[16]*z[7] + 32768*z[1]*z[11]*z[16]*z[8] - 65280*z[1]*z[11]*z[16] + 4*z[1]*z[11]*z[2]*z[9] - 1020*z[1]*z[11]*z[2] + 8*z[1]*z[11]*z[3]*z[9] - 2040*z[1]*z[11]*z[3] + 16*z[1]*z[11]*z[4]*z[9] - 4080*z[1]*z[11]*z[4] + 32*z[1]*z[11]*z[5]*z[9] - 8160*z[1]*z[11]*z[5] + 64*z[1]*z[11]*z[6]*z[9] - 16320*z[1]*z[11]*z[6] + 128*z[1]*z[11]*z[7]*z[9] - 32640*z[1]*z[11]*z[7] + 256*z[1]*z[11]*z[8]*z[9] - 65280*z[1]*z[11]*z[8] - 510*z[1]*z[11]*z[9] + 130050*z[1]*z[11] + 128*z[1]*z[12]*z[13]*z[2] + 256*z[1]*z[12]*z[13]*z[3] + 512*z[1]*z[12]*z[13]*z[4] + 1024*z[1]*z[12]*z[13]*z[5] + 2048*z[1]*z[12]*z[13]*z[6] + 4096*z[1]*z[12]*z[13]*z[7] + 8192*z[1]*z[12]*z[13]*z[8] - 16320*z[1]*z[12]*z[13] + 256*z[1]*z[12]*z[14]*z[2] + 512*z[1]*z[12]*z[14]*z[3] + 1024*z[1]*z[12]*z[14]*z[4] + 2048*z[1]*z[12]*z[14]*z[5] + 4096*z[1]*z[12]*z[14]*z[6] + 8192*z[1]*z[12]*z[14]*z[7] + 16384*z[1]*z[12]*z[14]*z[8] - 32640*z[1]*z[12]*z[14] + 512*z[1]*z[12]*z[15]*z[2] + 1024*z[1]*z[12]*z[15]*z[3] + 2048*z[1]*z[12]*z[15]*z[4] + 4096*z[1]*z[12]*z[15]*z[5] + 8192*z[1]*z[12]*z[15]*z[6] + 16384*z[1]*z[12]*z[15]*z[7] + 32768*z[1]*z[12]*z[15]*z[8] - 65280*z[1]*z[12]*z[15] + 1024*z[1]*z[12]*z[16]*z[2] + 2048*z[1]*z[12]*z[16]*z[3] + 4096*z[1]*z[12]*z[16]*z[4] + 8192*z[1]*z[12]*z[16]*z[5] + 16384*z[1]*z[12]*z[16]*z[6] + 32768*z[1]*z[12]*z[16]*z[7] + 65536*z[1]*z[12]*z[16]*z[8] - 130560*z[1]*z[12]*z[16] + 8*z[1]*z[12]*z[2]*z[9] - 2040*z[1]*z[12]*z[2] + 16*z[1]*z[12]*z[3]*z[9] - 4080*z[1]*z[12]*z[3] + 32*z[1]*z[12]*z[4]*z[9] - 8160*z[1]*z[12]*z[4] + 64*z[1]*z[12]*z[5]*z[9] - 16320*z[1]*z[12]*z[5] + 128*z[1]*z[12]*z[6]*z[9] - 32640*z[1]*z[12]*z[6] + 256*z[1]*z[12]*z[7]*z[9] - 65280*z[1]*z[12]*z[7] + 512*z[1]*z[12]*z[8]*z[9] - 130560*z[1]*z[12]*z[8] - 1020*z[1]*z[12]*z[9] + 260100*z[1]*z[12] + 512*z[1]*z[13]*z[14]*z[2] + 1024*z[1]*z[13]*z[14]*z[3] + 2048*z[1]*z[13]*z[14]*z[4] + 4096*z[1]*z[13]*z[14]*z[5] + 8192*z[1]*z[13]*z[14]*z[6] + 16384*z[1]*z[13]*z[14]*z[7] + 32768*z[1]*z[13]*z[14]*z[8] - 65280*z[1]*z[13]*z[14] + 1024*z[1]*z[13]*z[15]*z[2] + 2048*z[1]*z[13]*z[15]*z[3] + 4096*z[1]*z[13]*z[15]*z[4] + 8192*z[1]*z[13]*z[15]*z[5] + 16384*z[1]*z[13]*z[15]*z[6] + 32768*z[1]*z[13]*z[15]*z[7] + 65536*z[1]*z[13]*z[15]*z[8] - 130560*z[1]*z[13]*z[15] + 2048*z[1]*z[13]*z[16]*z[2] + 4096*z[1]*z[13]*z[16]*z[3] + 8192*z[1]*z[13]*z[16]*z[4] + 16384*z[1]*z[13]*z[16]*z[5] + 32768*z[1]*z[13]*z[16]*z[6] + 65536*z[1]*z[13]*z[16]*z[7] + 131072*z[1]*z[13]*z[16]*z[8] - 261120*z[1]*z[13]*z[16] + 16*z[1]*z[13]*z[2]*z[9] - 4080*z[1]*z[13]*z[2] + 32*z[1]*z[13]*z[3]*z[9] - 8160*z[1]*z[13]*z[3] + 64*z[1]*z[13]*z[4]*z[9] - 16320*z[1]*z[13]*z[4] + 128*z[1]*z[13]*z[5]*z[9] - 32640*z[1]*z[13]*z[5] + 256*z[1]*z[13]*z[6]*z[9] - 65280*z[1]*z[13]*z[6] + 512*z[1]*z[13]*z[7]*z[9] - 130560*z[1]*z[13]*z[7] + 1024*z[1]*z[13]*z[8]*z[9] - 261120*z[1]*z[13]*z[8] - 2040*z[1]*z[13]*z[9] + 520200*z[1]*z[13] + 2048*z[1]*z[14]*z[15]*z[2] + 4096*z[1]*z[14]*z[15]*z[3] + 8192*z[1]*z[14]*z[15]*z[4] + 16384*z[1]*z[14]*z[15]*z[5] + 32768*z[1]*z[14]*z[15]*z[6] + 65536*z[1]*z[14]*z[15]*z[7] + 131072*z[1]*z[14]*z[15]*z[8] - 261120*z[1]*z[14]*z[15] + 4096*z[1]*z[14]*z[16]*z[2] + 8192*z[1]*z[14]*z[16]*z[3] + 16384*z[1]*z[14]*z[16]*z[4] + 32768*z[1]*z[14]*z[16]*z[5] + 65536*z[1]*z[14]*z[16]*z[6] + 131072*z[1]*z[14]*z[16]*z[7] + 262144*z[1]*z[14]*z[16]*z[8] - 522240*z[1]*z[14]*z[16] + 32*z[1]*z[14]*z[2]*z[9] - 8160*z[1]*z[14]*z[2] + 64*z[1]*z[14]*z[3]*z[9] - 16320*z[1]*z[14]*z[3] + 128*z[1]*z[14]*z[4]*z[9] - 32640*z[1]*z[14]*z[4] + 256*z[1]*z[14]*z[5]*z[9] - 65280*z[1]*z[14]*z[5] + 512*z[1]*z[14]*z[6]*z[9] - 130560*z[1]*z[14]*z[6] + 1024*z[1]*z[14]*z[7]*z[9] - 261120*z[1]*z[14]*z[7] + 2048*z[1]*z[14]*z[8]*z[9] - 522240*z[1]*z[14]*z[8] - 4080*z[1]*z[14]*z[9] + 1040400*z[1]*z[14] + 8192*z[1]*z[15]*z[16]*z[2] + 16384*z[1]*z[15]*z[16]*z[3] + 32768*z[1]*z[15]*z[16]*z[4] + 65536*z[1]*z[15]*z[16]*z[5] + 131072*z[1]*z[15]*z[16]*z[6] + 262144*z[1]*z[15]*z[16]*z[7] + 524288*z[1]*z[15]*z[16]*z[8] - 1044480*z[1]*z[15]*z[16] + 64*z[1]*z[15]*z[2]*z[9] - 16320*z[1]*z[15]*z[2] + 128*z[1]*z[15]*z[3]*z[9] - 32640*z[1]*z[15]*z[3] + 256*z[1]*z[15]*z[4]*z[9] - 65280*z[1]*z[15]*z[4] + 512*z[1]*z[15]*z[5]*z[9] - 130560*z[1]*z[15]*z[5] + 1024*z[1]*z[15]*z[6]*z[9] - 261120*z[1]*z[15]*z[6] + 2048*z[1]*z[15]*z[7]*z[9] - 522240*z[1]*z[15]*z[7] + 4096*z[1]*z[15]*z[8]*z[9] - 1044480*z[1]*z[15]*z[8] - 8160*z[1]*z[15]*z[9] + 2080800*z[1]*z[15] + 128*z[1]*z[16]*z[2]*z[9] - 32640*z[1]*z[16]*z[2] + 256*z[1]*z[16]*z[3]*z[9] - 65280*z[1]*z[16]*z[3] + 512*z[1]*z[16]*z[4]*z[9] - 130560*z[1]*z[16]*z[4] + 1024*z[1]*z[16]*z[5]*z[9] - 261120*z[1]*z[16]*z[5] + 2048*z[1]*z[16]*z[6]*z[9] - 522240*z[1]*z[16]*z[6] + 4096*z[1]*z[16]*z[7]*z[9] - 1044480*z[1]*z[16]*z[7] + 8192*z[1]*z[16]*z[8]*z[9] - 2088960*z[1]*z[16]*z[8] - 16320*z[1]*z[16]*z[9] + 4161600*z[1]*z[16] + 96*z[1]*z[2]*z[3]*z[4] + 192*z[1]*z[2]*z[3]*z[5] + 384*z[1]*z[2]*z[3]*z[6] + 768*z[1]*z[2]*z[3]*z[7] + 1536*z[1]*z[2]*z[3]*z[8] - 3060*z[1]*z[2]*z[3] + 384*z[1]*z[2]*z[4]*z[5] + 768*z[1]*z[2]*z[4]*z[6] + 1536*z[1]*z[2]*z[4]*z[7] + 3072*z[1]*z[2]*z[4]*z[8] - 6120*z[1]*z[2]*z[4] + 1536*z[1]*z[2]*z[5]*z[6] + 3072*z[1]*z[2]*z[5]*z[7] + 6144*z[1]*z[2]*z[5]*z[8] - 12240*z[1]*z[2]*z[5] + 6144*z[1]*z[2]*z[6]*z[7] + 12288*z[1]*z[2]*z[6]*z[8] - 24480*z[1]*z[2]*z[6] + 24576*z[1]*z[2]*z[7]*z[8] - 48960*z[1]*z[2]*z[7] - 97920*z[1]*z[2]*z[8] - 255*z[1]*z[2]*z[9] + 173729*z[1]*z[2] + 768*z[1]*z[3]*z[4]*z[5] + 1536*z[1]*z[3]*z[4]*z[6] + 3072*z[1]*z[3]*z[4]*z[7] + 6144*z[1]*z[3]*z[4]*z[8] - 12240*z[1]*z[3]*z[4] + 3072*z[1]*z[3]*z[5]*z[6] + 6144*z[1]*z[3]*z[5]*z[7] + 12288*z[1]*z[3]*z[5]*z[8] - 24480*z[1]*z[3]*z[5] + 12288*z[1]*z[3]*z[6]*z[7] + 24576*z[1]*z[3]*z[6]*z[8] - 48960*z[1]*z[3]*z[6] + 49152*z[1]*z[3]*z[7]*z[8] - 97920*z[1]*z[3]*z[7] - 195840*z[1]*z[3]*z[8] - 510*z[1]*z[3]*z[9] + 347434*z[1]*z[3] + 6144*z[1]*z[4]*z[5]*z[6] + 12288*z[1]*z[4]*z[5]*z[7] + 24576*z[1]*z[4]*z[5]*z[8] - 48960*z[1]*z[4]*z[5] + 24576*z[1]*z[4]*z[6]*z[7] + 49152*z[1]*z[4]*z[6]*z[8] - 97920*z[1]*z[4]*z[6] + 98304*z[1]*z[4]*z[7]*z[8] - 195840*z[1]*z[4]*z[7] - 391680*z[1]*z[4]*z[8] - 1020*z[1]*z[4]*z[9] + 694676*z[1]*z[4] + 49152*z[1]*z[5]*z[6]*z[7] + 98304*z[1]*z[5]*z[6]*z[8] - 195840*z[1]*z[5]*z[6] + 196608*z[1]*z[5]*z[7]*z[8] - 391680*z[1]*z[5]*z[7] - 783360*z[1]*z[5]*z[8] - 2040*z[1]*z[5]*z[9] + 1387816*z[1]*z[5] + 393216*z[1]*z[6]*z[7]*z[8] - 783360*z[1]*z[6]*z[7] - 1566720*z[1]*z[6]*z[8] - 4080*z[1]*z[6]*z[9] + 2763344*z[1]*z[6] - 3133440*z[1]*z[7]*z[8] - 8160*z[1]*z[7]*z[9] + 5428384*z[1]*z[7] - 16320*z[1]*z[8]*z[9] + 10070336*z[1]*z[8] + 65025*z[1]*z[9]/2 - 13860270*z[1] + 1536*z[10]*z[11]*z[12]*z[13] + 3072*z[10]*z[11]*z[12]*z[14] + 6144*z[10]*z[11]*z[12]*z[15] + 12288*z[10]*z[11]*z[12]*z[16] + 96*z[10]*z[11]*z[12]*z[9] - 24480*z[10]*z[11]*z[12] + 6144*z[10]*z[11]*z[13]*z[14] + 12288*z[10]*z[11]*z[13]*z[15] + 24576*z[10]*z[11]*z[13]*z[16] + 192*z[10]*z[11]*z[13]*z[9] - 48960*z[10]*z[11]*z[13] + 24576*z[10]*z[11]*z[14]*z[15] + 49152*z[10]*z[11]*z[14]*z[16] + 384*z[10]*z[11]*z[14]*z[9] - 97920*z[10]*z[11]*z[14] + 98304*z[10]*z[11]*z[15]*z[16] + 768*z[10]*z[11]*z[15]*z[9] - 195840*z[10]*z[11]*z[15] + 1536*z[10]*z[11]*z[16]*z[9] - 391680*z[10]*z[11]*z[16] + 32*z[10]*z[11]*z[2]*z[3] + 64*z[10]*z[11]*z[2]*z[4] + 128*z[10]*z[11]*z[2]*z[5] + 256*z[10]*z[11]*z[2]*z[6] + 512*z[10]*z[11]*z[2]*z[7] + 1024*z[10]*z[11]*z[2]*z[8] - 2040*z[10]*z[11]*z[2] + 128*z[10]*z[11]*z[3]*z[4] + 256*z[10]*z[11]*z[3]*z[5] + 512*z[10]*z[11]*z[3]*z[6] + 1024*z[10]*z[11]*z[3]*z[7] + 2048*z[10]*z[11]*z[3]*z[8] - 4080*z[10]*z[11]*z[3] + 512*z[10]*z[11]*z[4]*z[5] + 1024*z[10]*z[11]*z[4]*z[6] + 2048*z[10]*z[11]*z[4]*z[7] + 4096*z[10]*z[11]*z[4]*z[8] - 8160*z[10]*z[11]*z[4] + 2048*z[10]*z[11]*z[5]*z[6] + 4096*z[10]*z[11]*z[5]*z[7] + 8192*z[10]*z[11]*z[5]*z[8] - 16320*z[10]*z[11]*z[5] + 8192*z[10]*z[11]*z[6]*z[7] + 16384*z[10]*z[11]*z[6]*z[8] - 32640*z[10]*z[11]*z[6] + 32768*z[10]*z[11]*z[7]*z[8] - 65280*z[10]*z[11]*z[7] - 130560*z[10]*z[11]*z[8] - 3060*z[10]*z[11]*z[9] + 694856*z[10]*z[11] + 12288*z[10]*z[12]*z[13]*z[14] + 24576*z[10]*z[12]*z[13]*z[15] + 49152*z[10]*z[12]*z[13]*z[16] + 384*z[10]*z[12]*z[13]*z[9] - 97920*z[10]*z[12]*z[13] + 49152*z[10]*z[12]*z[14]*z[15] + 98304*z[10]*z[12]*z[14]*z[16] + 768*z[10]*z[12]*z[14]*z[9] - 195840*z[10]*z[12]*z[14] + 196608*z[10]*z[12]*z[15]*z[16] + 1536*z[10]*z[12]*z[15]*z[9] - 391680*z[10]*z[12]*z[15] + 3072*z[10]*z[12]*z[16]*z[9] - 783360*z[10]*z[12]*z[16] + 64*z[10]*z[12]*z[2]*z[3] + 128*z[10]*z[12]*z[2]*z[4] + 256*z[10]*z[12]*z[2]*z[5] + 512*z[10]*z[12]*z[2]*z[6] + 1024*z[10]*z[12]*z[2]*z[7] + 2048*z[10]*z[12]*z[2]*z[8] - 4080*z[10]*z[12]*z[2] + 256*z[10]*z[12]*z[3]*z[4] + 512*z[10]*z[12]*z[3]*z[5] + 1024*z[10]*z[12]*z[3]*z[6] + 2048*z[10]*z[12]*z[3]*z[7] + 4096*z[10]*z[12]*z[3]*z[8] - 8160*z[10]*z[12]*z[3] + 1024*z[10]*z[12]*z[4]*z[5] + 2048*z[10]*z[12]*z[4]*z[6] + 4096*z[10]*z[12]*z[4]*z[7] + 8192*z[10]*z[12]*z[4]*z[8] - 16320*z[10]*z[12]*z[4] + 4096*z[10]*z[12]*z[5]*z[6] + 8192*z[10]*z[12]*z[5]*z[7] + 16384*z[10]*z[12]*z[5]*z[8] - 32640*z[10]*z[12]*z[5] + 16384*z[10]*z[12]*z[6]*z[7] + 32768*z[10]*z[12]*z[6]*z[8] - 65280*z[10]*z[12]*z[6] + 65536*z[10]*z[12]*z[7]*z[8] - 130560*z[10]*z[12]*z[7] - 261120*z[10]*z[12]*z[8] - 6120*z[10]*z[12]*z[9] + 1389328*z[10]*z[12] + 98304*z[10]*z[13]*z[14]*z[15] + 196608*z[10]*z[13]*z[14]*z[16] + 1536*z[10]*z[13]*z[14]*z[9] - 391680*z[10]*z[13]*z[14] + 393216*z[10]*z[13]*z[15]*z[16] + 3072*z[10]*z[13]*z[15]*z[9] - 783360*z[10]*z[13]*z[15] + 6144*z[10]*z[13]*z[16]*z[9] - 1566720*z[10]*z[13]*z[16] + 128*z[10]*z[13]*z[2]*z[3] + 256*z[10]*z[13]*z[2]*z[4] + 512*z[10]*z[13]*z[2]*z[5] + 1024*z[10]*z[13]*z[2]*z[6] + 2048*z[10]*z[13]*z[2]*z[7] + 4096*z[10]*z[13]*z[2]*z[8] - 8160*z[10]*z[13]*z[2] + 512*z[10]*z[13]*z[3]*z[4] + 1024*z[10]*z[13]*z[3]*z[5] + 2048*z[10]*z[13]*z[3]*z[6] + 4096*z[10]*z[13]*z[3]*z[7] + 8192*z[10]*z[13]*z[3]*z[8] - 16320*z[10]*z[13]*z[3] + 2048*z[10]*z[13]*z[4]*z[5] + 4096*z[10]*z[13]*z[4]*z[6] + 8192*z[10]*z[13]*z[4]*z[7] + 16384*z[10]*z[13]*z[4]*z[8] - 32640*z[10]*z[13]*z[4] + 8192*z[10]*z[13]*z[5]*z[6] + 16384*z[10]*z[13]*z[5]*z[7] + 32768*z[10]*z[13]*z[5]*z[8] - 65280*z[10]*z[13]*z[5] + 32768*z[10]*z[13]*z[6]*z[7] + 65536*z[10]*z[13]*z[6]*z[8] - 130560*z[10]*z[13]*z[6] + 131072*z[10]*z[13]*z[7]*z[8] - 261120*z[10]*z[13]*z[7] - 522240*z[10]*z[13]*z[8] - 12240*z[10]*z[13]*z[9] + 2775584*z[10]*z[13] + 786432*z[10]*z[14]*z[15]*z[16] + 6144*z[10]*z[14]*z[15]*z[9] - 1566720*z[10]*z[14]*z[15] + 12288*z[10]*z[14]*z[16]*z[9] - 3133440*z[10]*z[14]*z[16] + 256*z[10]*z[14]*z[2]*z[3] + 512*z[10]*z[14]*z[2]*z[4] + 1024*z[10]*z[14]*z[2]*z[5] + 2048*z[10]*z[14]*z[2]*z[6] + 4096*z[10]*z[14]*z[2]*z[7] + 8192*z[10]*z[14]*z[2]*z[8] - 16320*z[10]*z[14]*z[2] + 1024*z[10]*z[14]*z[3]*z[4] + 2048*z[10]*z[14]*z[3]*z[5] + 4096*z[10]*z[14]*z[3]*z[6] + 8192*z[10]*z[14]*z[3]*z[7] + 16384*z[10]*z[14]*z[3]*z[8] - 32640*z[10]*z[14]*z[3] + 4096*z[10]*z[14]*z[4]*z[5] + 8192*z[10]*z[14]*z[4]*z[6] + 16384*z[10]*z[14]*z[4]*z[7] + 32768*z[10]*z[14]*z[4]*z[8] - 65280*z[10]*z[14]*z[4] + 16384*z[10]*z[14]*z[5]*z[6] + 32768*z[10]*z[14]*z[5]*z[7] + 65536*z[10]*z[14]*z[5]*z[8] - 130560*z[10]*z[14]*z[5] + 65536*z[10]*z[14]*z[6]*z[7] + 131072*z[10]*z[14]*z[6]*z[8] - 261120*z[10]*z[14]*z[6] + 262144*z[10]*z[14]*z[7]*z[8] - 522240*z[10]*z[14]*z[7] - 1044480*z[10]*z[14]*z[8] - 24480*z[10]*z[14]*z[9] + 5526592*z[10]*z[14] + 24576*z[10]*z[15]*z[16]*z[9] - 6266880*z[10]*z[15]*z[16] + 512*z[10]*z[15]*z[2]*z[3] + 1024*z[10]*z[15]*z[2]*z[4] + 2048*z[10]*z[15]*z[2]*z[5] + 4096*z[10]*z[15]*z[2]*z[6] + 8192*z[10]*z[15]*z[2]*z[7] + 16384*z[10]*z[15]*z[2]*z[8] - 32640*z[10]*z[15]*z[2] + 2048*z[10]*z[15]*z[3]*z[4] + 4096*z[10]*z[15]*z[3]*z[5] + 8192*z[10]*z[15]*z[3]*z[6] + 16384*z[10]*z[15]*z[3]*z[7] + 32768*z[10]*z[15]*z[3]*z[8] - 65280*z[10]*z[15]*z[3] + 8192*z[10]*z[15]*z[4]*z[5] + 16384*z[10]*z[15]*z[4]*z[6] + 32768*z[10]*z[15]*z[4]*z[7] + 65536*z[10]*z[15]*z[4]*z[8] - 130560*z[10]*z[15]*z[4] + 32768*z[10]*z[15]*z[5]*z[6] + 65536*z[10]*z[15]*z[5]*z[7] + 131072*z[10]*z[15]*z[5]*z[8] - 261120*z[10]*z[15]*z[5] + 131072*z[10]*z[15]*z[6]*z[7] + 262144*z[10]*z[15]*z[6]*z[8] - 522240*z[10]*z[15]*z[6] + 524288*z[10]*z[15]*z[7]*z[8] - 1044480*z[10]*z[15]*z[7] - 2088960*z[10]*z[15]*z[8] - 48960*z[10]*z[15]*z[9] + 10856576*z[10]*z[15] + 1024*z[10]*z[16]*z[2]*z[3] + 2048*z[10]*z[16]*z[2]*z[4] + 4096*z[10]*z[16]*z[2]*z[5] + 8192*z[10]*z[16]*z[2]*z[6] + 16384*z[10]*z[16]*z[2]*z[7] + 32768*z[10]*z[16]*z[2]*z[8] - 65280*z[10]*z[16]*z[2] + 4096*z[10]*z[16]*z[3]*z[4] + 8192*z[10]*z[16]*z[3]*z[5] + 16384*z[10]*z[16]*z[3]*z[6] + 32768*z[10]*z[16]*z[3]*z[7] + 65536*z[10]*z[16]*z[3]*z[8] - 130560*z[10]*z[16]*z[3] + 16384*z[10]*z[16]*z[4]*z[5] + 32768*z[10]*z[16]*z[4]*z[6] + 65536*z[10]*z[16]*z[4]*z[7] + 131072*z[10]*z[16]*z[4]*z[8] - 261120*z[10]*z[16]*z[4] + 65536*z[10]*z[16]*z[5]*z[6] + 131072*z[10]*z[16]*z[5]*z[7] + 262144*z[10]*z[16]*z[5]*z[8] - 522240*z[10]*z[16]*z[5] + 262144*z[10]*z[16]*z[6]*z[7] + 524288*z[10]*z[16]*z[6]*z[8] - 1044480*z[10]*z[16]*z[6] + 1048576*z[10]*z[16]*z[7]*z[8] - 2088960*z[10]*z[16]*z[7] - 4177920*z[10]*z[16]*z[8] - 97920*z[10]*z[16]*z[9] + 20140288*z[10]*z[16] + 8*z[10]*z[2]*z[3]*z[9] - 2040*z[10]*z[2]*z[3] + 16*z[10]*z[2]*z[4]*z[9] - 4080*z[10]*z[2]*z[4] + 32*z[10]*z[2]*z[5]*z[9] - 8160*z[10]*z[2]*z[5] + 64*z[10]*z[2]*z[6]*z[9] - 16320*z[10]*z[2]*z[6] + 128*z[10]*z[2]*z[7]*z[9] - 32640*z[10]*z[2]*z[7] + 256*z[10]*z[2]*z[8]*z[9] - 65280*z[10]*z[2]*z[8] - 510*z[10]*z[2]*z[9] + 130050*z[10]*z[2] + 32*z[10]*z[3]*z[4]*z[9] - 8160*z[10]*z[3]*z[4] + 64*z[10]*z[3]*z[5]*z[9] - 16320*z[10]*z[3]*z[5] + 128*z[10]*z[3]*z[6]*z[9] - 32640*z[10]*z[3]*z[6] + 256*z[10]*z[3]*z[7]*z[9] - 65280*z[10]*z[3]*z[7] + 512*z[10]*z[3]*z[8]*z[9] - 130560*z[10]*z[3]*z[8] - 1020*z[10]*z[3]*z[9] + 260100*z[10]*z[3] + 128*z[10]*z[4]*z[5]*z[9] - 32640*z[10]*z[4]*z[5] + 256*z[10]*z[4]*z[6]*z[9] - 65280*z[10]*z[4]*z[6] + 512*z[10]*z[4]*z[7]*z[9] - 130560*z[10]*z[4]*z[7] + 1024*z[10]*z[4]*z[8]*z[9] - 261120*z[10]*z[4]*z[8] - 2040*z[10]*z[4]*z[9] + 520200*z[10]*z[4] + 512*z[10]*z[5]*z[6]*z[9] - 130560*z[10]*z[5]*z[6] + 1024*z[10]*z[5]*z[7]*z[9] - 261120*z[10]*z[5]*z[7] + 2048*z[10]*z[5]*z[8]*z[9] - 522240*z[10]*z[5]*z[8] - 4080*z[10]*z[5]*z[9] + 1040400*z[10]*z[5] + 2048*z[10]*z[6]*z[7]*z[9] - 522240*z[10]*z[6]*z[7] + 4096*z[10]*z[6]*z[8]*z[9] - 1044480*z[10]*z[6]*z[8] - 8160*z[10]*z[6]*z[9] + 2080800*z[10]*z[6] + 8192*z[10]*z[7]*z[8]*z[9] - 2088960*z[10]*z[7]*z[8] - 16320*z[10]*z[7]*z[9] + 4161600*z[10]*z[7] - 32640*z[10]*z[8]*z[9] + 8323200*z[10]*z[8] + 173729*z[10]*z[9] - 27719775*z[10] + 24576*z[11]*z[12]*z[13]*z[14] + 49152*z[11]*z[12]*z[13]*z[15] + 98304*z[11]*z[12]*z[13]*z[16] + 768*z[11]*z[12]*z[13]*z[9] - 195840*z[11]*z[12]*z[13] + 98304*z[11]*z[12]*z[14]*z[15] + 196608*z[11]*z[12]*z[14]*z[16] + 1536*z[11]*z[12]*z[14]*z[9] - 391680*z[11]*z[12]*z[14] + 393216*z[11]*z[12]*z[15]*z[16] + 3072*z[11]*z[12]*z[15]*z[9] - 783360*z[11]*z[12]*z[15] + 6144*z[11]*z[12]*z[16]*z[9] - 1566720*z[11]*z[12]*z[16] + 128*z[11]*z[12]*z[2]*z[3] + 256*z[11]*z[12]*z[2]*z[4] + 512*z[11]*z[12]*z[2]*z[5] + 1024*z[11]*z[12]*z[2]*z[6] + 2048*z[11]*z[12]*z[2]*z[7] + 4096*z[11]*z[12]*z[2]*z[8] - 8160*z[11]*z[12]*z[2] + 512*z[11]*z[12]*z[3]*z[4] + 1024*z[11]*z[12]*z[3]*z[5] + 2048*z[11]*z[12]*z[3]*z[6] + 4096*z[11]*z[12]*z[3]*z[7] + 8192*z[11]*z[12]*z[3]*z[8] - 16320*z[11]*z[12]*z[3] + 2048*z[11]*z[12]*z[4]*z[5] + 4096*z[11]*z[12]*z[4]*z[6] + 8192*z[11]*z[12]*z[4]*z[7] + 16384*z[11]*z[12]*z[4]*z[8] - 32640*z[11]*z[12]*z[4] + 8192*z[11]*z[12]*z[5]*z[6] + 16384*z[11]*z[12]*z[5]*z[7] + 32768*z[11]*z[12]*z[5]*z[8] - 65280*z[11]*z[12]*z[5] + 32768*z[11]*z[12]*z[6]*z[7] + 65536*z[11]*z[12]*z[6]*z[8] - 130560*z[11]*z[12]*z[6] + 131072*z[11]*z[12]*z[7]*z[8] - 261120*z[11]*z[12]*z[7] - 522240*z[11]*z[12]*z[8] - 12240*z[11]*z[12]*z[9] + 2778464*z[11]*z[12] + 196608*z[11]*z[13]*z[14]*z[15] + 393216*z[11]*z[13]*z[14]*z[16] + 3072*z[11]*z[13]*z[14]*z[9] - 783360*z[11]*z[13]*z[14] + 786432*z[11]*z[13]*z[15]*z[16] + 6144*z[11]*z[13]*z[15]*z[9] - 1566720*z[11]*z[13]*z[15] + 12288*z[11]*z[13]*z[16]*z[9] - 3133440*z[11]*z[13]*z[16] + 256*z[11]*z[13]*z[2]*z[3] + 512*z[11]*z[13]*z[2]*z[4] + 1024*z[11]*z[13]*z[2]*z[5] + 2048*z[11]*z[13]*z[2]*z[6] + 4096*z[11]*z[13]*z[2]*z[7] + 8192*z[11]*z[13]*z[2]*z[8] - 16320*z[11]*z[13]*z[2] + 1024*z[11]*z[13]*z[3]*z[4] + 2048*z[11]*z[13]*z[3]*z[5] + 4096*z[11]*z[13]*z[3]*z[6] + 8192*z[11]*z[13]*z[3]*z[7] + 16384*z[11]*z[13]*z[3]*z[8] - 32640*z[11]*z[13]*z[3] + 4096*z[11]*z[13]*z[4]*z[5] + 8192*z[11]*z[13]*z[4]*z[6] + 16384*z[11]*z[13]*z[4]*z[7] + 32768*z[11]*z[13]*z[4]*z[8] - 65280*z[11]*z[13]*z[4] + 16384*z[11]*z[13]*z[5]*z[6] + 32768*z[11]*z[13]*z[5]*z[7] + 65536*z[11]*z[13]*z[5]*z[8] - 130560*z[11]*z[13]*z[5] + 65536*z[11]*z[13]*z[6]*z[7] + 131072*z[11]*z[13]*z[6]*z[8] - 261120*z[11]*z[13]*z[6] + 262144*z[11]*z[13]*z[7]*z[8] - 522240*z[11]*z[13]*z[7] - 1044480*z[11]*z[13]*z[8] - 24480*z[11]*z[13]*z[9] + 5550784*z[11]*z[13] + 1572864*z[11]*z[14]*z[15]*z[16] + 12288*z[11]*z[14]*z[15]*z[9] - 3133440*z[11]*z[14]*z[15] + 24576*z[11]*z[14]*z[16]*z[9] - 6266880*z[11]*z[14]*z[16] + 512*z[11]*z[14]*z[2]*z[3] + 1024*z[11]*z[14]*z[2]*z[4] + 2048*z[11]*z[14]*z[2]*z[5] + 4096*z[11]*z[14]*z[2]*z[6] + 8192*z[11]*z[14]*z[2]*z[7] + 16384*z[11]*z[14]*z[2]*z[8] - 32640*z[11]*z[14]*z[2] + 2048*z[11]*z[14]*z[3]*z[4] + 4096*z[11]*z[14]*z[3]*z[5] + 8192*z[11]*z[14]*z[3]*z[6] + 16384*z[11]*z[14]*z[3]*z[7] + 32768*z[11]*z[14]*z[3]*z[8] - 65280*z[11]*z[14]*z[3] + 8192*z[11]*z[14]*z[4]*z[5] + 16384*z[11]*z[14]*z[4]*z[6] + 32768*z[11]*z[14]*z[4]*z[7] + 65536*z[11]*z[14]*z[4]*z[8] - 130560*z[11]*z[14]*z[4] + 32768*z[11]*z[14]*z[5]*z[6] + 65536*z[11]*z[14]*z[5]*z[7] + 131072*z[11]*z[14]*z[5]*z[8] - 261120*z[11]*z[14]*z[5] + 131072*z[11]*z[14]*z[6]*z[7] + 262144*z[11]*z[14]*z[6]*z[8] - 522240*z[11]*z[14]*z[6] + 524288*z[11]*z[14]*z[7]*z[8] - 1044480*z[11]*z[14]*z[7] - 2088960*z[11]*z[14]*z[8] - 48960*z[11]*z[14]*z[9] + 11052416*z[11]*z[14] + 49152*z[11]*z[15]*z[16]*z[9] - 12533760*z[11]*z[15]*z[16] + 1024*z[11]*z[15]*z[2]*z[3] + 2048*z[11]*z[15]*z[2]*z[4] + 4096*z[11]*z[15]*z[2]*z[5] + 8192*z[11]*z[15]*z[2]*z[6] + 16384*z[11]*z[15]*z[2]*z[7] + 32768*z[11]*z[15]*z[2]*z[8] - 65280*z[11]*z[15]*z[2] + 4096*z[11]*z[15]*z[3]*z[4] + 8192*z[11]*z[15]*z[3]*z[5] + 16384*z[11]*z[15]*z[3]*z[6] + 32768*z[11]*z[15]*z[3]*z[7] + 65536*z[11]*z[15]*z[3]*z[8] - 130560*z[11]*z[15]*z[3] + 16384*z[11]*z[15]*z[4]*z[5] + 32768*z[11]*z[15]*z[4]*z[6] + 65536*z[11]*z[15]*z[4]*z[7] + 131072*z[11]*z[15]*z[4]*z[8] - 261120*z[11]*z[15]*z[4] + 65536*z[11]*z[15]*z[5]*z[6] + 131072*z[11]*z[15]*z[5]*z[7] + 262144*z[11]*z[15]*z[5]*z[8] - 522240*z[11]*z[15]*z[5] + 262144*z[11]*z[15]*z[6]*z[7] + 524288*z[11]*z[15]*z[6]*z[8] - 1044480*z[11]*z[15]*z[6] + 1048576*z[11]*z[15]*z[7]*z[8] - 2088960*z[11]*z[15]*z[7] - 4177920*z[11]*z[15]*z[8] - 97920*z[11]*z[15]*z[9] + 21711616*z[11]*z[15] + 2048*z[11]*z[16]*z[2]*z[3] + 4096*z[11]*z[16]*z[2]*z[4] + 8192*z[11]*z[16]*z[2]*z[5] + 16384*z[11]*z[16]*z[2]*z[6] + 32768*z[11]*z[16]*z[2]*z[7] + 65536*z[11]*z[16]*z[2]*z[8] - 130560*z[11]*z[16]*z[2] + 8192*z[11]*z[16]*z[3]*z[4] + 16384*z[11]*z[16]*z[3]*z[5] + 32768*z[11]*z[16]*z[3]*z[6] + 65536*z[11]*z[16]*z[3]*z[7] + 131072*z[11]*z[16]*z[3]*z[8] - 261120*z[11]*z[16]*z[3] + 32768*z[11]*z[16]*z[4]*z[5] + 65536*z[11]*z[16]*z[4]*z[6] + 131072*z[11]*z[16]*z[4]*z[7] + 262144*z[11]*z[16]*z[4]*z[8] - 522240*z[11]*z[16]*z[4] + 131072*z[11]*z[16]*z[5]*z[6] + 262144*z[11]*z[16]*z[5]*z[7] + 524288*z[11]*z[16]*z[5]*z[8] - 1044480*z[11]*z[16]*z[5] + 524288*z[11]*z[16]*z[6]*z[7] + 1048576*z[11]*z[16]*z[6]*z[8] - 2088960*z[11]*z[16]*z[6] + 2097152*z[11]*z[16]*z[7]*z[8] - 4177920*z[11]*z[16]*z[7] - 8355840*z[11]*z[16]*z[8] - 195840*z[11]*z[16]*z[9] + 40277504*z[11]*z[16] + 16*z[11]*z[2]*z[3]*z[9] - 4080*z[11]*z[2]*z[3] + 32*z[11]*z[2]*z[4]*z[9] - 8160*z[11]*z[2]*z[4] + 64*z[11]*z[2]*z[5]*z[9] - 16320*z[11]*z[2]*z[5] + 128*z[11]*z[2]*z[6]*z[9] - 32640*z[11]*z[2]*z[6] + 256*z[11]*z[2]*z[7]*z[9] - 65280*z[11]*z[2]*z[7] + 512*z[11]*z[2]*z[8]*z[9] - 130560*z[11]*z[2]*z[8] - 1020*z[11]*z[2]*z[9] + 260100*z[11]*z[2] + 64*z[11]*z[3]*z[4]*z[9] - 16320*z[11]*z[3]*z[4] + 128*z[11]*z[3]*z[5]*z[9] - 32640*z[11]*z[3]*z[5] + 256*z[11]*z[3]*z[6]*z[9] - 65280*z[11]*z[3]*z[6] + 512*z[11]*z[3]*z[7]*z[9] - 130560*z[11]*z[3]*z[7] + 1024*z[11]*z[3]*z[8]*z[9] - 261120*z[11]*z[3]*z[8] - 2040*z[11]*z[3]*z[9] + 520200*z[11]*z[3] + 256*z[11]*z[4]*z[5]*z[9] - 65280*z[11]*z[4]*z[5] + 512*z[11]*z[4]*z[6]*z[9] - 130560*z[11]*z[4]*z[6] + 1024*z[11]*z[4]*z[7]*z[9] - 261120*z[11]*z[4]*z[7] + 2048*z[11]*z[4]*z[8]*z[9] - 522240*z[11]*z[4]*z[8] - 4080*z[11]*z[4]*z[9] + 1040400*z[11]*z[4] + 1024*z[11]*z[5]*z[6]*z[9] - 261120*z[11]*z[5]*z[6] + 2048*z[11]*z[5]*z[7]*z[9] - 522240*z[11]*z[5]*z[7] + 4096*z[11]*z[5]*z[8]*z[9] - 1044480*z[11]*z[5]*z[8] - 8160*z[11]*z[5]*z[9] + 2080800*z[11]*z[5] + 4096*z[11]*z[6]*z[7]*z[9] - 1044480*z[11]*z[6]*z[7] + 8192*z[11]*z[6]*z[8]*z[9] - 2088960*z[11]*z[6]*z[8] - 16320*z[11]*z[6]*z[9] + 4161600*z[11]*z[6] + 16384*z[11]*z[7]*z[8]*z[9] - 4177920*z[11]*z[7]*z[8] - 32640*z[11]*z[7]*z[9] + 8323200*z[11]*z[7] - 65280*z[11]*z[8]*z[9] + 16646400*z[11]*z[8] + 347434*z[11]*z[9] - 55433430*z[11] + 393216*z[12]*z[13]*z[14]*z[15] + 786432*z[12]*z[13]*z[14]*z[16] + 6144*z[12]*z[13]*z[14]*z[9] - 1566720*z[12]*z[13]*z[14] + 1572864*z[12]*z[13]*z[15]*z[16] + 12288*z[12]*z[13]*z[15]*z[9] - 3133440*z[12]*z[13]*z[15] + 24576*z[12]*z[13]*z[16]*z[9] - 6266880*z[12]*z[13]*z[16] + 512*z[12]*z[13]*z[2]*z[3] + 1024*z[12]*z[13]*z[2]*z[4] + 2048*z[12]*z[13]*z[2]*z[5] + 4096*z[12]*z[13]*z[2]*z[6] + 8192*z[12]*z[13]*z[2]*z[7] + 16384*z[12]*z[13]*z[2]*z[8] - 32640*z[12]*z[13]*z[2] + 2048*z[12]*z[13]*z[3]*z[4] + 4096*z[12]*z[13]*z[3]*z[5] + 8192*z[12]*z[13]*z[3]*z[6] + 16384*z[12]*z[13]*z[3]*z[7] + 32768*z[12]*z[13]*z[3]*z[8] - 65280*z[12]*z[13]*z[3] + 8192*z[12]*z[13]*z[4]*z[5] + 16384*z[12]*z[13]*z[4]*z[6] + 32768*z[12]*z[13]*z[4]*z[7] + 65536*z[12]*z[13]*z[4]*z[8] - 130560*z[12]*z[13]*z[4] + 32768*z[12]*z[13]*z[5]*z[6] + 65536*z[12]*z[13]*z[5]*z[7] + 131072*z[12]*z[13]*z[5]*z[8] - 261120*z[12]*z[13]*z[5] + 131072*z[12]*z[13]*z[6]*z[7] + 262144*z[12]*z[13]*z[6]*z[8] - 522240*z[12]*z[13]*z[6] + 524288*z[12]*z[13]*z[7]*z[8] - 1044480*z[12]*z[13]*z[7] - 2088960*z[12]*z[13]*z[8] - 48960*z[12]*z[13]*z[9] + 11098496*z[12]*z[13] + 3145728*z[12]*z[14]*z[15]*z[16] + 24576*z[12]*z[14]*z[15]*z[9] - 6266880*z[12]*z[14]*z[15] + 49152*z[12]*z[14]*z[16]*z[9] - 12533760*z[12]*z[14]*z[16] + 1024*z[12]*z[14]*z[2]*z[3] + 2048*z[12]*z[14]*z[2]*z[4] + 4096*z[12]*z[14]*z[2]*z[5] + 8192*z[12]*z[14]*z[2]*z[6] + 16384*z[12]*z[14]*z[2]*z[7] + 32768*z[12]*z[14]*z[2]*z[8] - 65280*z[12]*z[14]*z[2] + 4096*z[12]*z[14]*z[3]*z[4] + 8192*z[12]*z[14]*z[3]*z[5] + 16384*z[12]*z[14]*z[3]*z[6] + 32768*z[12]*z[14]*z[3]*z[7] + 65536*z[12]*z[14]*z[3]*z[8] - 130560*z[12]*z[14]*z[3] + 16384*z[12]*z[14]*z[4]*z[5] + 32768*z[12]*z[14]*z[4]*z[6] + 65536*z[12]*z[14]*z[4]*z[7] + 131072*z[12]*z[14]*z[4]*z[8] - 261120*z[12]*z[14]*z[4] + 65536*z[12]*z[14]*z[5]*z[6] + 131072*z[12]*z[14]*z[5]*z[7] + 262144*z[12]*z[14]*z[5]*z[8] - 522240*z[12]*z[14]*z[5] + 262144*z[12]*z[14]*z[6]*z[7] + 524288*z[12]*z[14]*z[6]*z[8] - 1044480*z[12]*z[14]*z[6] + 1048576*z[12]*z[14]*z[7]*z[8] - 2088960*z[12]*z[14]*z[7] - 4177920*z[12]*z[14]*z[8] - 97920*z[12]*z[14]*z[9] + 22098688*z[12]*z[14] + 98304*z[12]*z[15]*z[16]*z[9] - 25067520*z[12]*z[15]*z[16] + 2048*z[12]*z[15]*z[2]*z[3] + 4096*z[12]*z[15]*z[2]*z[4] + 8192*z[12]*z[15]*z[2]*z[5] + 16384*z[12]*z[15]*z[2]*z[6] + 32768*z[12]*z[15]*z[2]*z[7] + 65536*z[12]*z[15]*z[2]*z[8] - 130560*z[12]*z[15]*z[2] + 8192*z[12]*z[15]*z[3]*z[4] + 16384*z[12]*z[15]*z[3]*z[5] + 32768*z[12]*z[15]*z[3]*z[6] + 65536*z[12]*z[15]*z[3]*z[7] + 131072*z[12]*z[15]*z[3]*z[8] - 261120*z[12]*z[15]*z[3] + 32768*z[12]*z[15]*z[4]*z[5] + 65536*z[12]*z[15]*z[4]*z[6] + 131072*z[12]*z[15]*z[4]*z[7] + 262144*z[12]*z[15]*z[4]*z[8] - 522240*z[12]*z[15]*z[4] + 131072*z[12]*z[15]*z[5]*z[6] + 262144*z[12]*z[15]*z[5]*z[7] + 524288*z[12]*z[15]*z[5]*z[8] - 1044480*z[12]*z[15]*z[5] + 524288*z[12]*z[15]*z[6]*z[7] + 1048576*z[12]*z[15]*z[6]*z[8] - 2088960*z[12]*z[15]*z[6] + 2097152*z[12]*z[15]*z[7]*z[8] - 4177920*z[12]*z[15]*z[7] - 8355840*z[12]*z[15]*z[8] - 195840*z[12]*z[15]*z[9] + 43410944*z[12]*z[15] + 4096*z[12]*z[16]*z[2]*z[3] + 8192*z[12]*z[16]*z[2]*z[4] + 16384*z[12]*z[16]*z[2]*z[5] + 32768*z[12]*z[16]*z[2]*z[6] + 65536*z[12]*z[16]*z[2]*z[7] + 131072*z[12]*z[16]*z[2]*z[8] - 261120*z[12]*z[16]*z[2] + 16384*z[12]*z[16]*z[3]*z[4] + 32768*z[12]*z[16]*z[3]*z[5] + 65536*z[12]*z[16]*z[3]*z[6] + 131072*z[12]*z[16]*z[3]*z[7] + 262144*z[12]*z[16]*z[3]*z[8] - 522240*z[12]*z[16]*z[3] + 65536*z[12]*z[16]*z[4]*z[5] + 131072*z[12]*z[16]*z[4]*z[6] + 262144*z[12]*z[16]*z[4]*z[7] + 524288*z[12]*z[16]*z[4]*z[8] - 1044480*z[12]*z[16]*z[4] + 262144*z[12]*z[16]*z[5]*z[6] + 524288*z[12]*z[16]*z[5]*z[7] + 1048576*z[12]*z[16]*z[5]*z[8] - 2088960*z[12]*z[16]*z[5] + 1048576*z[12]*z[16]*z[6]*z[7] + 2097152*z[12]*z[16]*z[6]*z[8] - 4177920*z[12]*z[16]*z[6] + 4194304*z[12]*z[16]*z[7]*z[8] - 8355840*z[12]*z[16]*z[7] - 16711680*z[12]*z[16]*z[8] - 391680*z[12]*z[16]*z[9] + 80530432*z[12]*z[16] + 32*z[12]*z[2]*z[3]*z[9] - 8160*z[12]*z[2]*z[3] + 64*z[12]*z[2]*z[4]*z[9] - 16320*z[12]*z[2]*z[4] + 128*z[12]*z[2]*z[5]*z[9] - 32640*z[12]*z[2]*z[5] + 256*z[12]*z[2]*z[6]*z[9] - 65280*z[12]*z[2]*z[6] + 512*z[12]*z[2]*z[7]*z[9] - 130560*z[12]*z[2]*z[7] + 1024*z[12]*z[2]*z[8]*z[9] - 261120*z[12]*z[2]*z[8] - 2040*z[12]*z[2]*z[9] + 520200*z[12]*z[2] + 128*z[12]*z[3]*z[4]*z[9] - 32640*z[12]*z[3]*z[4] + 256*z[12]*z[3]*z[5]*z[9] - 65280*z[12]*z[3]*z[5] + 512*z[12]*z[3]*z[6]*z[9] - 130560*z[12]*z[3]*z[6] + 1024*z[12]*z[3]*z[7]*z[9] - 261120*z[12]*z[3]*z[7] + 2048*z[12]*z[3]*z[8]*z[9] - 522240*z[12]*z[3]*z[8] - 4080*z[12]*z[3]*z[9] + 1040400*z[12]*z[3] + 512*z[12]*z[4]*z[5]*z[9] - 130560*z[12]*z[4]*z[5] + 1024*z[12]*z[4]*z[6]*z[9] - 261120*z[12]*z[4]*z[6] + 2048*z[12]*z[4]*z[7]*z[9] - 522240*z[12]*z[4]*z[7] + 4096*z[12]*z[4]*z[8]*z[9] - 1044480*z[12]*z[4]*z[8] - 8160*z[12]*z[4]*z[9] + 2080800*z[12]*z[4] + 2048*z[12]*z[5]*z[6]*z[9] - 522240*z[12]*z[5]*z[6] + 4096*z[12]*z[5]*z[7]*z[9] - 1044480*z[12]*z[5]*z[7] + 8192*z[12]*z[5]*z[8]*z[9] - 2088960*z[12]*z[5]*z[8] - 16320*z[12]*z[5]*z[9] + 4161600*z[12]*z[5] + 8192*z[12]*z[6]*z[7]*z[9] - 2088960*z[12]*z[6]*z[7] + 16384*z[12]*z[6]*z[8]*z[9] - 4177920*z[12]*z[6]*z[8] - 32640*z[12]*z[6]*z[9] + 8323200*z[12]*z[6] + 32768*z[12]*z[7]*z[8]*z[9] - 8355840*z[12]*z[7]*z[8] - 65280*z[12]*z[7]*z[9] + 16646400*z[12]*z[7] - 130560*z[12]*z[8]*z[9] + 33292800*z[12]*z[8] + 694676*z[12]*z[9] - 110817900*z[12] + 6291456*z[13]*z[14]*z[15]*z[16] + 49152*z[13]*z[14]*z[15]*z[9] - 12533760*z[13]*z[14]*z[15] + 98304*z[13]*z[14]*z[16]*z[9] - 25067520*z[13]*z[14]*z[16] + 2048*z[13]*z[14]*z[2]*z[3] + 4096*z[13]*z[14]*z[2]*z[4] + 8192*z[13]*z[14]*z[2]*z[5] + 16384*z[13]*z[14]*z[2]*z[6] + 32768*z[13]*z[14]*z[2]*z[7] + 65536*z[13]*z[14]*z[2]*z[8] - 130560*z[13]*z[14]*z[2] + 8192*z[13]*z[14]*z[3]*z[4] + 16384*z[13]*z[14]*z[3]*z[5] + 32768*z[13]*z[14]*z[3]*z[6] + 65536*z[13]*z[14]*z[3]*z[7] + 131072*z[13]*z[14]*z[3]*z[8] - 261120*z[13]*z[14]*z[3] + 32768*z[13]*z[14]*z[4]*z[5] + 65536*z[13]*z[14]*z[4]*z[6] + 131072*z[13]*z[14]*z[4]*z[7] + 262144*z[13]*z[14]*z[4]*z[8] - 522240*z[13]*z[14]*z[4] + 131072*z[13]*z[14]*z[5]*z[6] + 262144*z[13]*z[14]*z[5]*z[7] + 524288*z[13]*z[14]*z[5]*z[8] - 1044480*z[13]*z[14]*z[5] + 524288*z[13]*z[14]*z[6]*z[7] + 1048576*z[13]*z[14]*z[6]*z[8] - 2088960*z[13]*z[14]*z[6] + 2097152*z[13]*z[14]*z[7]*z[8] - 4177920*z[13]*z[14]*z[7] - 8355840*z[13]*z[14]*z[8] - 195840*z[13]*z[14]*z[9] + 44148224*z[13]*z[14] + 196608*z[13]*z[15]*z[16]*z[9] - 50135040*z[13]*z[15]*z[16] + 4096*z[13]*z[15]*z[2]*z[3] + 8192*z[13]*z[15]*z[2]*z[4] + 16384*z[13]*z[15]*z[2]*z[5] + 32768*z[13]*z[15]*z[2]*z[6] + 65536*z[13]*z[15]*z[2]*z[7] + 131072*z[13]*z[15]*z[2]*z[8] - 261120*z[13]*z[15]*z[2] + 16384*z[13]*z[15]*z[3]*z[4] + 32768*z[13]*z[15]*z[3]*z[5] + 65536*z[13]*z[15]*z[3]*z[6] + 131072*z[13]*z[15]*z[3]*z[7] + 262144*z[13]*z[15]*z[3]*z[8] - 522240*z[13]*z[15]*z[3] + 65536*z[13]*z[15]*z[4]*z[5] + 131072*z[13]*z[15]*z[4]*z[6] + 262144*z[13]*z[15]*z[4]*z[7] + 524288*z[13]*z[15]*z[4]*z[8] - 1044480*z[13]*z[15]*z[4] + 262144*z[13]*z[15]*z[5]*z[6] + 524288*z[13]*z[15]*z[5]*z[7] + 1048576*z[13]*z[15]*z[5]*z[8] - 2088960*z[13]*z[15]*z[5] + 1048576*z[13]*z[15]*z[6]*z[7] + 2097152*z[13]*z[15]*z[6]*z[8] - 4177920*z[13]*z[15]*z[6] + 4194304*z[13]*z[15]*z[7]*z[8] - 8355840*z[13]*z[15]*z[7] - 16711680*z[13]*z[15]*z[8] - 391680*z[13]*z[15]*z[9] + 86723584*z[13]*z[15] + 8192*z[13]*z[16]*z[2]*z[3] + 16384*z[13]*z[16]*z[2]*z[4] + 32768*z[13]*z[16]*z[2]*z[5] + 65536*z[13]*z[16]*z[2]*z[6] + 131072*z[13]*z[16]*z[2]*z[7] + 262144*z[13]*z[16]*z[2]*z[8] - 522240*z[13]*z[16]*z[2] + 32768*z[13]*z[16]*z[3]*z[4] + 65536*z[13]*z[16]*z[3]*z[5] + 131072*z[13]*z[16]*z[3]*z[6] + 262144*z[13]*z[16]*z[3]*z[7] + 524288*z[13]*z[16]*z[3]*z[8] - 1044480*z[13]*z[16]*z[3] + 131072*z[13]*z[16]*z[4]*z[5] + 262144*z[13]*z[16]*z[4]*z[6] + 524288*z[13]*z[16]*z[4]*z[7] + 1048576*z[13]*z[16]*z[4]*z[8] - 2088960*z[13]*z[16]*z[4] + 524288*z[13]*z[16]*z[5]*z[6] + 1048576*z[13]*z[16]*z[5]*z[7] + 2097152*z[13]*z[16]*z[5]*z[8] - 4177920*z[13]*z[16]*z[5] + 2097152*z[13]*z[16]*z[6]*z[7] + 4194304*z[13]*z[16]*z[6]*z[8] - 8355840*z[13]*z[16]*z[6] + 8388608*z[13]*z[16]*z[7]*z[8] - 16711680*z[13]*z[16]*z[7] - 33423360*z[13]*z[16]*z[8] - 783360*z[13]*z[16]*z[9] + 160864256*z[13]*z[16] + 64*z[13]*z[2]*z[3]*z[9] - 16320*z[13]*z[2]*z[3] + 128*z[13]*z[2]*z[4]*z[9] - 32640*z[13]*z[2]*z[4] + 256*z[13]*z[2]*z[5]*z[9] - 65280*z[13]*z[2]*z[5] + 512*z[13]*z[2]*z[6]*z[9] - 130560*z[13]*z[2]*z[6] + 1024*z[13]*z[2]*z[7]*z[9] - 261120*z[13]*z[2]*z[7] + 2048*z[13]*z[2]*z[8]*z[9] - 522240*z[13]*z[2]*z[8] - 4080*z[13]*z[2]*z[9] + 1040400*z[13]*z[2] + 256*z[13]*z[3]*z[4]*z[9] - 65280*z[13]*z[3]*z[4] + 512*z[13]*z[3]*z[5]*z[9] - 130560*z[13]*z[3]*z[5] + 1024*z[13]*z[3]*z[6]*z[9] - 261120*z[13]*z[3]*z[6] + 2048*z[13]*z[3]*z[7]*z[9] - 522240*z[13]*z[3]*z[7] + 4096*z[13]*z[3]*z[8]*z[9] - 1044480*z[13]*z[3]*z[8] - 8160*z[13]*z[3]*z[9] + 2080800*z[13]*z[3] + 1024*z[13]*z[4]*z[5]*z[9] - 261120*z[13]*z[4]*z[5] + 2048*z[13]*z[4]*z[6]*z[9] - 522240*z[13]*z[4]*z[6] + 4096*z[13]*z[4]*z[7]*z[9] - 1044480*z[13]*z[4]*z[7] + 8192*z[13]*z[4]*z[8]*z[9] - 2088960*z[13]*z[4]*z[8] - 16320*z[13]*z[4]*z[9] + 4161600*z[13]*z[4] + 4096*z[13]*z[5]*z[6]*z[9] - 1044480*z[13]*z[5]*z[6] + 8192*z[13]*z[5]*z[7]*z[9] - 2088960*z[13]*z[5]*z[7] + 16384*z[13]*z[5]*z[8]*z[9] - 4177920*z[13]*z[5]*z[8] - 32640*z[13]*z[5]*z[9] + 8323200*z[13]*z[5] + 16384*z[13]*z[6]*z[7]*z[9] - 4177920*z[13]*z[6]*z[7] + 32768*z[13]*z[6]*z[8]*z[9] - 8355840*z[13]*z[6]*z[8] - 65280*z[13]*z[6]*z[9] + 16646400*z[13]*z[6] + 65536*z[13]*z[7]*z[8]*z[9] - 16711680*z[13]*z[7]*z[8] - 130560*z[13]*z[7]*z[9] + 33292800*z[13]*z[7] - 261120*z[13]*z[8]*z[9] + 66585600*z[13]*z[8] + 1387816*z[13]*z[9] - 221244120*z[13] + 393216*z[14]*z[15]*z[16]*z[9] - 100270080*z[14]*z[15]*z[16] + 8192*z[14]*z[15]*z[2]*z[3] + 16384*z[14]*z[15]*z[2]*z[4] + 32768*z[14]*z[15]*z[2]*z[5] + 65536*z[14]*z[15]*z[2]*z[6] + 131072*z[14]*z[15]*z[2]*z[7] + 262144*z[14]*z[15]*z[2]*z[8] - 522240*z[14]*z[15]*z[2] + 32768*z[14]*z[15]*z[3]*z[4] + 65536*z[14]*z[15]*z[3]*z[5] + 131072*z[14]*z[15]*z[3]*z[6] + 262144*z[14]*z[15]*z[3]*z[7] + 524288*z[14]*z[15]*z[3]*z[8] - 1044480*z[14]*z[15]*z[3] + 131072*z[14]*z[15]*z[4]*z[5] + 262144*z[14]*z[15]*z[4]*z[6] + 524288*z[14]*z[15]*z[4]*z[7] + 1048576*z[14]*z[15]*z[4]*z[8] - 2088960*z[14]*z[15]*z[4] + 524288*z[14]*z[15]*z[5]*z[6] + 1048576*z[14]*z[15]*z[5]*z[7] + 2097152*z[14]*z[15]*z[5]*z[8] - 4177920*z[14]*z[15]*z[5] + 2097152*z[14]*z[15]*z[6]*z[7] + 4194304*z[14]*z[15]*z[6]*z[8] - 8355840*z[14]*z[15]*z[6] + 8388608*z[14]*z[15]*z[7]*z[8] - 16711680*z[14]*z[15]*z[7] - 33423360*z[14]*z[15]*z[8] - 783360*z[14]*z[15]*z[9] + 172660736*z[14]*z[15] + 16384*z[14]*z[16]*z[2]*z[3] + 32768*z[14]*z[16]*z[2]*z[4] + 65536*z[14]*z[16]*z[2]*z[5] + 131072*z[14]*z[16]*z[2]*z[6] + 262144*z[14]*z[16]*z[2]*z[7] + 524288*z[14]*z[16]*z[2]*z[8] - 1044480*z[14]*z[16]*z[2] + 65536*z[14]*z[16]*z[3]*z[4] + 131072*z[14]*z[16]*z[3]*z[5] + 262144*z[14]*z[16]*z[3]*z[6] + 524288*z[14]*z[16]*z[3]*z[7] + 1048576*z[14]*z[16]*z[3]*z[8] - 2088960*z[14]*z[16]*z[3] + 262144*z[14]*z[16]*z[4]*z[5] + 524288*z[14]*z[16]*z[4]*z[6] + 1048576*z[14]*z[16]*z[4]*z[7] + 2097152*z[14]*z[16]*z[4]*z[8] - 4177920*z[14]*z[16]*z[4] + 1048576*z[14]*z[16]*z[5]*z[6] + 2097152*z[14]*z[16]*z[5]*z[7] + 4194304*z[14]*z[16]*z[5]*z[8] - 8355840*z[14]*z[16]*z[5] + 4194304*z[14]*z[16]*z[6]*z[7] + 8388608*z[14]*z[16]*z[6]*z[8] - 16711680*z[14]*z[16]*z[6] + 16777216*z[14]*z[16]*z[7]*z[8] - 33423360*z[14]*z[16]*z[7] - 66846720*z[14]*z[16]*z[8] - 1566720*z[14]*z[16]*z[9] + 320155648*z[14]*z[16] + 128*z[14]*z[2]*z[3]*z[9] - 32640*z[14]*z[2]*z[3] + 256*z[14]*z[2]*z[4]*z[9] - 65280*z[14]*z[2]*z[4] + 512*z[14]*z[2]*z[5]*z[9] - 130560*z[14]*z[2]*z[5] + 1024*z[14]*z[2]*z[6]*z[9] - 261120*z[14]*z[2]*z[6] + 2048*z[14]*z[2]*z[7]*z[9] - 522240*z[14]*z[2]*z[7] + 4096*z[14]*z[2]*z[8]*z[9] - 1044480*z[14]*z[2]*z[8] - 8160*z[14]*z[2]*z[9] + 2080800*z[14]*z[2] + 512*z[14]*z[3]*z[4]*z[9] - 130560*z[14]*z[3]*z[4] + 1024*z[14]*z[3]*z[5]*z[9] - 261120*z[14]*z[3]*z[5] + 2048*z[14]*z[3]*z[6]*z[9] - 522240*z[14]*z[3]*z[6] + 4096*z[14]*z[3]*z[7]*z[9] - 1044480*z[14]*z[3]*z[7] + 8192*z[14]*z[3]*z[8]*z[9] - 2088960*z[14]*z[3]*z[8] - 16320*z[14]*z[3]*z[9] + 4161600*z[14]*z[3] + 2048*z[14]*z[4]*z[5]*z[9] - 522240*z[14]*z[4]*z[5] + 4096*z[14]*z[4]*z[6]*z[9] - 1044480*z[14]*z[4]*z[6] + 8192*z[14]*z[4]*z[7]*z[9] - 2088960*z[14]*z[4]*z[7] + 16384*z[14]*z[4]*z[8]*z[9] - 4177920*z[14]*z[4]*z[8] - 32640*z[14]*z[4]*z[9] + 8323200*z[14]*z[4] + 8192*z[14]*z[5]*z[6]*z[9] - 2088960*z[14]*z[5]*z[6] + 16384*z[14]*z[5]*z[7]*z[9] - 4177920*z[14]*z[5]*z[7] + 32768*z[14]*z[5]*z[8]*z[9] - 8355840*z[14]*z[5]*z[8] - 65280*z[14]*z[5]*z[9] + 16646400*z[14]*z[5] + 32768*z[14]*z[6]*z[7]*z[9] - 8355840*z[14]*z[6]*z[7] + 65536*z[14]*z[6]*z[8]*z[9] - 16711680*z[14]*z[6]*z[8] - 130560*z[14]*z[6]*z[9] + 33292800*z[14]*z[6] + 131072*z[14]*z[7]*z[8]*z[9] - 33423360*z[14]*z[7]*z[8] - 261120*z[14]*z[7]*z[9] + 66585600*z[14]*z[7] - 522240*z[14]*z[8]*z[9] + 133171200*z[14]*z[8] + 2763344*z[14]*z[9] - 439354800*z[14] + 32768*z[15]*z[16]*z[2]*z[3] + 65536*z[15]*z[16]*z[2]*z[4] + 131072*z[15]*z[16]*z[2]*z[5] + 262144*z[15]*z[16]*z[2]*z[6] + 524288*z[15]*z[16]*z[2]*z[7] + 1048576*z[15]*z[16]*z[2]*z[8] - 2088960*z[15]*z[16]*z[2] + 131072*z[15]*z[16]*z[3]*z[4] + 262144*z[15]*z[16]*z[3]*z[5] + 524288*z[15]*z[16]*z[3]*z[6] + 1048576*z[15]*z[16]*z[3]*z[7] + 2097152*z[15]*z[16]*z[3]*z[8] - 4177920*z[15]*z[16]*z[3] + 524288*z[15]*z[16]*z[4]*z[5] + 1048576*z[15]*z[16]*z[4]*z[6] + 2097152*z[15]*z[16]*z[4]*z[7] + 4194304*z[15]*z[16]*z[4]*z[8] - 8355840*z[15]*z[16]*z[4] + 2097152*z[15]*z[16]*z[5]*z[6] + 4194304*z[15]*z[16]*z[5]*z[7] + 8388608*z[15]*z[16]*z[5]*z[8] - 16711680*z[15]*z[16]*z[5] + 8388608*z[15]*z[16]*z[6]*z[7] + 16777216*z[15]*z[16]*z[6]*z[8] - 33423360*z[15]*z[16]*z[6] + 33554432*z[15]*z[16]*z[7]*z[8] - 66846720*z[15]*z[16]*z[7] - 133693440*z[15]*z[16]*z[8] - 3133440*z[15]*z[16]*z[9] + 627728384*z[15]*z[16] + 256*z[15]*z[2]*z[3]*z[9] - 65280*z[15]*z[2]*z[3] + 512*z[15]*z[2]*z[4]*z[9] - 130560*z[15]*z[2]*z[4] + 1024*z[15]*z[2]*z[5]*z[9] - 261120*z[15]*z[2]*z[5] + 2048*z[15]*z[2]*z[6]*z[9] - 522240*z[15]*z[2]*z[6] + 4096*z[15]*z[2]*z[7]*z[9] - 1044480*z[15]*z[2]*z[7] + 8192*z[15]*z[2]*z[8]*z[9] - 2088960*z[15]*z[2]*z[8] - 16320*z[15]*z[2]*z[9] + 4161600*z[15]*z[2] + 1024*z[15]*z[3]*z[4]*z[9] - 261120*z[15]*z[3]*z[4] + 2048*z[15]*z[3]*z[5]*z[9] - 522240*z[15]*z[3]*z[5] + 4096*z[15]*z[3]*z[6]*z[9] - 1044480*z[15]*z[3]*z[6] + 8192*z[15]*z[3]*z[7]*z[9] - 2088960*z[15]*z[3]*z[7] + 16384*z[15]*z[3]*z[8]*z[9] - 4177920*z[15]*z[3]*z[8] - 32640*z[15]*z[3]*z[9] + 8323200*z[15]*z[3] + 4096*z[15]*z[4]*z[5]*z[9] - 1044480*z[15]*z[4]*z[5] + 8192*z[15]*z[4]*z[6]*z[9] - 2088960*z[15]*z[4]*z[6] + 16384*z[15]*z[4]*z[7]*z[9] - 4177920*z[15]*z[4]*z[7] + 32768*z[15]*z[4]*z[8]*z[9] - 8355840*z[15]*z[4]*z[8] - 65280*z[15]*z[4]*z[9] + 16646400*z[15]*z[4] + 16384*z[15]*z[5]*z[6]*z[9] - 4177920*z[15]*z[5]*z[6] + 32768*z[15]*z[5]*z[7]*z[9] - 8355840*z[15]*z[5]*z[7] + 65536*z[15]*z[5]*z[8]*z[9] - 16711680*z[15]*z[5]*z[8] - 130560*z[15]*z[5]*z[9] + 33292800*z[15]*z[5] + 65536*z[15]*z[6]*z[7]*z[9] - 16711680*z[15]*z[6]*z[7] + 131072*z[15]*z[6]*z[8]*z[9] - 33423360*z[15]*z[6]*z[8] - 261120*z[15]*z[6]*z[9] + 66585600*z[15]*z[6] + 262144*z[15]*z[7]*z[8]*z[9] - 66846720*z[15]*z[7]*z[8] - 522240*z[15]*z[7]*z[9] + 133171200*z[15]*z[7] - 1044480*z[15]*z[8]*z[9] + 266342400*z[15]*z[8] + 5428384*z[15]*z[9] - 853642080*z[15] + 512*z[16]*z[2]*z[3]*z[9] - 130560*z[16]*z[2]*z[3] + 1024*z[16]*z[2]*z[4]*z[9] - 261120*z[16]*z[2]*z[4] + 2048*z[16]*z[2]*z[5]*z[9] - 522240*z[16]*z[2]*z[5] + 4096*z[16]*z[2]*z[6]*z[9] - 1044480*z[16]*z[2]*z[6] + 8192*z[16]*z[2]*z[7]*z[9] - 2088960*z[16]*z[2]*z[7] + 16384*z[16]*z[2]*z[8]*z[9] - 4177920*z[16]*z[2]*z[8] - 32640*z[16]*z[2]*z[9] + 8323200*z[16]*z[2] + 2048*z[16]*z[3]*z[4]*z[9] - 522240*z[16]*z[3]*z[4] + 4096*z[16]*z[3]*z[5]*z[9] - 1044480*z[16]*z[3]*z[5] + 8192*z[16]*z[3]*z[6]*z[9] - 2088960*z[16]*z[3]*z[6] + 16384*z[16]*z[3]*z[7]*z[9] - 4177920*z[16]*z[3]*z[7] + 32768*z[16]*z[3]*z[8]*z[9] - 8355840*z[16]*z[3]*z[8] - 65280*z[16]*z[3]*z[9] + 16646400*z[16]*z[3] + 8192*z[16]*z[4]*z[5]*z[9] - 2088960*z[16]*z[4]*z[5] + 16384*z[16]*z[4]*z[6]*z[9] - 4177920*z[16]*z[4]*z[6] + 32768*z[16]*z[4]*z[7]*z[9] - 8355840*z[16]*z[4]*z[7] + 65536*z[16]*z[4]*z[8]*z[9] - 16711680*z[16]*z[4]*z[8] - 130560*z[16]*z[4]*z[9] + 33292800*z[16]*z[4] + 32768*z[16]*z[5]*z[6]*z[9] - 8355840*z[16]*z[5]*z[6] + 65536*z[16]*z[5]*z[7]*z[9] - 16711680*z[16]*z[5]*z[7] + 131072*z[16]*z[5]*z[8]*z[9] - 33423360*z[16]*z[5]*z[8] - 261120*z[16]*z[5]*z[9] + 66585600*z[16]*z[5] + 131072*z[16]*z[6]*z[7]*z[9] - 33423360*z[16]*z[6]*z[7] + 262144*z[16]*z[6]*z[8]*z[9] - 66846720*z[16]*z[6]*z[8] - 522240*z[16]*z[6]*z[9] + 133171200*z[16]*z[6] + 524288*z[16]*z[7]*z[8]*z[9] - 133693440*z[16]*z[7]*z[8] - 1044480*z[16]*z[7]*z[9] + 266342400*z[16]*z[7] - 2088960*z[16]*z[8]*z[9] + 532684800*z[16]*z[8] + 10070336*z[16]*z[9] - 1506744000*z[16] + 1536*z[2]*z[3]*z[4]*z[5] + 3072*z[2]*z[3]*z[4]*z[6] + 6144*z[2]*z[3]*z[4]*z[7] + 12288*z[2]*z[3]*z[4]*z[8] - 24480*z[2]*z[3]*z[4] + 6144*z[2]*z[3]*z[5]*z[6] + 12288*z[2]*z[3]*z[5]*z[7] + 24576*z[2]*z[3]*z[5]*z[8] - 48960*z[2]*z[3]*z[5] + 24576*z[2]*z[3]*z[6]*z[7] + 49152*z[2]*z[3]*z[6]*z[8] - 97920*z[2]*z[3]*z[6] + 98304*z[2]*z[3]*z[7]*z[8] - 195840*z[2]*z[3]*z[7] - 391680*z[2]*z[3]*z[8] - 1020*z[2]*z[3]*z[9] + 694856*z[2]*z[3] + 12288*z[2]*z[4]*z[5]*z[6] + 24576*z[2]*z[4]*z[5]*z[7] + 49152*z[2]*z[4]*z[5]*z[8] - 97920*z[2]*z[4]*z[5] + 49152*z[2]*z[4]*z[6]*z[7] + 98304*z[2]*z[4]*z[6]*z[8] - 195840*z[2]*z[4]*z[6] + 196608*z[2]*z[4]*z[7]*z[8] - 391680*z[2]*z[4]*z[7] - 783360*z[2]*z[4]*z[8] - 2040*z[2]*z[4]*z[9] + 1389328*z[2]*z[4] + 98304*z[2]*z[5]*z[6]*z[7] + 196608*z[2]*z[5]*z[6]*z[8] - 391680*z[2]*z[5]*z[6] + 393216*z[2]*z[5]*z[7]*z[8] - 783360*z[2]*z[5]*z[7] - 1566720*z[2]*z[5]*z[8] - 4080*z[2]*z[5]*z[9] + 2775584*z[2]*z[5] + 786432*z[2]*z[6]*z[7]*z[8] - 1566720*z[2]*z[6]*z[7] - 3133440*z[2]*z[6]*z[8] - 8160*z[2]*z[6]*z[9] + 5526592*z[2]*z[6] - 6266880*z[2]*z[7]*z[8] - 16320*z[2]*z[7]*z[9] + 10856576*z[2]*z[7] - 32640*z[2]*z[8]*z[9] + 20140288*z[2]*z[8] + 65025*z[2]*z[9] - 27719775*z[2] + 24576*z[3]*z[4]*z[5]*z[6] + 49152*z[3]*z[4]*z[5]*z[7] + 98304*z[3]*z[4]*z[5]*z[8] - 195840*z[3]*z[4]*z[5] + 98304*z[3]*z[4]*z[6]*z[7] + 196608*z[3]*z[4]*z[6]*z[8] - 391680*z[3]*z[4]*z[6] + 393216*z[3]*z[4]*z[7]*z[8] - 783360*z[3]*z[4]*z[7] - 1566720*z[3]*z[4]*z[8] - 4080*z[3]*z[4]*z[9] + 2778464*z[3]*z[4] + 196608*z[3]*z[5]*z[6]*z[7] + 393216*z[3]*z[5]*z[6]*z[8] - 783360*z[3]*z[5]*z[6] + 786432*z[3]*z[5]*z[7]*z[8] - 1566720*z[3]*z[5]*z[7] - 3133440*z[3]*z[5]*z[8] - 8160*z[3]*z[5]*z[9] + 5550784*z[3]*z[5] + 1572864*z[3]*z[6]*z[7]*z[8] - 3133440*z[3]*z[6]*z[7] - 6266880*z[3]*z[6]*z[8] - 16320*z[3]*z[6]*z[9] + 11052416*z[3]*z[6] - 12533760*z[3]*z[7]*z[8] - 32640*z[3]*z[7]*z[9] + 21711616*z[3]*z[7] - 65280*z[3]*z[8]*z[9] + 40277504*z[3]*z[8] + 130050*z[3]*z[9] - 55433430*z[3] + 393216*z[4]*z[5]*z[6]*z[7] + 786432*z[4]*z[5]*z[6]*z[8] - 1566720*z[4]*z[5]*z[6] + 1572864*z[4]*z[5]*z[7]*z[8] - 3133440*z[4]*z[5]*z[7] - 6266880*z[4]*z[5]*z[8] - 16320*z[4]*z[5]*z[9] + 11098496*z[4]*z[5] + 3145728*z[4]*z[6]*z[7]*z[8] - 6266880*z[4]*z[6]*z[7] - 12533760*z[4]*z[6]*z[8] - 32640*z[4]*z[6]*z[9] + 22098688*z[4]*z[6] - 25067520*z[4]*z[7]*z[8] - 65280*z[4]*z[7]*z[9] + 43410944*z[4]*z[7] - 130560*z[4]*z[8]*z[9] + 80530432*z[4]*z[8] + 260100*z[4]*z[9] - 110817900*z[4] + 6291456*z[5]*z[6]*z[7]*z[8] - 12533760*z[5]*z[6]*z[7] - 25067520*z[5]*z[6]*z[8] - 65280*z[5]*z[6]*z[9] + 44148224*z[5]*z[6] - 50135040*z[5]*z[7]*z[8] - 130560*z[5]*z[7]*z[9] + 86723584*z[5]*z[7] - 261120*z[5]*z[8]*z[9] + 160864256*z[5]*z[8] + 520200*z[5]*z[9] - 221244120*z[5] - 100270080*z[6]*z[7]*z[8] - 261120*z[6]*z[7]*z[9] + 172660736*z[6]*z[7] - 522240*z[6]*z[8]*z[9] + 320155648*z[6]*z[8] + 1040400*z[6]*z[9] - 439354800*z[6] - 1044480*z[7]*z[8]*z[9] + 627728384*z[7]*z[8] + 2080800*z[7]*z[9] - 853642080*z[7] + 4161600*z[8]*z[9] - 1506744000*z[8] - 13860270*z[9] + 5288584809/2)

In [31]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 16  # 
p = 1  # QAOA depth

hamiltonian_less_255_2 = build_cost_hamiltonian_1(num_qubits, parse_hamiltonian_expr(paso3_D_equation_less_255_2))
final_circuit_less_255_2, result_less_255_2 = qaoa(num_qubits, hamiltonian_less_255_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [-0.04795822 -0.99187343]
Minimum expectation value: 1405674448.4921875


In [32]:
top_solutions_less_255_2 = find_best_bitstrings(final_circuit_less_255_2, hamiltonian_less_255_2)

Top 5 bitstrings:
Bitstring: 0000011000001001, Cost: -5288571812.0000, Count: 1
Bitstring: 0000101100001011, Cost: -5288527688.0000, Count: 1
Bitstring: 0010111000010111, Cost: -5281604644.0000, Count: 1
Bitstring: 0011001000010101, Cost: -5279952964.0000, Count: 1
Bitstring: 0001001001010011, Cost: -5236600708.0000, Count: 1


In [35]:
bitstring_to_pm1(top_solutions_less_255_2, evaluate_hamiltonian_less_255_2)

["Bitstring: ('0000011000001001', -5288571812.0, 1), Evaluated cost: 12996.0",
 "Bitstring: ('0000011000001001', -5288571812.0, 1), Evaluated cost: 57121.0",
 "Bitstring: ('0000011000001001', -5288571812.0, 1), Evaluated cost: 6980164.0",
 "Bitstring: ('0000011000001001', -5288571812.0, 1), Evaluated cost: 8631844.0",
 "Bitstring: ('0000011000001001', -5288571812.0, 1), Evaluated cost: 51984100.0"]

# Erdös-Strauss Diophantine equation $4xyz = n(xy+xz+yz)$ Case $n=4$ for positive integers

In [38]:
from qiskit.opflow import PauliOp, SummedOp
from qiskit.quantum_info import Pauli

def build_cost_hamiltonian_penalty(num_qubits, terms, lambda_penalty, msb_indices=None):

    zero_op = PauliOp(Pauli('I' * num_qubits)) * 0
    hamiltonian = SummedOp([zero_op])

    # Add the problem-specific cost terms
    for coeff, qubits_idx in terms:
        pauli_label = ['I'] * num_qubits
        for i in qubits_idx:
            pauli_label[i] = 'Z'
        pauli_str = ''.join(pauli_label)
        hamiltonian += PauliOp(Pauli(pauli_str)) * coeff

    # Add penalty terms for positivity (optional)
    if msb_indices:
        for msb in msb_indices:
            pauli_label = ['I'] * num_qubits
            pauli_label[msb] = 'Z'
            pauli_str = ''.join(pauli_label)

            # Penalty: λ * (1 - Z_i)/2 = λ/2 * (I - Z_i)
            hamiltonian += PauliOp(Pauli('I' * num_qubits)) * (lambda_penalty / 2)
            hamiltonian += PauliOp(Pauli(pauli_str)) * (-lambda_penalty / 2)

    return hamiltonian

In [39]:
def D_Erdös_Straus(x,y,z):
    return(4*x*y*z-4*(y*z+x*z+x*y))

In [40]:
paso1_D_Erdös_Straus_less_15_1 = substitute_with_global_binary_symbols(D_Erdös_Straus(x[1],x[2],x[3])**2, 4, base_name="b")
paso2_D_Erdös_Straus_less_15_1 = remove_variable_exponents(paso1_D_Erdös_Straus_less_15_1)
paso3_D_Erdös_Straus_less_15_1 = substitute_with_spin_variables(paso2_D_Erdös_Straus_less_15_1)

In [41]:
def evaluate_hamiltonian_D_Erdös_Straus_less_15(z):
    return(64*z[1]*z[10]*z[11]*z[2]*z[5]*z[6] + 128*z[1]*z[10]*z[11]*z[2]*z[5]*z[7] + 256*z[1]*z[10]*z[11]*z[2]*z[5]*z[8] - 416*z[1]*z[10]*z[11]*z[2]*z[5] + 256*z[1]*z[10]*z[11]*z[2]*z[6]*z[7] + 512*z[1]*z[10]*z[11]*z[2]*z[6]*z[8] - 832*z[1]*z[10]*z[11]*z[2]*z[6] + 1024*z[1]*z[10]*z[11]*z[2]*z[7]*z[8] - 1664*z[1]*z[10]*z[11]*z[2]*z[7] - 3328*z[1]*z[10]*z[11]*z[2]*z[8] + 4064*z[1]*z[10]*z[11]*z[2] + 128*z[1]*z[10]*z[11]*z[3]*z[5]*z[6] + 256*z[1]*z[10]*z[11]*z[3]*z[5]*z[7] + 512*z[1]*z[10]*z[11]*z[3]*z[5]*z[8] - 832*z[1]*z[10]*z[11]*z[3]*z[5] + 512*z[1]*z[10]*z[11]*z[3]*z[6]*z[7] + 1024*z[1]*z[10]*z[11]*z[3]*z[6]*z[8] - 1664*z[1]*z[10]*z[11]*z[3]*z[6] + 2048*z[1]*z[10]*z[11]*z[3]*z[7]*z[8] - 3328*z[1]*z[10]*z[11]*z[3]*z[7] - 6656*z[1]*z[10]*z[11]*z[3]*z[8] + 8128*z[1]*z[10]*z[11]*z[3] + 256*z[1]*z[10]*z[11]*z[4]*z[5]*z[6] + 512*z[1]*z[10]*z[11]*z[4]*z[5]*z[7] + 1024*z[1]*z[10]*z[11]*z[4]*z[5]*z[8] - 1664*z[1]*z[10]*z[11]*z[4]*z[5] + 1024*z[1]*z[10]*z[11]*z[4]*z[6]*z[7] + 2048*z[1]*z[10]*z[11]*z[4]*z[6]*z[8] - 3328*z[1]*z[10]*z[11]*z[4]*z[6] + 4096*z[1]*z[10]*z[11]*z[4]*z[7]*z[8] - 6656*z[1]*z[10]*z[11]*z[4]*z[7] - 13312*z[1]*z[10]*z[11]*z[4]*z[8] + 16256*z[1]*z[10]*z[11]*z[4] - 416*z[1]*z[10]*z[11]*z[5]*z[6] - 832*z[1]*z[10]*z[11]*z[5]*z[7] - 1664*z[1]*z[10]*z[11]*z[5]*z[8] + 2672*z[1]*z[10]*z[11]*z[5] - 1664*z[1]*z[10]*z[11]*z[6]*z[7] - 3328*z[1]*z[10]*z[11]*z[6]*z[8] + 5344*z[1]*z[10]*z[11]*z[6] - 6656*z[1]*z[10]*z[11]*z[7]*z[8] + 10688*z[1]*z[10]*z[11]*z[7] + 21376*z[1]*z[10]*z[11]*z[8] - 26000*z[1]*z[10]*z[11] + 128*z[1]*z[10]*z[12]*z[2]*z[5]*z[6] + 256*z[1]*z[10]*z[12]*z[2]*z[5]*z[7] + 512*z[1]*z[10]*z[12]*z[2]*z[5]*z[8] - 832*z[1]*z[10]*z[12]*z[2]*z[5] + 512*z[1]*z[10]*z[12]*z[2]*z[6]*z[7] + 1024*z[1]*z[10]*z[12]*z[2]*z[6]*z[8] - 1664*z[1]*z[10]*z[12]*z[2]*z[6] + 2048*z[1]*z[10]*z[12]*z[2]*z[7]*z[8] - 3328*z[1]*z[10]*z[12]*z[2]*z[7] - 6656*z[1]*z[10]*z[12]*z[2]*z[8] + 8128*z[1]*z[10]*z[12]*z[2] + 256*z[1]*z[10]*z[12]*z[3]*z[5]*z[6] + 512*z[1]*z[10]*z[12]*z[3]*z[5]*z[7] + 1024*z[1]*z[10]*z[12]*z[3]*z[5]*z[8] - 1664*z[1]*z[10]*z[12]*z[3]*z[5] + 1024*z[1]*z[10]*z[12]*z[3]*z[6]*z[7] + 2048*z[1]*z[10]*z[12]*z[3]*z[6]*z[8] - 3328*z[1]*z[10]*z[12]*z[3]*z[6] + 4096*z[1]*z[10]*z[12]*z[3]*z[7]*z[8] - 6656*z[1]*z[10]*z[12]*z[3]*z[7] - 13312*z[1]*z[10]*z[12]*z[3]*z[8] + 16256*z[1]*z[10]*z[12]*z[3] + 512*z[1]*z[10]*z[12]*z[4]*z[5]*z[6] + 1024*z[1]*z[10]*z[12]*z[4]*z[5]*z[7] + 2048*z[1]*z[10]*z[12]*z[4]*z[5]*z[8] - 3328*z[1]*z[10]*z[12]*z[4]*z[5] + 2048*z[1]*z[10]*z[12]*z[4]*z[6]*z[7] + 4096*z[1]*z[10]*z[12]*z[4]*z[6]*z[8] - 6656*z[1]*z[10]*z[12]*z[4]*z[6] + 8192*z[1]*z[10]*z[12]*z[4]*z[7]*z[8] - 13312*z[1]*z[10]*z[12]*z[4]*z[7] - 26624*z[1]*z[10]*z[12]*z[4]*z[8] + 32512*z[1]*z[10]*z[12]*z[4] - 832*z[1]*z[10]*z[12]*z[5]*z[6] - 1664*z[1]*z[10]*z[12]*z[5]*z[7] - 3328*z[1]*z[10]*z[12]*z[5]*z[8] + 5344*z[1]*z[10]*z[12]*z[5] - 3328*z[1]*z[10]*z[12]*z[6]*z[7] - 6656*z[1]*z[10]*z[12]*z[6]*z[8] + 10688*z[1]*z[10]*z[12]*z[6] - 13312*z[1]*z[10]*z[12]*z[7]*z[8] + 21376*z[1]*z[10]*z[12]*z[7] + 42752*z[1]*z[10]*z[12]*z[8] - 52000*z[1]*z[10]*z[12] + 16*z[1]*z[10]*z[2]*z[5]*z[6]*z[9] - 208*z[1]*z[10]*z[2]*z[5]*z[6] + 32*z[1]*z[10]*z[2]*z[5]*z[7]*z[9] - 416*z[1]*z[10]*z[2]*z[5]*z[7] + 64*z[1]*z[10]*z[2]*z[5]*z[8]*z[9] - 832*z[1]*z[10]*z[2]*z[5]*z[8] - 104*z[1]*z[10]*z[2]*z[5]*z[9] + 1336*z[1]*z[10]*z[2]*z[5] + 64*z[1]*z[10]*z[2]*z[6]*z[7]*z[9] - 832*z[1]*z[10]*z[2]*z[6]*z[7] + 128*z[1]*z[10]*z[2]*z[6]*z[8]*z[9] - 1664*z[1]*z[10]*z[2]*z[6]*z[8] - 208*z[1]*z[10]*z[2]*z[6]*z[9] + 2672*z[1]*z[10]*z[2]*z[6] + 256*z[1]*z[10]*z[2]*z[7]*z[8]*z[9] - 3328*z[1]*z[10]*z[2]*z[7]*z[8] - 416*z[1]*z[10]*z[2]*z[7]*z[9] + 5344*z[1]*z[10]*z[2]*z[7] - 832*z[1]*z[10]*z[2]*z[8]*z[9] + 10688*z[1]*z[10]*z[2]*z[8] + 1016*z[1]*z[10]*z[2]*z[9] - 13000*z[1]*z[10]*z[2] + 32*z[1]*z[10]*z[3]*z[5]*z[6]*z[9] - 416*z[1]*z[10]*z[3]*z[5]*z[6] + 64*z[1]*z[10]*z[3]*z[5]*z[7]*z[9] - 832*z[1]*z[10]*z[3]*z[5]*z[7] + 128*z[1]*z[10]*z[3]*z[5]*z[8]*z[9] - 1664*z[1]*z[10]*z[3]*z[5]*z[8] - 208*z[1]*z[10]*z[3]*z[5]*z[9] + 2672*z[1]*z[10]*z[3]*z[5] + 128*z[1]*z[10]*z[3]*z[6]*z[7]*z[9] - 1664*z[1]*z[10]*z[3]*z[6]*z[7] + 256*z[1]*z[10]*z[3]*z[6]*z[8]*z[9] - 3328*z[1]*z[10]*z[3]*z[6]*z[8] - 416*z[1]*z[10]*z[3]*z[6]*z[9] + 5344*z[1]*z[10]*z[3]*z[6] + 512*z[1]*z[10]*z[3]*z[7]*z[8]*z[9] - 6656*z[1]*z[10]*z[3]*z[7]*z[8] - 832*z[1]*z[10]*z[3]*z[7]*z[9] + 10688*z[1]*z[10]*z[3]*z[7] - 1664*z[1]*z[10]*z[3]*z[8]*z[9] + 21376*z[1]*z[10]*z[3]*z[8] + 2032*z[1]*z[10]*z[3]*z[9] - 26000*z[1]*z[10]*z[3] + 64*z[1]*z[10]*z[4]*z[5]*z[6]*z[9] - 832*z[1]*z[10]*z[4]*z[5]*z[6] + 128*z[1]*z[10]*z[4]*z[5]*z[7]*z[9] - 1664*z[1]*z[10]*z[4]*z[5]*z[7] + 256*z[1]*z[10]*z[4]*z[5]*z[8]*z[9] - 3328*z[1]*z[10]*z[4]*z[5]*z[8] - 416*z[1]*z[10]*z[4]*z[5]*z[9] + 5344*z[1]*z[10]*z[4]*z[5] + 256*z[1]*z[10]*z[4]*z[6]*z[7]*z[9] - 3328*z[1]*z[10]*z[4]*z[6]*z[7] + 512*z[1]*z[10]*z[4]*z[6]*z[8]*z[9] - 6656*z[1]*z[10]*z[4]*z[6]*z[8] - 832*z[1]*z[10]*z[4]*z[6]*z[9] + 10688*z[1]*z[10]*z[4]*z[6] + 1024*z[1]*z[10]*z[4]*z[7]*z[8]*z[9] - 13312*z[1]*z[10]*z[4]*z[7]*z[8] - 1664*z[1]*z[10]*z[4]*z[7]*z[9] + 21376*z[1]*z[10]*z[4]*z[7] - 3328*z[1]*z[10]*z[4]*z[8]*z[9] + 42752*z[1]*z[10]*z[4]*z[8] + 4064*z[1]*z[10]*z[4]*z[9] - 52000*z[1]*z[10]*z[4] - 104*z[1]*z[10]*z[5]*z[6]*z[9] + 1336*z[1]*z[10]*z[5]*z[6] - 208*z[1]*z[10]*z[5]*z[7]*z[9] + 2672*z[1]*z[10]*z[5]*z[7] - 416*z[1]*z[10]*z[5]*z[8]*z[9] + 5344*z[1]*z[10]*z[5]*z[8] + 668*z[1]*z[10]*z[5]*z[9] - 8460*z[1]*z[10]*z[5] - 416*z[1]*z[10]*z[6]*z[7]*z[9] + 5344*z[1]*z[10]*z[6]*z[7] - 832*z[1]*z[10]*z[6]*z[8]*z[9] + 10688*z[1]*z[10]*z[6]*z[8] + 1336*z[1]*z[10]*z[6]*z[9] - 16920*z[1]*z[10]*z[6] - 1664*z[1]*z[10]*z[7]*z[8]*z[9] + 21376*z[1]*z[10]*z[7]*z[8] + 2672*z[1]*z[10]*z[7]*z[9] - 33840*z[1]*z[10]*z[7] + 5344*z[1]*z[10]*z[8]*z[9] - 67680*z[1]*z[10]*z[8] - 6500*z[1]*z[10]*z[9] + 81940*z[1]*z[10] + 256*z[1]*z[11]*z[12]*z[2]*z[5]*z[6] + 512*z[1]*z[11]*z[12]*z[2]*z[5]*z[7] + 1024*z[1]*z[11]*z[12]*z[2]*z[5]*z[8] - 1664*z[1]*z[11]*z[12]*z[2]*z[5] + 1024*z[1]*z[11]*z[12]*z[2]*z[6]*z[7] + 2048*z[1]*z[11]*z[12]*z[2]*z[6]*z[8] - 3328*z[1]*z[11]*z[12]*z[2]*z[6] + 4096*z[1]*z[11]*z[12]*z[2]*z[7]*z[8] - 6656*z[1]*z[11]*z[12]*z[2]*z[7] - 13312*z[1]*z[11]*z[12]*z[2]*z[8] + 16256*z[1]*z[11]*z[12]*z[2] + 512*z[1]*z[11]*z[12]*z[3]*z[5]*z[6] + 1024*z[1]*z[11]*z[12]*z[3]*z[5]*z[7] + 2048*z[1]*z[11]*z[12]*z[3]*z[5]*z[8] - 3328*z[1]*z[11]*z[12]*z[3]*z[5] + 2048*z[1]*z[11]*z[12]*z[3]*z[6]*z[7] + 4096*z[1]*z[11]*z[12]*z[3]*z[6]*z[8] - 6656*z[1]*z[11]*z[12]*z[3]*z[6] + 8192*z[1]*z[11]*z[12]*z[3]*z[7]*z[8] - 13312*z[1]*z[11]*z[12]*z[3]*z[7] - 26624*z[1]*z[11]*z[12]*z[3]*z[8] + 32512*z[1]*z[11]*z[12]*z[3] + 1024*z[1]*z[11]*z[12]*z[4]*z[5]*z[6] + 2048*z[1]*z[11]*z[12]*z[4]*z[5]*z[7] + 4096*z[1]*z[11]*z[12]*z[4]*z[5]*z[8] - 6656*z[1]*z[11]*z[12]*z[4]*z[5] + 4096*z[1]*z[11]*z[12]*z[4]*z[6]*z[7] + 8192*z[1]*z[11]*z[12]*z[4]*z[6]*z[8] - 13312*z[1]*z[11]*z[12]*z[4]*z[6] + 16384*z[1]*z[11]*z[12]*z[4]*z[7]*z[8] - 26624*z[1]*z[11]*z[12]*z[4]*z[7] - 53248*z[1]*z[11]*z[12]*z[4]*z[8] + 65024*z[1]*z[11]*z[12]*z[4] - 1664*z[1]*z[11]*z[12]*z[5]*z[6] - 3328*z[1]*z[11]*z[12]*z[5]*z[7] - 6656*z[1]*z[11]*z[12]*z[5]*z[8] + 10688*z[1]*z[11]*z[12]*z[5] - 6656*z[1]*z[11]*z[12]*z[6]*z[7] - 13312*z[1]*z[11]*z[12]*z[6]*z[8] + 21376*z[1]*z[11]*z[12]*z[6] - 26624*z[1]*z[11]*z[12]*z[7]*z[8] + 42752*z[1]*z[11]*z[12]*z[7] + 85504*z[1]*z[11]*z[12]*z[8] - 104000*z[1]*z[11]*z[12] + 32*z[1]*z[11]*z[2]*z[5]*z[6]*z[9] - 416*z[1]*z[11]*z[2]*z[5]*z[6] + 64*z[1]*z[11]*z[2]*z[5]*z[7]*z[9] - 832*z[1]*z[11]*z[2]*z[5]*z[7] + 128*z[1]*z[11]*z[2]*z[5]*z[8]*z[9] - 1664*z[1]*z[11]*z[2]*z[5]*z[8] - 208*z[1]*z[11]*z[2]*z[5]*z[9] + 2672*z[1]*z[11]*z[2]*z[5] + 128*z[1]*z[11]*z[2]*z[6]*z[7]*z[9] - 1664*z[1]*z[11]*z[2]*z[6]*z[7] + 256*z[1]*z[11]*z[2]*z[6]*z[8]*z[9] - 3328*z[1]*z[11]*z[2]*z[6]*z[8] - 416*z[1]*z[11]*z[2]*z[6]*z[9] + 5344*z[1]*z[11]*z[2]*z[6] + 512*z[1]*z[11]*z[2]*z[7]*z[8]*z[9] - 6656*z[1]*z[11]*z[2]*z[7]*z[8] - 832*z[1]*z[11]*z[2]*z[7]*z[9] + 10688*z[1]*z[11]*z[2]*z[7] - 1664*z[1]*z[11]*z[2]*z[8]*z[9] + 21376*z[1]*z[11]*z[2]*z[8] + 2032*z[1]*z[11]*z[2]*z[9] - 26000*z[1]*z[11]*z[2] + 64*z[1]*z[11]*z[3]*z[5]*z[6]*z[9] - 832*z[1]*z[11]*z[3]*z[5]*z[6] + 128*z[1]*z[11]*z[3]*z[5]*z[7]*z[9] - 1664*z[1]*z[11]*z[3]*z[5]*z[7] + 256*z[1]*z[11]*z[3]*z[5]*z[8]*z[9] - 3328*z[1]*z[11]*z[3]*z[5]*z[8] - 416*z[1]*z[11]*z[3]*z[5]*z[9] + 5344*z[1]*z[11]*z[3]*z[5] + 256*z[1]*z[11]*z[3]*z[6]*z[7]*z[9] - 3328*z[1]*z[11]*z[3]*z[6]*z[7] + 512*z[1]*z[11]*z[3]*z[6]*z[8]*z[9] - 6656*z[1]*z[11]*z[3]*z[6]*z[8] - 832*z[1]*z[11]*z[3]*z[6]*z[9] + 10688*z[1]*z[11]*z[3]*z[6] + 1024*z[1]*z[11]*z[3]*z[7]*z[8]*z[9] - 13312*z[1]*z[11]*z[3]*z[7]*z[8] - 1664*z[1]*z[11]*z[3]*z[7]*z[9] + 21376*z[1]*z[11]*z[3]*z[7] - 3328*z[1]*z[11]*z[3]*z[8]*z[9] + 42752*z[1]*z[11]*z[3]*z[8] + 4064*z[1]*z[11]*z[3]*z[9] - 52000*z[1]*z[11]*z[3] + 128*z[1]*z[11]*z[4]*z[5]*z[6]*z[9] - 1664*z[1]*z[11]*z[4]*z[5]*z[6] + 256*z[1]*z[11]*z[4]*z[5]*z[7]*z[9] - 3328*z[1]*z[11]*z[4]*z[5]*z[7] + 512*z[1]*z[11]*z[4]*z[5]*z[8]*z[9] - 6656*z[1]*z[11]*z[4]*z[5]*z[8] - 832*z[1]*z[11]*z[4]*z[5]*z[9] + 10688*z[1]*z[11]*z[4]*z[5] + 512*z[1]*z[11]*z[4]*z[6]*z[7]*z[9] - 6656*z[1]*z[11]*z[4]*z[6]*z[7] + 1024*z[1]*z[11]*z[4]*z[6]*z[8]*z[9] - 13312*z[1]*z[11]*z[4]*z[6]*z[8] - 1664*z[1]*z[11]*z[4]*z[6]*z[9] + 21376*z[1]*z[11]*z[4]*z[6] + 2048*z[1]*z[11]*z[4]*z[7]*z[8]*z[9] - 26624*z[1]*z[11]*z[4]*z[7]*z[8] - 3328*z[1]*z[11]*z[4]*z[7]*z[9] + 42752*z[1]*z[11]*z[4]*z[7] - 6656*z[1]*z[11]*z[4]*z[8]*z[9] + 85504*z[1]*z[11]*z[4]*z[8] + 8128*z[1]*z[11]*z[4]*z[9] - 104000*z[1]*z[11]*z[4] - 208*z[1]*z[11]*z[5]*z[6]*z[9] + 2672*z[1]*z[11]*z[5]*z[6] - 416*z[1]*z[11]*z[5]*z[7]*z[9] + 5344*z[1]*z[11]*z[5]*z[7] - 832*z[1]*z[11]*z[5]*z[8]*z[9] + 10688*z[1]*z[11]*z[5]*z[8] + 1336*z[1]*z[11]*z[5]*z[9] - 16920*z[1]*z[11]*z[5] - 832*z[1]*z[11]*z[6]*z[7]*z[9] + 10688*z[1]*z[11]*z[6]*z[7] - 1664*z[1]*z[11]*z[6]*z[8]*z[9] + 21376*z[1]*z[11]*z[6]*z[8] + 2672*z[1]*z[11]*z[6]*z[9] - 33840*z[1]*z[11]*z[6] - 3328*z[1]*z[11]*z[7]*z[8]*z[9] + 42752*z[1]*z[11]*z[7]*z[8] + 5344*z[1]*z[11]*z[7]*z[9] - 67680*z[1]*z[11]*z[7] + 10688*z[1]*z[11]*z[8]*z[9] - 135360*z[1]*z[11]*z[8] - 13000*z[1]*z[11]*z[9] + 163880*z[1]*z[11] + 64*z[1]*z[12]*z[2]*z[5]*z[6]*z[9] - 832*z[1]*z[12]*z[2]*z[5]*z[6] + 128*z[1]*z[12]*z[2]*z[5]*z[7]*z[9] - 1664*z[1]*z[12]*z[2]*z[5]*z[7] + 256*z[1]*z[12]*z[2]*z[5]*z[8]*z[9] - 3328*z[1]*z[12]*z[2]*z[5]*z[8] - 416*z[1]*z[12]*z[2]*z[5]*z[9] + 5344*z[1]*z[12]*z[2]*z[5] + 256*z[1]*z[12]*z[2]*z[6]*z[7]*z[9] - 3328*z[1]*z[12]*z[2]*z[6]*z[7] + 512*z[1]*z[12]*z[2]*z[6]*z[8]*z[9] - 6656*z[1]*z[12]*z[2]*z[6]*z[8] - 832*z[1]*z[12]*z[2]*z[6]*z[9] + 10688*z[1]*z[12]*z[2]*z[6] + 1024*z[1]*z[12]*z[2]*z[7]*z[8]*z[9] - 13312*z[1]*z[12]*z[2]*z[7]*z[8] - 1664*z[1]*z[12]*z[2]*z[7]*z[9] + 21376*z[1]*z[12]*z[2]*z[7] - 3328*z[1]*z[12]*z[2]*z[8]*z[9] + 42752*z[1]*z[12]*z[2]*z[8] + 4064*z[1]*z[12]*z[2]*z[9] - 52000*z[1]*z[12]*z[2] + 128*z[1]*z[12]*z[3]*z[5]*z[6]*z[9] - 1664*z[1]*z[12]*z[3]*z[5]*z[6] + 256*z[1]*z[12]*z[3]*z[5]*z[7]*z[9] - 3328*z[1]*z[12]*z[3]*z[5]*z[7] + 512*z[1]*z[12]*z[3]*z[5]*z[8]*z[9] - 6656*z[1]*z[12]*z[3]*z[5]*z[8] - 832*z[1]*z[12]*z[3]*z[5]*z[9] + 10688*z[1]*z[12]*z[3]*z[5] + 512*z[1]*z[12]*z[3]*z[6]*z[7]*z[9] - 6656*z[1]*z[12]*z[3]*z[6]*z[7] + 1024*z[1]*z[12]*z[3]*z[6]*z[8]*z[9] - 13312*z[1]*z[12]*z[3]*z[6]*z[8] - 1664*z[1]*z[12]*z[3]*z[6]*z[9] + 21376*z[1]*z[12]*z[3]*z[6] + 2048*z[1]*z[12]*z[3]*z[7]*z[8]*z[9] - 26624*z[1]*z[12]*z[3]*z[7]*z[8] - 3328*z[1]*z[12]*z[3]*z[7]*z[9] + 42752*z[1]*z[12]*z[3]*z[7] - 6656*z[1]*z[12]*z[3]*z[8]*z[9] + 85504*z[1]*z[12]*z[3]*z[8] + 8128*z[1]*z[12]*z[3]*z[9] - 104000*z[1]*z[12]*z[3] + 256*z[1]*z[12]*z[4]*z[5]*z[6]*z[9] - 3328*z[1]*z[12]*z[4]*z[5]*z[6] + 512*z[1]*z[12]*z[4]*z[5]*z[7]*z[9] - 6656*z[1]*z[12]*z[4]*z[5]*z[7] + 1024*z[1]*z[12]*z[4]*z[5]*z[8]*z[9] - 13312*z[1]*z[12]*z[4]*z[5]*z[8] - 1664*z[1]*z[12]*z[4]*z[5]*z[9] + 21376*z[1]*z[12]*z[4]*z[5] + 1024*z[1]*z[12]*z[4]*z[6]*z[7]*z[9] - 13312*z[1]*z[12]*z[4]*z[6]*z[7] + 2048*z[1]*z[12]*z[4]*z[6]*z[8]*z[9] - 26624*z[1]*z[12]*z[4]*z[6]*z[8] - 3328*z[1]*z[12]*z[4]*z[6]*z[9] + 42752*z[1]*z[12]*z[4]*z[6] + 4096*z[1]*z[12]*z[4]*z[7]*z[8]*z[9] - 53248*z[1]*z[12]*z[4]*z[7]*z[8] - 6656*z[1]*z[12]*z[4]*z[7]*z[9] + 85504*z[1]*z[12]*z[4]*z[7] - 13312*z[1]*z[12]*z[4]*z[8]*z[9] + 171008*z[1]*z[12]*z[4]*z[8] + 16256*z[1]*z[12]*z[4]*z[9] - 208000*z[1]*z[12]*z[4] - 416*z[1]*z[12]*z[5]*z[6]*z[9] + 5344*z[1]*z[12]*z[5]*z[6] - 832*z[1]*z[12]*z[5]*z[7]*z[9] + 10688*z[1]*z[12]*z[5]*z[7] - 1664*z[1]*z[12]*z[5]*z[8]*z[9] + 21376*z[1]*z[12]*z[5]*z[8] + 2672*z[1]*z[12]*z[5]*z[9] - 33840*z[1]*z[12]*z[5] - 1664*z[1]*z[12]*z[6]*z[7]*z[9] + 21376*z[1]*z[12]*z[6]*z[7] - 3328*z[1]*z[12]*z[6]*z[8]*z[9] + 42752*z[1]*z[12]*z[6]*z[8] + 5344*z[1]*z[12]*z[6]*z[9] - 67680*z[1]*z[12]*z[6] - 6656*z[1]*z[12]*z[7]*z[8]*z[9] + 85504*z[1]*z[12]*z[7]*z[8] + 10688*z[1]*z[12]*z[7]*z[9] - 135360*z[1]*z[12]*z[7] + 21376*z[1]*z[12]*z[8]*z[9] - 270720*z[1]*z[12]*z[8] - 26000*z[1]*z[12]*z[9] + 327760*z[1]*z[12] - 104*z[1]*z[2]*z[5]*z[6]*z[9] + 1016*z[1]*z[2]*z[5]*z[6] - 208*z[1]*z[2]*z[5]*z[7]*z[9] + 2032*z[1]*z[2]*z[5]*z[7] - 416*z[1]*z[2]*z[5]*z[8]*z[9] + 4064*z[1]*z[2]*z[5]*z[8] + 668*z[1]*z[2]*z[5]*z[9] - 6500*z[1]*z[2]*z[5] - 416*z[1]*z[2]*z[6]*z[7]*z[9] + 4064*z[1]*z[2]*z[6]*z[7] - 832*z[1]*z[2]*z[6]*z[8]*z[9] + 8128*z[1]*z[2]*z[6]*z[8] + 1336*z[1]*z[2]*z[6]*z[9] - 13000*z[1]*z[2]*z[6] - 1664*z[1]*z[2]*z[7]*z[8]*z[9] + 16256*z[1]*z[2]*z[7]*z[8] + 2672*z[1]*z[2]*z[7]*z[9] - 26000*z[1]*z[2]*z[7] + 5344*z[1]*z[2]*z[8]*z[9] - 52000*z[1]*z[2]*z[8] - 6500*z[1]*z[2]*z[9] + 63180*z[1]*z[2] - 208*z[1]*z[3]*z[5]*z[6]*z[9] + 2032*z[1]*z[3]*z[5]*z[6] - 416*z[1]*z[3]*z[5]*z[7]*z[9] + 4064*z[1]*z[3]*z[5]*z[7] - 832*z[1]*z[3]*z[5]*z[8]*z[9] + 8128*z[1]*z[3]*z[5]*z[8] + 1336*z[1]*z[3]*z[5]*z[9] - 13000*z[1]*z[3]*z[5] - 832*z[1]*z[3]*z[6]*z[7]*z[9] + 8128*z[1]*z[3]*z[6]*z[7] - 1664*z[1]*z[3]*z[6]*z[8]*z[9] + 16256*z[1]*z[3]*z[6]*z[8] + 2672*z[1]*z[3]*z[6]*z[9] - 26000*z[1]*z[3]*z[6] - 3328*z[1]*z[3]*z[7]*z[8]*z[9] + 32512*z[1]*z[3]*z[7]*z[8] + 5344*z[1]*z[3]*z[7]*z[9] - 52000*z[1]*z[3]*z[7] + 10688*z[1]*z[3]*z[8]*z[9] - 104000*z[1]*z[3]*z[8] - 13000*z[1]*z[3]*z[9] + 126360*z[1]*z[3] - 416*z[1]*z[4]*z[5]*z[6]*z[9] + 4064*z[1]*z[4]*z[5]*z[6] - 832*z[1]*z[4]*z[5]*z[7]*z[9] + 8128*z[1]*z[4]*z[5]*z[7] - 1664*z[1]*z[4]*z[5]*z[8]*z[9] + 16256*z[1]*z[4]*z[5]*z[8] + 2672*z[1]*z[4]*z[5]*z[9] - 26000*z[1]*z[4]*z[5] - 1664*z[1]*z[4]*z[6]*z[7]*z[9] + 16256*z[1]*z[4]*z[6]*z[7] - 3328*z[1]*z[4]*z[6]*z[8]*z[9] + 32512*z[1]*z[4]*z[6]*z[8] + 5344*z[1]*z[4]*z[6]*z[9] - 52000*z[1]*z[4]*z[6] - 6656*z[1]*z[4]*z[7]*z[8]*z[9] + 65024*z[1]*z[4]*z[7]*z[8] + 10688*z[1]*z[4]*z[7]*z[9] - 104000*z[1]*z[4]*z[7] + 21376*z[1]*z[4]*z[8]*z[9] - 208000*z[1]*z[4]*z[8] - 26000*z[1]*z[4]*z[9] + 252720*z[1]*z[4] + 668*z[1]*z[5]*z[6]*z[9] - 6500*z[1]*z[5]*z[6] + 1336*z[1]*z[5]*z[7]*z[9] - 13000*z[1]*z[5]*z[7] + 2672*z[1]*z[5]*z[8]*z[9] - 26000*z[1]*z[5]*z[8] - 4230*z[1]*z[5]*z[9] + 40970*z[1]*z[5] + 2672*z[1]*z[6]*z[7]*z[9] - 26000*z[1]*z[6]*z[7] + 5344*z[1]*z[6]*z[8]*z[9] - 52000*z[1]*z[6]*z[8] - 8460*z[1]*z[6]*z[9] + 81940*z[1]*z[6] + 10688*z[1]*z[7]*z[8]*z[9] - 104000*z[1]*z[7]*z[8] - 16920*z[1]*z[7]*z[9] + 163880*z[1]*z[7] - 33840*z[1]*z[8]*z[9] + 327760*z[1]*z[8] + 40970*z[1]*z[9] - 396350*z[1] + 256*z[10]*z[11]*z[2]*z[3]*z[5]*z[6] + 512*z[10]*z[11]*z[2]*z[3]*z[5]*z[7] + 1024*z[10]*z[11]*z[2]*z[3]*z[5]*z[8] - 1664*z[10]*z[11]*z[2]*z[3]*z[5] + 1024*z[10]*z[11]*z[2]*z[3]*z[6]*z[7] + 2048*z[10]*z[11]*z[2]*z[3]*z[6]*z[8] - 3328*z[10]*z[11]*z[2]*z[3]*z[6] + 4096*z[10]*z[11]*z[2]*z[3]*z[7]*z[8] - 6656*z[10]*z[11]*z[2]*z[3]*z[7] - 13312*z[10]*z[11]*z[2]*z[3]*z[8] + 16256*z[10]*z[11]*z[2]*z[3] + 512*z[10]*z[11]*z[2]*z[4]*z[5]*z[6] + 1024*z[10]*z[11]*z[2]*z[4]*z[5]*z[7] + 2048*z[10]*z[11]*z[2]*z[4]*z[5]*z[8] - 3328*z[10]*z[11]*z[2]*z[4]*z[5] + 2048*z[10]*z[11]*z[2]*z[4]*z[6]*z[7] + 4096*z[10]*z[11]*z[2]*z[4]*z[6]*z[8] - 6656*z[10]*z[11]*z[2]*z[4]*z[6] + 8192*z[10]*z[11]*z[2]*z[4]*z[7]*z[8] - 13312*z[10]*z[11]*z[2]*z[4]*z[7] - 26624*z[10]*z[11]*z[2]*z[4]*z[8] + 32512*z[10]*z[11]*z[2]*z[4] - 832*z[10]*z[11]*z[2]*z[5]*z[6] - 1664*z[10]*z[11]*z[2]*z[5]*z[7] - 3328*z[10]*z[11]*z[2]*z[5]*z[8] + 5344*z[10]*z[11]*z[2]*z[5] - 3328*z[10]*z[11]*z[2]*z[6]*z[7] - 6656*z[10]*z[11]*z[2]*z[6]*z[8] + 10688*z[10]*z[11]*z[2]*z[6] - 13312*z[10]*z[11]*z[2]*z[7]*z[8] + 21376*z[10]*z[11]*z[2]*z[7] + 42752*z[10]*z[11]*z[2]*z[8] - 52000*z[10]*z[11]*z[2] + 1024*z[10]*z[11]*z[3]*z[4]*z[5]*z[6] + 2048*z[10]*z[11]*z[3]*z[4]*z[5]*z[7] + 4096*z[10]*z[11]*z[3]*z[4]*z[5]*z[8] - 6656*z[10]*z[11]*z[3]*z[4]*z[5] + 4096*z[10]*z[11]*z[3]*z[4]*z[6]*z[7] + 8192*z[10]*z[11]*z[3]*z[4]*z[6]*z[8] - 13312*z[10]*z[11]*z[3]*z[4]*z[6] + 16384*z[10]*z[11]*z[3]*z[4]*z[7]*z[8] - 26624*z[10]*z[11]*z[3]*z[4]*z[7] - 53248*z[10]*z[11]*z[3]*z[4]*z[8] + 65024*z[10]*z[11]*z[3]*z[4] - 1664*z[10]*z[11]*z[3]*z[5]*z[6] - 3328*z[10]*z[11]*z[3]*z[5]*z[7] - 6656*z[10]*z[11]*z[3]*z[5]*z[8] + 10688*z[10]*z[11]*z[3]*z[5] - 6656*z[10]*z[11]*z[3]*z[6]*z[7] - 13312*z[10]*z[11]*z[3]*z[6]*z[8] + 21376*z[10]*z[11]*z[3]*z[6] - 26624*z[10]*z[11]*z[3]*z[7]*z[8] + 42752*z[10]*z[11]*z[3]*z[7] + 85504*z[10]*z[11]*z[3]*z[8] - 104000*z[10]*z[11]*z[3] - 3328*z[10]*z[11]*z[4]*z[5]*z[6] - 6656*z[10]*z[11]*z[4]*z[5]*z[7] - 13312*z[10]*z[11]*z[4]*z[5]*z[8] + 21376*z[10]*z[11]*z[4]*z[5] - 13312*z[10]*z[11]*z[4]*z[6]*z[7] - 26624*z[10]*z[11]*z[4]*z[6]*z[8] + 42752*z[10]*z[11]*z[4]*z[6] - 53248*z[10]*z[11]*z[4]*z[7]*z[8] + 85504*z[10]*z[11]*z[4]*z[7] + 171008*z[10]*z[11]*z[4]*z[8] - 208000*z[10]*z[11]*z[4] + 4064*z[10]*z[11]*z[5]*z[6] + 8128*z[10]*z[11]*z[5]*z[7] + 16256*z[10]*z[11]*z[5]*z[8] - 26000*z[10]*z[11]*z[5] + 16256*z[10]*z[11]*z[6]*z[7] + 32512*z[10]*z[11]*z[6]*z[8] - 52000*z[10]*z[11]*z[6] + 65024*z[10]*z[11]*z[7]*z[8] - 104000*z[10]*z[11]*z[7] - 208000*z[10]*z[11]*z[8] + 252720*z[10]*z[11] + 512*z[10]*z[12]*z[2]*z[3]*z[5]*z[6] + 1024*z[10]*z[12]*z[2]*z[3]*z[5]*z[7] + 2048*z[10]*z[12]*z[2]*z[3]*z[5]*z[8] - 3328*z[10]*z[12]*z[2]*z[3]*z[5] + 2048*z[10]*z[12]*z[2]*z[3]*z[6]*z[7] + 4096*z[10]*z[12]*z[2]*z[3]*z[6]*z[8] - 6656*z[10]*z[12]*z[2]*z[3]*z[6] + 8192*z[10]*z[12]*z[2]*z[3]*z[7]*z[8] - 13312*z[10]*z[12]*z[2]*z[3]*z[7] - 26624*z[10]*z[12]*z[2]*z[3]*z[8] + 32512*z[10]*z[12]*z[2]*z[3] + 1024*z[10]*z[12]*z[2]*z[4]*z[5]*z[6] + 2048*z[10]*z[12]*z[2]*z[4]*z[5]*z[7] + 4096*z[10]*z[12]*z[2]*z[4]*z[5]*z[8] - 6656*z[10]*z[12]*z[2]*z[4]*z[5] + 4096*z[10]*z[12]*z[2]*z[4]*z[6]*z[7] + 8192*z[10]*z[12]*z[2]*z[4]*z[6]*z[8] - 13312*z[10]*z[12]*z[2]*z[4]*z[6] + 16384*z[10]*z[12]*z[2]*z[4]*z[7]*z[8] - 26624*z[10]*z[12]*z[2]*z[4]*z[7] - 53248*z[10]*z[12]*z[2]*z[4]*z[8] + 65024*z[10]*z[12]*z[2]*z[4] - 1664*z[10]*z[12]*z[2]*z[5]*z[6] - 3328*z[10]*z[12]*z[2]*z[5]*z[7] - 6656*z[10]*z[12]*z[2]*z[5]*z[8] + 10688*z[10]*z[12]*z[2]*z[5] - 6656*z[10]*z[12]*z[2]*z[6]*z[7] - 13312*z[10]*z[12]*z[2]*z[6]*z[8] + 21376*z[10]*z[12]*z[2]*z[6] - 26624*z[10]*z[12]*z[2]*z[7]*z[8] + 42752*z[10]*z[12]*z[2]*z[7] + 85504*z[10]*z[12]*z[2]*z[8] - 104000*z[10]*z[12]*z[2] + 2048*z[10]*z[12]*z[3]*z[4]*z[5]*z[6] + 4096*z[10]*z[12]*z[3]*z[4]*z[5]*z[7] + 8192*z[10]*z[12]*z[3]*z[4]*z[5]*z[8] - 13312*z[10]*z[12]*z[3]*z[4]*z[5] + 8192*z[10]*z[12]*z[3]*z[4]*z[6]*z[7] + 16384*z[10]*z[12]*z[3]*z[4]*z[6]*z[8] - 26624*z[10]*z[12]*z[3]*z[4]*z[6] + 32768*z[10]*z[12]*z[3]*z[4]*z[7]*z[8] - 53248*z[10]*z[12]*z[3]*z[4]*z[7] - 106496*z[10]*z[12]*z[3]*z[4]*z[8] + 130048*z[10]*z[12]*z[3]*z[4] - 3328*z[10]*z[12]*z[3]*z[5]*z[6] - 6656*z[10]*z[12]*z[3]*z[5]*z[7] - 13312*z[10]*z[12]*z[3]*z[5]*z[8] + 21376*z[10]*z[12]*z[3]*z[5] - 13312*z[10]*z[12]*z[3]*z[6]*z[7] - 26624*z[10]*z[12]*z[3]*z[6]*z[8] + 42752*z[10]*z[12]*z[3]*z[6] - 53248*z[10]*z[12]*z[3]*z[7]*z[8] + 85504*z[10]*z[12]*z[3]*z[7] + 171008*z[10]*z[12]*z[3]*z[8] - 208000*z[10]*z[12]*z[3] - 6656*z[10]*z[12]*z[4]*z[5]*z[6] - 13312*z[10]*z[12]*z[4]*z[5]*z[7] - 26624*z[10]*z[12]*z[4]*z[5]*z[8] + 42752*z[10]*z[12]*z[4]*z[5] - 26624*z[10]*z[12]*z[4]*z[6]*z[7] - 53248*z[10]*z[12]*z[4]*z[6]*z[8] + 85504*z[10]*z[12]*z[4]*z[6] - 106496*z[10]*z[12]*z[4]*z[7]*z[8] + 171008*z[10]*z[12]*z[4]*z[7] + 342016*z[10]*z[12]*z[4]*z[8] - 416000*z[10]*z[12]*z[4] + 8128*z[10]*z[12]*z[5]*z[6] + 16256*z[10]*z[12]*z[5]*z[7] + 32512*z[10]*z[12]*z[5]*z[8] - 52000*z[10]*z[12]*z[5] + 32512*z[10]*z[12]*z[6]*z[7] + 65024*z[10]*z[12]*z[6]*z[8] - 104000*z[10]*z[12]*z[6] + 130048*z[10]*z[12]*z[7]*z[8] - 208000*z[10]*z[12]*z[7] - 416000*z[10]*z[12]*z[8] + 505440*z[10]*z[12] + 64*z[10]*z[2]*z[3]*z[5]*z[6]*z[9] - 832*z[10]*z[2]*z[3]*z[5]*z[6] + 128*z[10]*z[2]*z[3]*z[5]*z[7]*z[9] - 1664*z[10]*z[2]*z[3]*z[5]*z[7] + 256*z[10]*z[2]*z[3]*z[5]*z[8]*z[9] - 3328*z[10]*z[2]*z[3]*z[5]*z[8] - 416*z[10]*z[2]*z[3]*z[5]*z[9] + 5344*z[10]*z[2]*z[3]*z[5] + 256*z[10]*z[2]*z[3]*z[6]*z[7]*z[9] - 3328*z[10]*z[2]*z[3]*z[6]*z[7] + 512*z[10]*z[2]*z[3]*z[6]*z[8]*z[9] - 6656*z[10]*z[2]*z[3]*z[6]*z[8] - 832*z[10]*z[2]*z[3]*z[6]*z[9] + 10688*z[10]*z[2]*z[3]*z[6] + 1024*z[10]*z[2]*z[3]*z[7]*z[8]*z[9] - 13312*z[10]*z[2]*z[3]*z[7]*z[8] - 1664*z[10]*z[2]*z[3]*z[7]*z[9] + 21376*z[10]*z[2]*z[3]*z[7] - 3328*z[10]*z[2]*z[3]*z[8]*z[9] + 42752*z[10]*z[2]*z[3]*z[8] + 4064*z[10]*z[2]*z[3]*z[9] - 52000*z[10]*z[2]*z[3] + 128*z[10]*z[2]*z[4]*z[5]*z[6]*z[9] - 1664*z[10]*z[2]*z[4]*z[5]*z[6] + 256*z[10]*z[2]*z[4]*z[5]*z[7]*z[9] - 3328*z[10]*z[2]*z[4]*z[5]*z[7] + 512*z[10]*z[2]*z[4]*z[5]*z[8]*z[9] - 6656*z[10]*z[2]*z[4]*z[5]*z[8] - 832*z[10]*z[2]*z[4]*z[5]*z[9] + 10688*z[10]*z[2]*z[4]*z[5] + 512*z[10]*z[2]*z[4]*z[6]*z[7]*z[9] - 6656*z[10]*z[2]*z[4]*z[6]*z[7] + 1024*z[10]*z[2]*z[4]*z[6]*z[8]*z[9] - 13312*z[10]*z[2]*z[4]*z[6]*z[8] - 1664*z[10]*z[2]*z[4]*z[6]*z[9] + 21376*z[10]*z[2]*z[4]*z[6] + 2048*z[10]*z[2]*z[4]*z[7]*z[8]*z[9] - 26624*z[10]*z[2]*z[4]*z[7]*z[8] - 3328*z[10]*z[2]*z[4]*z[7]*z[9] + 42752*z[10]*z[2]*z[4]*z[7] - 6656*z[10]*z[2]*z[4]*z[8]*z[9] + 85504*z[10]*z[2]*z[4]*z[8] + 8128*z[10]*z[2]*z[4]*z[9] - 104000*z[10]*z[2]*z[4] - 208*z[10]*z[2]*z[5]*z[6]*z[9] + 2672*z[10]*z[2]*z[5]*z[6] - 416*z[10]*z[2]*z[5]*z[7]*z[9] + 5344*z[10]*z[2]*z[5]*z[7] - 832*z[10]*z[2]*z[5]*z[8]*z[9] + 10688*z[10]*z[2]*z[5]*z[8] + 1336*z[10]*z[2]*z[5]*z[9] - 16920*z[10]*z[2]*z[5] - 832*z[10]*z[2]*z[6]*z[7]*z[9] + 10688*z[10]*z[2]*z[6]*z[7] - 1664*z[10]*z[2]*z[6]*z[8]*z[9] + 21376*z[10]*z[2]*z[6]*z[8] + 2672*z[10]*z[2]*z[6]*z[9] - 33840*z[10]*z[2]*z[6] - 3328*z[10]*z[2]*z[7]*z[8]*z[9] + 42752*z[10]*z[2]*z[7]*z[8] + 5344*z[10]*z[2]*z[7]*z[9] - 67680*z[10]*z[2]*z[7] + 10688*z[10]*z[2]*z[8]*z[9] - 135360*z[10]*z[2]*z[8] - 13000*z[10]*z[2]*z[9] + 163880*z[10]*z[2] + 256*z[10]*z[3]*z[4]*z[5]*z[6]*z[9] - 3328*z[10]*z[3]*z[4]*z[5]*z[6] + 512*z[10]*z[3]*z[4]*z[5]*z[7]*z[9] - 6656*z[10]*z[3]*z[4]*z[5]*z[7] + 1024*z[10]*z[3]*z[4]*z[5]*z[8]*z[9] - 13312*z[10]*z[3]*z[4]*z[5]*z[8] - 1664*z[10]*z[3]*z[4]*z[5]*z[9] + 21376*z[10]*z[3]*z[4]*z[5] + 1024*z[10]*z[3]*z[4]*z[6]*z[7]*z[9] - 13312*z[10]*z[3]*z[4]*z[6]*z[7] + 2048*z[10]*z[3]*z[4]*z[6]*z[8]*z[9] - 26624*z[10]*z[3]*z[4]*z[6]*z[8] - 3328*z[10]*z[3]*z[4]*z[6]*z[9] + 42752*z[10]*z[3]*z[4]*z[6] + 4096*z[10]*z[3]*z[4]*z[7]*z[8]*z[9] - 53248*z[10]*z[3]*z[4]*z[7]*z[8] - 6656*z[10]*z[3]*z[4]*z[7]*z[9] + 85504*z[10]*z[3]*z[4]*z[7] - 13312*z[10]*z[3]*z[4]*z[8]*z[9] + 171008*z[10]*z[3]*z[4]*z[8] + 16256*z[10]*z[3]*z[4]*z[9] - 208000*z[10]*z[3]*z[4] - 416*z[10]*z[3]*z[5]*z[6]*z[9] + 5344*z[10]*z[3]*z[5]*z[6] - 832*z[10]*z[3]*z[5]*z[7]*z[9] + 10688*z[10]*z[3]*z[5]*z[7] - 1664*z[10]*z[3]*z[5]*z[8]*z[9] + 21376*z[10]*z[3]*z[5]*z[8] + 2672*z[10]*z[3]*z[5]*z[9] - 33840*z[10]*z[3]*z[5] - 1664*z[10]*z[3]*z[6]*z[7]*z[9] + 21376*z[10]*z[3]*z[6]*z[7] - 3328*z[10]*z[3]*z[6]*z[8]*z[9] + 42752*z[10]*z[3]*z[6]*z[8] + 5344*z[10]*z[3]*z[6]*z[9] - 67680*z[10]*z[3]*z[6] - 6656*z[10]*z[3]*z[7]*z[8]*z[9] + 85504*z[10]*z[3]*z[7]*z[8] + 10688*z[10]*z[3]*z[7]*z[9] - 135360*z[10]*z[3]*z[7] + 21376*z[10]*z[3]*z[8]*z[9] - 270720*z[10]*z[3]*z[8] - 26000*z[10]*z[3]*z[9] + 327760*z[10]*z[3] - 832*z[10]*z[4]*z[5]*z[6]*z[9] + 10688*z[10]*z[4]*z[5]*z[6] - 1664*z[10]*z[4]*z[5]*z[7]*z[9] + 21376*z[10]*z[4]*z[5]*z[7] - 3328*z[10]*z[4]*z[5]*z[8]*z[9] + 42752*z[10]*z[4]*z[5]*z[8] + 5344*z[10]*z[4]*z[5]*z[9] - 67680*z[10]*z[4]*z[5] - 3328*z[10]*z[4]*z[6]*z[7]*z[9] + 42752*z[10]*z[4]*z[6]*z[7] - 6656*z[10]*z[4]*z[6]*z[8]*z[9] + 85504*z[10]*z[4]*z[6]*z[8] + 10688*z[10]*z[4]*z[6]*z[9] - 135360*z[10]*z[4]*z[6] - 13312*z[10]*z[4]*z[7]*z[8]*z[9] + 171008*z[10]*z[4]*z[7]*z[8] + 21376*z[10]*z[4]*z[7]*z[9] - 270720*z[10]*z[4]*z[7] + 42752*z[10]*z[4]*z[8]*z[9] - 541440*z[10]*z[4]*z[8] - 52000*z[10]*z[4]*z[9] + 655520*z[10]*z[4] + 1016*z[10]*z[5]*z[6]*z[9] - 13000*z[10]*z[5]*z[6] + 2032*z[10]*z[5]*z[7]*z[9] - 26000*z[10]*z[5]*z[7] + 4064*z[10]*z[5]*z[8]*z[9] - 52000*z[10]*z[5]*z[8] - 6500*z[10]*z[5]*z[9] + 81940*z[10]*z[5] + 4064*z[10]*z[6]*z[7]*z[9] - 52000*z[10]*z[6]*z[7] + 8128*z[10]*z[6]*z[8]*z[9] - 104000*z[10]*z[6]*z[8] - 13000*z[10]*z[6]*z[9] + 163880*z[10]*z[6] + 16256*z[10]*z[7]*z[8]*z[9] - 208000*z[10]*z[7]*z[8] - 26000*z[10]*z[7]*z[9] + 327760*z[10]*z[7] - 52000*z[10]*z[8]*z[9] + 655520*z[10]*z[8] + 63180*z[10]*z[9] - 792700*z[10] + 1024*z[11]*z[12]*z[2]*z[3]*z[5]*z[6] + 2048*z[11]*z[12]*z[2]*z[3]*z[5]*z[7] + 4096*z[11]*z[12]*z[2]*z[3]*z[5]*z[8] - 6656*z[11]*z[12]*z[2]*z[3]*z[5] + 4096*z[11]*z[12]*z[2]*z[3]*z[6]*z[7] + 8192*z[11]*z[12]*z[2]*z[3]*z[6]*z[8] - 13312*z[11]*z[12]*z[2]*z[3]*z[6] + 16384*z[11]*z[12]*z[2]*z[3]*z[7]*z[8] - 26624*z[11]*z[12]*z[2]*z[3]*z[7] - 53248*z[11]*z[12]*z[2]*z[3]*z[8] + 65024*z[11]*z[12]*z[2]*z[3] + 2048*z[11]*z[12]*z[2]*z[4]*z[5]*z[6] + 4096*z[11]*z[12]*z[2]*z[4]*z[5]*z[7] + 8192*z[11]*z[12]*z[2]*z[4]*z[5]*z[8] - 13312*z[11]*z[12]*z[2]*z[4]*z[5] + 8192*z[11]*z[12]*z[2]*z[4]*z[6]*z[7] + 16384*z[11]*z[12]*z[2]*z[4]*z[6]*z[8] - 26624*z[11]*z[12]*z[2]*z[4]*z[6] + 32768*z[11]*z[12]*z[2]*z[4]*z[7]*z[8] - 53248*z[11]*z[12]*z[2]*z[4]*z[7] - 106496*z[11]*z[12]*z[2]*z[4]*z[8] + 130048*z[11]*z[12]*z[2]*z[4] - 3328*z[11]*z[12]*z[2]*z[5]*z[6] - 6656*z[11]*z[12]*z[2]*z[5]*z[7] - 13312*z[11]*z[12]*z[2]*z[5]*z[8] + 21376*z[11]*z[12]*z[2]*z[5] - 13312*z[11]*z[12]*z[2]*z[6]*z[7] - 26624*z[11]*z[12]*z[2]*z[6]*z[8] + 42752*z[11]*z[12]*z[2]*z[6] - 53248*z[11]*z[12]*z[2]*z[7]*z[8] + 85504*z[11]*z[12]*z[2]*z[7] + 171008*z[11]*z[12]*z[2]*z[8] - 208000*z[11]*z[12]*z[2] + 4096*z[11]*z[12]*z[3]*z[4]*z[5]*z[6] + 8192*z[11]*z[12]*z[3]*z[4]*z[5]*z[7] + 16384*z[11]*z[12]*z[3]*z[4]*z[5]*z[8] - 26624*z[11]*z[12]*z[3]*z[4]*z[5] + 16384*z[11]*z[12]*z[3]*z[4]*z[6]*z[7] + 32768*z[11]*z[12]*z[3]*z[4]*z[6]*z[8] - 53248*z[11]*z[12]*z[3]*z[4]*z[6] + 65536*z[11]*z[12]*z[3]*z[4]*z[7]*z[8] - 106496*z[11]*z[12]*z[3]*z[4]*z[7] - 212992*z[11]*z[12]*z[3]*z[4]*z[8] + 260096*z[11]*z[12]*z[3]*z[4] - 6656*z[11]*z[12]*z[3]*z[5]*z[6] - 13312*z[11]*z[12]*z[3]*z[5]*z[7] - 26624*z[11]*z[12]*z[3]*z[5]*z[8] + 42752*z[11]*z[12]*z[3]*z[5] - 26624*z[11]*z[12]*z[3]*z[6]*z[7] - 53248*z[11]*z[12]*z[3]*z[6]*z[8] + 85504*z[11]*z[12]*z[3]*z[6] - 106496*z[11]*z[12]*z[3]*z[7]*z[8] + 171008*z[11]*z[12]*z[3]*z[7] + 342016*z[11]*z[12]*z[3]*z[8] - 416000*z[11]*z[12]*z[3] - 13312*z[11]*z[12]*z[4]*z[5]*z[6] - 26624*z[11]*z[12]*z[4]*z[5]*z[7] - 53248*z[11]*z[12]*z[4]*z[5]*z[8] + 85504*z[11]*z[12]*z[4]*z[5] - 53248*z[11]*z[12]*z[4]*z[6]*z[7] - 106496*z[11]*z[12]*z[4]*z[6]*z[8] + 171008*z[11]*z[12]*z[4]*z[6] - 212992*z[11]*z[12]*z[4]*z[7]*z[8] + 342016*z[11]*z[12]*z[4]*z[7] + 684032*z[11]*z[12]*z[4]*z[8] - 832000*z[11]*z[12]*z[4] + 16256*z[11]*z[12]*z[5]*z[6] + 32512*z[11]*z[12]*z[5]*z[7] + 65024*z[11]*z[12]*z[5]*z[8] - 104000*z[11]*z[12]*z[5] + 65024*z[11]*z[12]*z[6]*z[7] + 130048*z[11]*z[12]*z[6]*z[8] - 208000*z[11]*z[12]*z[6] + 260096*z[11]*z[12]*z[7]*z[8] - 416000*z[11]*z[12]*z[7] - 832000*z[11]*z[12]*z[8] + 1010880*z[11]*z[12] + 128*z[11]*z[2]*z[3]*z[5]*z[6]*z[9] - 1664*z[11]*z[2]*z[3]*z[5]*z[6] + 256*z[11]*z[2]*z[3]*z[5]*z[7]*z[9] - 3328*z[11]*z[2]*z[3]*z[5]*z[7] + 512*z[11]*z[2]*z[3]*z[5]*z[8]*z[9] - 6656*z[11]*z[2]*z[3]*z[5]*z[8] - 832*z[11]*z[2]*z[3]*z[5]*z[9] + 10688*z[11]*z[2]*z[3]*z[5] + 512*z[11]*z[2]*z[3]*z[6]*z[7]*z[9] - 6656*z[11]*z[2]*z[3]*z[6]*z[7] + 1024*z[11]*z[2]*z[3]*z[6]*z[8]*z[9] - 13312*z[11]*z[2]*z[3]*z[6]*z[8] - 1664*z[11]*z[2]*z[3]*z[6]*z[9] + 21376*z[11]*z[2]*z[3]*z[6] + 2048*z[11]*z[2]*z[3]*z[7]*z[8]*z[9] - 26624*z[11]*z[2]*z[3]*z[7]*z[8] - 3328*z[11]*z[2]*z[3]*z[7]*z[9] + 42752*z[11]*z[2]*z[3]*z[7] - 6656*z[11]*z[2]*z[3]*z[8]*z[9] + 85504*z[11]*z[2]*z[3]*z[8] + 8128*z[11]*z[2]*z[3]*z[9] - 104000*z[11]*z[2]*z[3] + 256*z[11]*z[2]*z[4]*z[5]*z[6]*z[9] - 3328*z[11]*z[2]*z[4]*z[5]*z[6] + 512*z[11]*z[2]*z[4]*z[5]*z[7]*z[9] - 6656*z[11]*z[2]*z[4]*z[5]*z[7] + 1024*z[11]*z[2]*z[4]*z[5]*z[8]*z[9] - 13312*z[11]*z[2]*z[4]*z[5]*z[8] - 1664*z[11]*z[2]*z[4]*z[5]*z[9] + 21376*z[11]*z[2]*z[4]*z[5] + 1024*z[11]*z[2]*z[4]*z[6]*z[7]*z[9] - 13312*z[11]*z[2]*z[4]*z[6]*z[7] + 2048*z[11]*z[2]*z[4]*z[6]*z[8]*z[9] - 26624*z[11]*z[2]*z[4]*z[6]*z[8] - 3328*z[11]*z[2]*z[4]*z[6]*z[9] + 42752*z[11]*z[2]*z[4]*z[6] + 4096*z[11]*z[2]*z[4]*z[7]*z[8]*z[9] - 53248*z[11]*z[2]*z[4]*z[7]*z[8] - 6656*z[11]*z[2]*z[4]*z[7]*z[9] + 85504*z[11]*z[2]*z[4]*z[7] - 13312*z[11]*z[2]*z[4]*z[8]*z[9] + 171008*z[11]*z[2]*z[4]*z[8] + 16256*z[11]*z[2]*z[4]*z[9] - 208000*z[11]*z[2]*z[4] - 416*z[11]*z[2]*z[5]*z[6]*z[9] + 5344*z[11]*z[2]*z[5]*z[6] - 832*z[11]*z[2]*z[5]*z[7]*z[9] + 10688*z[11]*z[2]*z[5]*z[7] - 1664*z[11]*z[2]*z[5]*z[8]*z[9] + 21376*z[11]*z[2]*z[5]*z[8] + 2672*z[11]*z[2]*z[5]*z[9] - 33840*z[11]*z[2]*z[5] - 1664*z[11]*z[2]*z[6]*z[7]*z[9] + 21376*z[11]*z[2]*z[6]*z[7] - 3328*z[11]*z[2]*z[6]*z[8]*z[9] + 42752*z[11]*z[2]*z[6]*z[8] + 5344*z[11]*z[2]*z[6]*z[9] - 67680*z[11]*z[2]*z[6] - 6656*z[11]*z[2]*z[7]*z[8]*z[9] + 85504*z[11]*z[2]*z[7]*z[8] + 10688*z[11]*z[2]*z[7]*z[9] - 135360*z[11]*z[2]*z[7] + 21376*z[11]*z[2]*z[8]*z[9] - 270720*z[11]*z[2]*z[8] - 26000*z[11]*z[2]*z[9] + 327760*z[11]*z[2] + 512*z[11]*z[3]*z[4]*z[5]*z[6]*z[9] - 6656*z[11]*z[3]*z[4]*z[5]*z[6] + 1024*z[11]*z[3]*z[4]*z[5]*z[7]*z[9] - 13312*z[11]*z[3]*z[4]*z[5]*z[7] + 2048*z[11]*z[3]*z[4]*z[5]*z[8]*z[9] - 26624*z[11]*z[3]*z[4]*z[5]*z[8] - 3328*z[11]*z[3]*z[4]*z[5]*z[9] + 42752*z[11]*z[3]*z[4]*z[5] + 2048*z[11]*z[3]*z[4]*z[6]*z[7]*z[9] - 26624*z[11]*z[3]*z[4]*z[6]*z[7] + 4096*z[11]*z[3]*z[4]*z[6]*z[8]*z[9] - 53248*z[11]*z[3]*z[4]*z[6]*z[8] - 6656*z[11]*z[3]*z[4]*z[6]*z[9] + 85504*z[11]*z[3]*z[4]*z[6] + 8192*z[11]*z[3]*z[4]*z[7]*z[8]*z[9] - 106496*z[11]*z[3]*z[4]*z[7]*z[8] - 13312*z[11]*z[3]*z[4]*z[7]*z[9] + 171008*z[11]*z[3]*z[4]*z[7] - 26624*z[11]*z[3]*z[4]*z[8]*z[9] + 342016*z[11]*z[3]*z[4]*z[8] + 32512*z[11]*z[3]*z[4]*z[9] - 416000*z[11]*z[3]*z[4] - 832*z[11]*z[3]*z[5]*z[6]*z[9] + 10688*z[11]*z[3]*z[5]*z[6] - 1664*z[11]*z[3]*z[5]*z[7]*z[9] + 21376*z[11]*z[3]*z[5]*z[7] - 3328*z[11]*z[3]*z[5]*z[8]*z[9] + 42752*z[11]*z[3]*z[5]*z[8] + 5344*z[11]*z[3]*z[5]*z[9] - 67680*z[11]*z[3]*z[5] - 3328*z[11]*z[3]*z[6]*z[7]*z[9] + 42752*z[11]*z[3]*z[6]*z[7] - 6656*z[11]*z[3]*z[6]*z[8]*z[9] + 85504*z[11]*z[3]*z[6]*z[8] + 10688*z[11]*z[3]*z[6]*z[9] - 135360*z[11]*z[3]*z[6] - 13312*z[11]*z[3]*z[7]*z[8]*z[9] + 171008*z[11]*z[3]*z[7]*z[8] + 21376*z[11]*z[3]*z[7]*z[9] - 270720*z[11]*z[3]*z[7] + 42752*z[11]*z[3]*z[8]*z[9] - 541440*z[11]*z[3]*z[8] - 52000*z[11]*z[3]*z[9] + 655520*z[11]*z[3] - 1664*z[11]*z[4]*z[5]*z[6]*z[9] + 21376*z[11]*z[4]*z[5]*z[6] - 3328*z[11]*z[4]*z[5]*z[7]*z[9] + 42752*z[11]*z[4]*z[5]*z[7] - 6656*z[11]*z[4]*z[5]*z[8]*z[9] + 85504*z[11]*z[4]*z[5]*z[8] + 10688*z[11]*z[4]*z[5]*z[9] - 135360*z[11]*z[4]*z[5] - 6656*z[11]*z[4]*z[6]*z[7]*z[9] + 85504*z[11]*z[4]*z[6]*z[7] - 13312*z[11]*z[4]*z[6]*z[8]*z[9] + 171008*z[11]*z[4]*z[6]*z[8] + 21376*z[11]*z[4]*z[6]*z[9] - 270720*z[11]*z[4]*z[6] - 26624*z[11]*z[4]*z[7]*z[8]*z[9] + 342016*z[11]*z[4]*z[7]*z[8] + 42752*z[11]*z[4]*z[7]*z[9] - 541440*z[11]*z[4]*z[7] + 85504*z[11]*z[4]*z[8]*z[9] - 1082880*z[11]*z[4]*z[8] - 104000*z[11]*z[4]*z[9] + 1311040*z[11]*z[4] + 2032*z[11]*z[5]*z[6]*z[9] - 26000*z[11]*z[5]*z[6] + 4064*z[11]*z[5]*z[7]*z[9] - 52000*z[11]*z[5]*z[7] + 8128*z[11]*z[5]*z[8]*z[9] - 104000*z[11]*z[5]*z[8] - 13000*z[11]*z[5]*z[9] + 163880*z[11]*z[5] + 8128*z[11]*z[6]*z[7]*z[9] - 104000*z[11]*z[6]*z[7] + 16256*z[11]*z[6]*z[8]*z[9] - 208000*z[11]*z[6]*z[8] - 26000*z[11]*z[6]*z[9] + 327760*z[11]*z[6] + 32512*z[11]*z[7]*z[8]*z[9] - 416000*z[11]*z[7]*z[8] - 52000*z[11]*z[7]*z[9] + 655520*z[11]*z[7] - 104000*z[11]*z[8]*z[9] + 1311040*z[11]*z[8] + 126360*z[11]*z[9] - 1585400*z[11] + 256*z[12]*z[2]*z[3]*z[5]*z[6]*z[9] - 3328*z[12]*z[2]*z[3]*z[5]*z[6] + 512*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] - 6656*z[12]*z[2]*z[3]*z[5]*z[7] + 1024*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] - 13312*z[12]*z[2]*z[3]*z[5]*z[8] - 1664*z[12]*z[2]*z[3]*z[5]*z[9] + 21376*z[12]*z[2]*z[3]*z[5] + 1024*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] - 13312*z[12]*z[2]*z[3]*z[6]*z[7] + 2048*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] - 26624*z[12]*z[2]*z[3]*z[6]*z[8] - 3328*z[12]*z[2]*z[3]*z[6]*z[9] + 42752*z[12]*z[2]*z[3]*z[6] + 4096*z[12]*z[2]*z[3]*z[7]*z[8]*z[9] - 53248*z[12]*z[2]*z[3]*z[7]*z[8] - 6656*z[12]*z[2]*z[3]*z[7]*z[9] + 85504*z[12]*z[2]*z[3]*z[7] - 13312*z[12]*z[2]*z[3]*z[8]*z[9] + 171008*z[12]*z[2]*z[3]*z[8] + 16256*z[12]*z[2]*z[3]*z[9] - 208000*z[12]*z[2]*z[3] + 512*z[12]*z[2]*z[4]*z[5]*z[6]*z[9] - 6656*z[12]*z[2]*z[4]*z[5]*z[6] + 1024*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] - 13312*z[12]*z[2]*z[4]*z[5]*z[7] + 2048*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] - 26624*z[12]*z[2]*z[4]*z[5]*z[8] - 3328*z[12]*z[2]*z[4]*z[5]*z[9] + 42752*z[12]*z[2]*z[4]*z[5] + 2048*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] - 26624*z[12]*z[2]*z[4]*z[6]*z[7] + 4096*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] - 53248*z[12]*z[2]*z[4]*z[6]*z[8] - 6656*z[12]*z[2]*z[4]*z[6]*z[9] + 85504*z[12]*z[2]*z[4]*z[6] + 8192*z[12]*z[2]*z[4]*z[7]*z[8]*z[9] - 106496*z[12]*z[2]*z[4]*z[7]*z[8] - 13312*z[12]*z[2]*z[4]*z[7]*z[9] + 171008*z[12]*z[2]*z[4]*z[7] - 26624*z[12]*z[2]*z[4]*z[8]*z[9] + 342016*z[12]*z[2]*z[4]*z[8] + 32512*z[12]*z[2]*z[4]*z[9] - 416000*z[12]*z[2]*z[4] - 832*z[12]*z[2]*z[5]*z[6]*z[9] + 10688*z[12]*z[2]*z[5]*z[6] - 1664*z[12]*z[2]*z[5]*z[7]*z[9] + 21376*z[12]*z[2]*z[5]*z[7] - 3328*z[12]*z[2]*z[5]*z[8]*z[9] + 42752*z[12]*z[2]*z[5]*z[8] + 5344*z[12]*z[2]*z[5]*z[9] - 67680*z[12]*z[2]*z[5] - 3328*z[12]*z[2]*z[6]*z[7]*z[9] + 42752*z[12]*z[2]*z[6]*z[7] - 6656*z[12]*z[2]*z[6]*z[8]*z[9] + 85504*z[12]*z[2]*z[6]*z[8] + 10688*z[12]*z[2]*z[6]*z[9] - 135360*z[12]*z[2]*z[6] - 13312*z[12]*z[2]*z[7]*z[8]*z[9] + 171008*z[12]*z[2]*z[7]*z[8] + 21376*z[12]*z[2]*z[7]*z[9] - 270720*z[12]*z[2]*z[7] + 42752*z[12]*z[2]*z[8]*z[9] - 541440*z[12]*z[2]*z[8] - 52000*z[12]*z[2]*z[9] + 655520*z[12]*z[2] + 1024*z[12]*z[3]*z[4]*z[5]*z[6]*z[9] - 13312*z[12]*z[3]*z[4]*z[5]*z[6] + 2048*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] - 26624*z[12]*z[3]*z[4]*z[5]*z[7] + 4096*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] - 53248*z[12]*z[3]*z[4]*z[5]*z[8] - 6656*z[12]*z[3]*z[4]*z[5]*z[9] + 85504*z[12]*z[3]*z[4]*z[5] + 4096*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] - 53248*z[12]*z[3]*z[4]*z[6]*z[7] + 8192*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] - 106496*z[12]*z[3]*z[4]*z[6]*z[8] - 13312*z[12]*z[3]*z[4]*z[6]*z[9] + 171008*z[12]*z[3]*z[4]*z[6] + 16384*z[12]*z[3]*z[4]*z[7]*z[8]*z[9] - 212992*z[12]*z[3]*z[4]*z[7]*z[8] - 26624*z[12]*z[3]*z[4]*z[7]*z[9] + 342016*z[12]*z[3]*z[4]*z[7] - 53248*z[12]*z[3]*z[4]*z[8]*z[9] + 684032*z[12]*z[3]*z[4]*z[8] + 65024*z[12]*z[3]*z[4]*z[9] - 832000*z[12]*z[3]*z[4] - 1664*z[12]*z[3]*z[5]*z[6]*z[9] + 21376*z[12]*z[3]*z[5]*z[6] - 3328*z[12]*z[3]*z[5]*z[7]*z[9] + 42752*z[12]*z[3]*z[5]*z[7] - 6656*z[12]*z[3]*z[5]*z[8]*z[9] + 85504*z[12]*z[3]*z[5]*z[8] + 10688*z[12]*z[3]*z[5]*z[9] - 135360*z[12]*z[3]*z[5] - 6656*z[12]*z[3]*z[6]*z[7]*z[9] + 85504*z[12]*z[3]*z[6]*z[7] - 13312*z[12]*z[3]*z[6]*z[8]*z[9] + 171008*z[12]*z[3]*z[6]*z[8] + 21376*z[12]*z[3]*z[6]*z[9] - 270720*z[12]*z[3]*z[6] - 26624*z[12]*z[3]*z[7]*z[8]*z[9] + 342016*z[12]*z[3]*z[7]*z[8] + 42752*z[12]*z[3]*z[7]*z[9] - 541440*z[12]*z[3]*z[7] + 85504*z[12]*z[3]*z[8]*z[9] - 1082880*z[12]*z[3]*z[8] - 104000*z[12]*z[3]*z[9] + 1311040*z[12]*z[3] - 3328*z[12]*z[4]*z[5]*z[6]*z[9] + 42752*z[12]*z[4]*z[5]*z[6] - 6656*z[12]*z[4]*z[5]*z[7]*z[9] + 85504*z[12]*z[4]*z[5]*z[7] - 13312*z[12]*z[4]*z[5]*z[8]*z[9] + 171008*z[12]*z[4]*z[5]*z[8] + 21376*z[12]*z[4]*z[5]*z[9] - 270720*z[12]*z[4]*z[5] - 13312*z[12]*z[4]*z[6]*z[7]*z[9] + 171008*z[12]*z[4]*z[6]*z[7] - 26624*z[12]*z[4]*z[6]*z[8]*z[9] + 342016*z[12]*z[4]*z[6]*z[8] + 42752*z[12]*z[4]*z[6]*z[9] - 541440*z[12]*z[4]*z[6] - 53248*z[12]*z[4]*z[7]*z[8]*z[9] + 684032*z[12]*z[4]*z[7]*z[8] + 85504*z[12]*z[4]*z[7]*z[9] - 1082880*z[12]*z[4]*z[7] + 171008*z[12]*z[4]*z[8]*z[9] - 2165760*z[12]*z[4]*z[8] - 208000*z[12]*z[4]*z[9] + 2622080*z[12]*z[4] + 4064*z[12]*z[5]*z[6]*z[9] - 52000*z[12]*z[5]*z[6] + 8128*z[12]*z[5]*z[7]*z[9] - 104000*z[12]*z[5]*z[7] + 16256*z[12]*z[5]*z[8]*z[9] - 208000*z[12]*z[5]*z[8] - 26000*z[12]*z[5]*z[9] + 327760*z[12]*z[5] + 16256*z[12]*z[6]*z[7]*z[9] - 208000*z[12]*z[6]*z[7] + 32512*z[12]*z[6]*z[8]*z[9] - 416000*z[12]*z[6]*z[8] - 52000*z[12]*z[6]*z[9] + 655520*z[12]*z[6] + 65024*z[12]*z[7]*z[8]*z[9] - 832000*z[12]*z[7]*z[8] - 104000*z[12]*z[7]*z[9] + 1311040*z[12]*z[7] - 208000*z[12]*z[8]*z[9] + 2622080*z[12]*z[8] + 252720*z[12]*z[9] - 3170800*z[12] - 416*z[2]*z[3]*z[5]*z[6]*z[9] + 4064*z[2]*z[3]*z[5]*z[6] - 832*z[2]*z[3]*z[5]*z[7]*z[9] + 8128*z[2]*z[3]*z[5]*z[7] - 1664*z[2]*z[3]*z[5]*z[8]*z[9] + 16256*z[2]*z[3]*z[5]*z[8] + 2672*z[2]*z[3]*z[5]*z[9] - 26000*z[2]*z[3]*z[5] - 1664*z[2]*z[3]*z[6]*z[7]*z[9] + 16256*z[2]*z[3]*z[6]*z[7] - 3328*z[2]*z[3]*z[6]*z[8]*z[9] + 32512*z[2]*z[3]*z[6]*z[8] + 5344*z[2]*z[3]*z[6]*z[9] - 52000*z[2]*z[3]*z[6] - 6656*z[2]*z[3]*z[7]*z[8]*z[9] + 65024*z[2]*z[3]*z[7]*z[8] + 10688*z[2]*z[3]*z[7]*z[9] - 104000*z[2]*z[3]*z[7] + 21376*z[2]*z[3]*z[8]*z[9] - 208000*z[2]*z[3]*z[8] - 26000*z[2]*z[3]*z[9] + 252720*z[2]*z[3] - 832*z[2]*z[4]*z[5]*z[6]*z[9] + 8128*z[2]*z[4]*z[5]*z[6] - 1664*z[2]*z[4]*z[5]*z[7]*z[9] + 16256*z[2]*z[4]*z[5]*z[7] - 3328*z[2]*z[4]*z[5]*z[8]*z[9] + 32512*z[2]*z[4]*z[5]*z[8] + 5344*z[2]*z[4]*z[5]*z[9] - 52000*z[2]*z[4]*z[5] - 3328*z[2]*z[4]*z[6]*z[7]*z[9] + 32512*z[2]*z[4]*z[6]*z[7] - 6656*z[2]*z[4]*z[6]*z[8]*z[9] + 65024*z[2]*z[4]*z[6]*z[8] + 10688*z[2]*z[4]*z[6]*z[9] - 104000*z[2]*z[4]*z[6] - 13312*z[2]*z[4]*z[7]*z[8]*z[9] + 130048*z[2]*z[4]*z[7]*z[8] + 21376*z[2]*z[4]*z[7]*z[9] - 208000*z[2]*z[4]*z[7] + 42752*z[2]*z[4]*z[8]*z[9] - 416000*z[2]*z[4]*z[8] - 52000*z[2]*z[4]*z[9] + 505440*z[2]*z[4] + 1336*z[2]*z[5]*z[6]*z[9] - 13000*z[2]*z[5]*z[6] + 2672*z[2]*z[5]*z[7]*z[9] - 26000*z[2]*z[5]*z[7] + 5344*z[2]*z[5]*z[8]*z[9] - 52000*z[2]*z[5]*z[8] - 8460*z[2]*z[5]*z[9] + 81940*z[2]*z[5] + 5344*z[2]*z[6]*z[7]*z[9] - 52000*z[2]*z[6]*z[7] + 10688*z[2]*z[6]*z[8]*z[9] - 104000*z[2]*z[6]*z[8] - 16920*z[2]*z[6]*z[9] + 163880*z[2]*z[6] + 21376*z[2]*z[7]*z[8]*z[9] - 208000*z[2]*z[7]*z[8] - 33840*z[2]*z[7]*z[9] + 327760*z[2]*z[7] - 67680*z[2]*z[8]*z[9] + 655520*z[2]*z[8] + 81940*z[2]*z[9] - 792700*z[2] - 1664*z[3]*z[4]*z[5]*z[6]*z[9] + 16256*z[3]*z[4]*z[5]*z[6] - 3328*z[3]*z[4]*z[5]*z[7]*z[9] + 32512*z[3]*z[4]*z[5]*z[7] - 6656*z[3]*z[4]*z[5]*z[8]*z[9] + 65024*z[3]*z[4]*z[5]*z[8] + 10688*z[3]*z[4]*z[5]*z[9] - 104000*z[3]*z[4]*z[5] - 6656*z[3]*z[4]*z[6]*z[7]*z[9] + 65024*z[3]*z[4]*z[6]*z[7] - 13312*z[3]*z[4]*z[6]*z[8]*z[9] + 130048*z[3]*z[4]*z[6]*z[8] + 21376*z[3]*z[4]*z[6]*z[9] - 208000*z[3]*z[4]*z[6] - 26624*z[3]*z[4]*z[7]*z[8]*z[9] + 260096*z[3]*z[4]*z[7]*z[8] + 42752*z[3]*z[4]*z[7]*z[9] - 416000*z[3]*z[4]*z[7] + 85504*z[3]*z[4]*z[8]*z[9] - 832000*z[3]*z[4]*z[8] - 104000*z[3]*z[4]*z[9] + 1010880*z[3]*z[4] + 2672*z[3]*z[5]*z[6]*z[9] - 26000*z[3]*z[5]*z[6] + 5344*z[3]*z[5]*z[7]*z[9] - 52000*z[3]*z[5]*z[7] + 10688*z[3]*z[5]*z[8]*z[9] - 104000*z[3]*z[5]*z[8] - 16920*z[3]*z[5]*z[9] + 163880*z[3]*z[5] + 10688*z[3]*z[6]*z[7]*z[9] - 104000*z[3]*z[6]*z[7] + 21376*z[3]*z[6]*z[8]*z[9] - 208000*z[3]*z[6]*z[8] - 33840*z[3]*z[6]*z[9] + 327760*z[3]*z[6] + 42752*z[3]*z[7]*z[8]*z[9] - 416000*z[3]*z[7]*z[8] - 67680*z[3]*z[7]*z[9] + 655520*z[3]*z[7] - 135360*z[3]*z[8]*z[9] + 1311040*z[3]*z[8] + 163880*z[3]*z[9] - 1585400*z[3] + 5344*z[4]*z[5]*z[6]*z[9] - 52000*z[4]*z[5]*z[6] + 10688*z[4]*z[5]*z[7]*z[9] - 104000*z[4]*z[5]*z[7] + 21376*z[4]*z[5]*z[8]*z[9] - 208000*z[4]*z[5]*z[8] - 33840*z[4]*z[5]*z[9] + 327760*z[4]*z[5] + 21376*z[4]*z[6]*z[7]*z[9] - 208000*z[4]*z[6]*z[7] + 42752*z[4]*z[6]*z[8]*z[9] - 416000*z[4]*z[6]*z[8] - 67680*z[4]*z[6]*z[9] + 655520*z[4]*z[6] + 85504*z[4]*z[7]*z[8]*z[9] - 832000*z[4]*z[7]*z[8] - 135360*z[4]*z[7]*z[9] + 1311040*z[4]*z[7] - 270720*z[4]*z[8]*z[9] + 2622080*z[4]*z[8] + 327760*z[4]*z[9] - 3170800*z[4] - 6500*z[5]*z[6]*z[9] + 63180*z[5]*z[6] - 13000*z[5]*z[7]*z[9] + 126360*z[5]*z[7] - 26000*z[5]*z[8]*z[9] + 252720*z[5]*z[8] + 40970*z[5]*z[9] - 396350*z[5] - 26000*z[6]*z[7]*z[9] + 252720*z[6]*z[7] - 52000*z[6]*z[8]*z[9] + 505440*z[6]*z[8] + 81940*z[6]*z[9] - 792700*z[6] - 104000*z[7]*z[8]*z[9] + 1010880*z[7]*z[8] + 163880*z[7]*z[9] - 1585400*z[7] + 327760*z[8]*z[9] - 3170800*z[8] - 396350*z[9] + 3830050)

In [49]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 12  # 
p = 1  # QAOA depth

hamiltonian_less_15_1_Erdös = build_cost_hamiltonian_penalty(num_qubits, parse_hamiltonian_expr(paso3_D_Erdös_Straus_less_15_1),100)
final_circuit_less_15_1_Erdös, result_less_15_1_Erdös = qaoa(num_qubits, hamiltonian_less_15_1_Erdös, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.09723856 1.13120397]
Minimum expectation value: -1535971.51953125


In [50]:
top_solutions_less_15_1_Erdös = find_best_bitstrings(final_circuit_less_15_1_Erdös, hamiltonian_less_15_1_Erdös)

Top 5 bitstrings:
Bitstring: 000000001101, Cost: -7660100.0000, Count: 1
Bitstring: 011000100011, Cost: -7660100.0000, Count: 1
Bitstring: 001001100011, Cost: -7660100.0000, Count: 1
Bitstring: 011100100011, Cost: -7660084.0000, Count: 3
Bitstring: 010100100011, Cost: -7660084.0000, Count: 1


In [63]:
bitstring_to_pm1(top_solutions_less_15_1_Erdös, evaluate_hamiltonian_D_Erdös_Straus_less_15)

["Bitstring: ('000000001101', -7660100.0, 1), Evaluated cost: 0",
 "Bitstring: ('000000001101', -7660100.0, 1), Evaluated cost: 0",
 "Bitstring: ('000000001101', -7660100.0, 1), Evaluated cost: 0",
 "Bitstring: ('000000001101', -7660100.0, 1), Evaluated cost: 16",
 "Bitstring: ('000000001101', -7660100.0, 1), Evaluated cost: 16"]

In [64]:
bitstrings_less_15_1_Erdös = []
for i in range(len(top_solutions_less_15_1_Erdös)):
    bitstrings_less_15_1_Erdös.append(top_solutions_less_15_1_Erdös[i][0])
bitstrings_less_15_1_Erdös

['000000001101',
 '011000100011',
 '001001100011',
 '011100100011',
 '010100100011']

In [74]:
group_and_convert(bitstrings_less_15_1_Erdös,4)

[[11, 0, 0], [12, 4, 6], [12, 6, 4], [12, 4, 14], [12, 4, 10]]

In [75]:
def D_Erdös_Straus_2(x,y,z):
    return(4*x*y*z-(10**18)*(y*z+x*z+x*y))

In [76]:
paso1_D_Erdös_Straus_less_15_2 = substitute_with_global_binary_symbols(D_Erdös_Straus_2(x[1],x[2],x[3])**2, 4, base_name="b")
paso2_D_Erdös_Straus_less_15_2 = remove_variable_exponents(paso1_D_Erdös_Straus_less_15_2)
paso3_D_Erdös_Straus_less_15_2 = substitute_with_spin_variables(paso2_D_Erdös_Straus_less_15_2)

In [77]:
def evaluate_hamiltonian_D_Erdös_Straus_less_15_2(z):
    return(64*z[1]*z[10]*z[11]*z[2]*z[5]*z[6] + 128*z[1]*z[10]*z[11]*z[2]*z[5]*z[7] + 256*z[1]*z[10]*z[11]*z[2]*z[5]*z[8] + 15999999999999999520*z[1]*z[10]*z[11]*z[2]*z[5] + 256*z[1]*z[10]*z[11]*z[2]*z[6]*z[7] + 512*z[1]*z[10]*z[11]*z[2]*z[6]*z[8] + 31999999999999999040*z[1]*z[10]*z[11]*z[2]*z[6] + 1024*z[1]*z[10]*z[11]*z[2]*z[7]*z[8] + 63999999999999998080*z[1]*z[10]*z[11]*z[2]*z[7] + 127999999999999996160*z[1]*z[10]*z[11]*z[2]*z[8] + 3999999999999999760000000000000004960*z[1]*z[10]*z[11]*z[2] + 128*z[1]*z[10]*z[11]*z[3]*z[5]*z[6] + 256*z[1]*z[10]*z[11]*z[3]*z[5]*z[7] + 512*z[1]*z[10]*z[11]*z[3]*z[5]*z[8] + 31999999999999999040*z[1]*z[10]*z[11]*z[3]*z[5] + 512*z[1]*z[10]*z[11]*z[3]*z[6]*z[7] + 1024*z[1]*z[10]*z[11]*z[3]*z[6]*z[8] + 63999999999999998080*z[1]*z[10]*z[11]*z[3]*z[6] + 2048*z[1]*z[10]*z[11]*z[3]*z[7]*z[8] + 127999999999999996160*z[1]*z[10]*z[11]*z[3]*z[7] + 255999999999999992320*z[1]*z[10]*z[11]*z[3]*z[8] + 7999999999999999520000000000000009920*z[1]*z[10]*z[11]*z[3] + 256*z[1]*z[10]*z[11]*z[4]*z[5]*z[6] + 512*z[1]*z[10]*z[11]*z[4]*z[5]*z[7] + 1024*z[1]*z[10]*z[11]*z[4]*z[5]*z[8] + 63999999999999998080*z[1]*z[10]*z[11]*z[4]*z[5] + 1024*z[1]*z[10]*z[11]*z[4]*z[6]*z[7] + 2048*z[1]*z[10]*z[11]*z[4]*z[6]*z[8] + 127999999999999996160*z[1]*z[10]*z[11]*z[4]*z[6] + 4096*z[1]*z[10]*z[11]*z[4]*z[7]*z[8] + 255999999999999992320*z[1]*z[10]*z[11]*z[4]*z[7] + 511999999999999984640*z[1]*z[10]*z[11]*z[4]*z[8] + 15999999999999999040000000000000019840*z[1]*z[10]*z[11]*z[4] + 15999999999999999520*z[1]*z[10]*z[11]*z[5]*z[6] + 31999999999999999040*z[1]*z[10]*z[11]*z[5]*z[7] + 63999999999999998080*z[1]*z[10]*z[11]*z[5]*z[8] + 1999999999999999760000000000000003600*z[1]*z[10]*z[11]*z[5] + 63999999999999998080*z[1]*z[10]*z[11]*z[6]*z[7] + 127999999999999996160*z[1]*z[10]*z[11]*z[6]*z[8] + 3999999999999999520000000000000007200*z[1]*z[10]*z[11]*z[6] + 255999999999999992320*z[1]*z[10]*z[11]*z[7]*z[8] + 7999999999999999040000000000000014400*z[1]*z[10]*z[11]*z[7] + 15999999999999998080000000000000028800*z[1]*z[10]*z[11]*z[8] - 59999999999999996960000000000000037200*z[1]*z[10]*z[11] + 128*z[1]*z[10]*z[12]*z[2]*z[5]*z[6] + 256*z[1]*z[10]*z[12]*z[2]*z[5]*z[7] + 512*z[1]*z[10]*z[12]*z[2]*z[5]*z[8] + 31999999999999999040*z[1]*z[10]*z[12]*z[2]*z[5] + 512*z[1]*z[10]*z[12]*z[2]*z[6]*z[7] + 1024*z[1]*z[10]*z[12]*z[2]*z[6]*z[8] + 63999999999999998080*z[1]*z[10]*z[12]*z[2]*z[6] + 2048*z[1]*z[10]*z[12]*z[2]*z[7]*z[8] + 127999999999999996160*z[1]*z[10]*z[12]*z[2]*z[7] + 255999999999999992320*z[1]*z[10]*z[12]*z[2]*z[8] + 7999999999999999520000000000000009920*z[1]*z[10]*z[12]*z[2] + 256*z[1]*z[10]*z[12]*z[3]*z[5]*z[6] + 512*z[1]*z[10]*z[12]*z[3]*z[5]*z[7] + 1024*z[1]*z[10]*z[12]*z[3]*z[5]*z[8] + 63999999999999998080*z[1]*z[10]*z[12]*z[3]*z[5] + 1024*z[1]*z[10]*z[12]*z[3]*z[6]*z[7] + 2048*z[1]*z[10]*z[12]*z[3]*z[6]*z[8] + 127999999999999996160*z[1]*z[10]*z[12]*z[3]*z[6] + 4096*z[1]*z[10]*z[12]*z[3]*z[7]*z[8] + 255999999999999992320*z[1]*z[10]*z[12]*z[3]*z[7] + 511999999999999984640*z[1]*z[10]*z[12]*z[3]*z[8] + 15999999999999999040000000000000019840*z[1]*z[10]*z[12]*z[3] + 512*z[1]*z[10]*z[12]*z[4]*z[5]*z[6] + 1024*z[1]*z[10]*z[12]*z[4]*z[5]*z[7] + 2048*z[1]*z[10]*z[12]*z[4]*z[5]*z[8] + 127999999999999996160*z[1]*z[10]*z[12]*z[4]*z[5] + 2048*z[1]*z[10]*z[12]*z[4]*z[6]*z[7] + 4096*z[1]*z[10]*z[12]*z[4]*z[6]*z[8] + 255999999999999992320*z[1]*z[10]*z[12]*z[4]*z[6] + 8192*z[1]*z[10]*z[12]*z[4]*z[7]*z[8] + 511999999999999984640*z[1]*z[10]*z[12]*z[4]*z[7] + 1023999999999999969280*z[1]*z[10]*z[12]*z[4]*z[8] + 31999999999999998080000000000000039680*z[1]*z[10]*z[12]*z[4] + 31999999999999999040*z[1]*z[10]*z[12]*z[5]*z[6] + 63999999999999998080*z[1]*z[10]*z[12]*z[5]*z[7] + 127999999999999996160*z[1]*z[10]*z[12]*z[5]*z[8] + 3999999999999999520000000000000007200*z[1]*z[10]*z[12]*z[5] + 127999999999999996160*z[1]*z[10]*z[12]*z[6]*z[7] + 255999999999999992320*z[1]*z[10]*z[12]*z[6]*z[8] + 7999999999999999040000000000000014400*z[1]*z[10]*z[12]*z[6] + 511999999999999984640*z[1]*z[10]*z[12]*z[7]*z[8] + 15999999999999998080000000000000028800*z[1]*z[10]*z[12]*z[7] + 31999999999999996160000000000000057600*z[1]*z[10]*z[12]*z[8] - 119999999999999993920000000000000074400*z[1]*z[10]*z[12] + 16*z[1]*z[10]*z[2]*z[5]*z[6]*z[9] + 7999999999999999760*z[1]*z[10]*z[2]*z[5]*z[6] + 32*z[1]*z[10]*z[2]*z[5]*z[7]*z[9] + 15999999999999999520*z[1]*z[10]*z[2]*z[5]*z[7] + 64*z[1]*z[10]*z[2]*z[5]*z[8]*z[9] + 31999999999999999040*z[1]*z[10]*z[2]*z[5]*z[8] + 3999999999999999880*z[1]*z[10]*z[2]*z[5]*z[9] + 999999999999999880000000000000001800*z[1]*z[10]*z[2]*z[5] + 64*z[1]*z[10]*z[2]*z[6]*z[7]*z[9] + 31999999999999999040*z[1]*z[10]*z[2]*z[6]*z[7] + 128*z[1]*z[10]*z[2]*z[6]*z[8]*z[9] + 63999999999999998080*z[1]*z[10]*z[2]*z[6]*z[8] + 7999999999999999760*z[1]*z[10]*z[2]*z[6]*z[9] + 1999999999999999760000000000000003600*z[1]*z[10]*z[2]*z[6] + 256*z[1]*z[10]*z[2]*z[7]*z[8]*z[9] + 127999999999999996160*z[1]*z[10]*z[2]*z[7]*z[8] + 15999999999999999520*z[1]*z[10]*z[2]*z[7]*z[9] + 3999999999999999520000000000000007200*z[1]*z[10]*z[2]*z[7] + 31999999999999999040*z[1]*z[10]*z[2]*z[8]*z[9] + 7999999999999999040000000000000014400*z[1]*z[10]*z[2]*z[8] + 999999999999999940000000000000001240*z[1]*z[10]*z[2]*z[9] - 29999999999999998480000000000000018600*z[1]*z[10]*z[2] + 32*z[1]*z[10]*z[3]*z[5]*z[6]*z[9] + 15999999999999999520*z[1]*z[10]*z[3]*z[5]*z[6] + 64*z[1]*z[10]*z[3]*z[5]*z[7]*z[9] + 31999999999999999040*z[1]*z[10]*z[3]*z[5]*z[7] + 128*z[1]*z[10]*z[3]*z[5]*z[8]*z[9] + 63999999999999998080*z[1]*z[10]*z[3]*z[5]*z[8] + 7999999999999999760*z[1]*z[10]*z[3]*z[5]*z[9] + 1999999999999999760000000000000003600*z[1]*z[10]*z[3]*z[5] + 128*z[1]*z[10]*z[3]*z[6]*z[7]*z[9] + 63999999999999998080*z[1]*z[10]*z[3]*z[6]*z[7] + 256*z[1]*z[10]*z[3]*z[6]*z[8]*z[9] + 127999999999999996160*z[1]*z[10]*z[3]*z[6]*z[8] + 15999999999999999520*z[1]*z[10]*z[3]*z[6]*z[9] + 3999999999999999520000000000000007200*z[1]*z[10]*z[3]*z[6] + 512*z[1]*z[10]*z[3]*z[7]*z[8]*z[9] + 255999999999999992320*z[1]*z[10]*z[3]*z[7]*z[8] + 31999999999999999040*z[1]*z[10]*z[3]*z[7]*z[9] + 7999999999999999040000000000000014400*z[1]*z[10]*z[3]*z[7] + 63999999999999998080*z[1]*z[10]*z[3]*z[8]*z[9] + 15999999999999998080000000000000028800*z[1]*z[10]*z[3]*z[8] + 1999999999999999880000000000000002480*z[1]*z[10]*z[3]*z[9] - 59999999999999996960000000000000037200*z[1]*z[10]*z[3] + 64*z[1]*z[10]*z[4]*z[5]*z[6]*z[9] + 31999999999999999040*z[1]*z[10]*z[4]*z[5]*z[6] + 128*z[1]*z[10]*z[4]*z[5]*z[7]*z[9] + 63999999999999998080*z[1]*z[10]*z[4]*z[5]*z[7] + 256*z[1]*z[10]*z[4]*z[5]*z[8]*z[9] + 127999999999999996160*z[1]*z[10]*z[4]*z[5]*z[8] + 15999999999999999520*z[1]*z[10]*z[4]*z[5]*z[9] + 3999999999999999520000000000000007200*z[1]*z[10]*z[4]*z[5] + 256*z[1]*z[10]*z[4]*z[6]*z[7]*z[9] + 127999999999999996160*z[1]*z[10]*z[4]*z[6]*z[7] + 512*z[1]*z[10]*z[4]*z[6]*z[8]*z[9] + 255999999999999992320*z[1]*z[10]*z[4]*z[6]*z[8] + 31999999999999999040*z[1]*z[10]*z[4]*z[6]*z[9] + 7999999999999999040000000000000014400*z[1]*z[10]*z[4]*z[6] + 1024*z[1]*z[10]*z[4]*z[7]*z[8]*z[9] + 511999999999999984640*z[1]*z[10]*z[4]*z[7]*z[8] + 63999999999999998080*z[1]*z[10]*z[4]*z[7]*z[9] + 15999999999999998080000000000000028800*z[1]*z[10]*z[4]*z[7] + 127999999999999996160*z[1]*z[10]*z[4]*z[8]*z[9] + 31999999999999996160000000000000057600*z[1]*z[10]*z[4]*z[8] + 3999999999999999760000000000000004960*z[1]*z[10]*z[4]*z[9] - 119999999999999993920000000000000074400*z[1]*z[10]*z[4] + 3999999999999999880*z[1]*z[10]*z[5]*z[6]*z[9] + 999999999999999880000000000000001800*z[1]*z[10]*z[5]*z[6] + 7999999999999999760*z[1]*z[10]*z[5]*z[7]*z[9] + 1999999999999999760000000000000003600*z[1]*z[10]*z[5]*z[7] + 15999999999999999520*z[1]*z[10]*z[5]*z[8]*z[9] + 3999999999999999520000000000000007200*z[1]*z[10]*z[5]*z[8] + 499999999999999940000000000000000900*z[1]*z[10]*z[5]*z[9] - 22499999999999998650000000000000013500*z[1]*z[10]*z[5] + 15999999999999999520*z[1]*z[10]*z[6]*z[7]*z[9] + 3999999999999999520000000000000007200*z[1]*z[10]*z[6]*z[7] + 31999999999999999040*z[1]*z[10]*z[6]*z[8]*z[9] + 7999999999999999040000000000000014400*z[1]*z[10]*z[6]*z[8] + 999999999999999880000000000000001800*z[1]*z[10]*z[6]*z[9] - 44999999999999997300000000000000027000*z[1]*z[10]*z[6] + 63999999999999998080*z[1]*z[10]*z[7]*z[8]*z[9] + 15999999999999998080000000000000028800*z[1]*z[10]*z[7]*z[8] + 1999999999999999760000000000000003600*z[1]*z[10]*z[7]*z[9] - 89999999999999994600000000000000054000*z[1]*z[10]*z[7] + 3999999999999999520000000000000007200*z[1]*z[10]*z[8]*z[9] - 179999999999999989200000000000000108000*z[1]*z[10]*z[8] - 14999999999999999240000000000000009300*z[1]*z[10]*z[9] + 414999999999999983950000000000000139500*z[1]*z[10] + 256*z[1]*z[11]*z[12]*z[2]*z[5]*z[6] + 512*z[1]*z[11]*z[12]*z[2]*z[5]*z[7] + 1024*z[1]*z[11]*z[12]*z[2]*z[5]*z[8] + 63999999999999998080*z[1]*z[11]*z[12]*z[2]*z[5] + 1024*z[1]*z[11]*z[12]*z[2]*z[6]*z[7] + 2048*z[1]*z[11]*z[12]*z[2]*z[6]*z[8] + 127999999999999996160*z[1]*z[11]*z[12]*z[2]*z[6] + 4096*z[1]*z[11]*z[12]*z[2]*z[7]*z[8] + 255999999999999992320*z[1]*z[11]*z[12]*z[2]*z[7] + 511999999999999984640*z[1]*z[11]*z[12]*z[2]*z[8] + 15999999999999999040000000000000019840*z[1]*z[11]*z[12]*z[2] + 512*z[1]*z[11]*z[12]*z[3]*z[5]*z[6] + 1024*z[1]*z[11]*z[12]*z[3]*z[5]*z[7] + 2048*z[1]*z[11]*z[12]*z[3]*z[5]*z[8] + 127999999999999996160*z[1]*z[11]*z[12]*z[3]*z[5] + 2048*z[1]*z[11]*z[12]*z[3]*z[6]*z[7] + 4096*z[1]*z[11]*z[12]*z[3]*z[6]*z[8] + 255999999999999992320*z[1]*z[11]*z[12]*z[3]*z[6] + 8192*z[1]*z[11]*z[12]*z[3]*z[7]*z[8] + 511999999999999984640*z[1]*z[11]*z[12]*z[3]*z[7] + 1023999999999999969280*z[1]*z[11]*z[12]*z[3]*z[8] + 31999999999999998080000000000000039680*z[1]*z[11]*z[12]*z[3] + 1024*z[1]*z[11]*z[12]*z[4]*z[5]*z[6] + 2048*z[1]*z[11]*z[12]*z[4]*z[5]*z[7] + 4096*z[1]*z[11]*z[12]*z[4]*z[5]*z[8] + 255999999999999992320*z[1]*z[11]*z[12]*z[4]*z[5] + 4096*z[1]*z[11]*z[12]*z[4]*z[6]*z[7] + 8192*z[1]*z[11]*z[12]*z[4]*z[6]*z[8] + 511999999999999984640*z[1]*z[11]*z[12]*z[4]*z[6] + 16384*z[1]*z[11]*z[12]*z[4]*z[7]*z[8] + 1023999999999999969280*z[1]*z[11]*z[12]*z[4]*z[7] + 2047999999999999938560*z[1]*z[11]*z[12]*z[4]*z[8] + 63999999999999996160000000000000079360*z[1]*z[11]*z[12]*z[4] + 63999999999999998080*z[1]*z[11]*z[12]*z[5]*z[6] + 127999999999999996160*z[1]*z[11]*z[12]*z[5]*z[7] + 255999999999999992320*z[1]*z[11]*z[12]*z[5]*z[8] + 7999999999999999040000000000000014400*z[1]*z[11]*z[12]*z[5] + 255999999999999992320*z[1]*z[11]*z[12]*z[6]*z[7] + 511999999999999984640*z[1]*z[11]*z[12]*z[6]*z[8] + 15999999999999998080000000000000028800*z[1]*z[11]*z[12]*z[6] + 1023999999999999969280*z[1]*z[11]*z[12]*z[7]*z[8] + 31999999999999996160000000000000057600*z[1]*z[11]*z[12]*z[7] + 63999999999999992320000000000000115200*z[1]*z[11]*z[12]*z[8] - 239999999999999987840000000000000148800*z[1]*z[11]*z[12] + 32*z[1]*z[11]*z[2]*z[5]*z[6]*z[9] + 15999999999999999520*z[1]*z[11]*z[2]*z[5]*z[6] + 64*z[1]*z[11]*z[2]*z[5]*z[7]*z[9] + 31999999999999999040*z[1]*z[11]*z[2]*z[5]*z[7] + 128*z[1]*z[11]*z[2]*z[5]*z[8]*z[9] + 63999999999999998080*z[1]*z[11]*z[2]*z[5]*z[8] + 7999999999999999760*z[1]*z[11]*z[2]*z[5]*z[9] + 1999999999999999760000000000000003600*z[1]*z[11]*z[2]*z[5] + 128*z[1]*z[11]*z[2]*z[6]*z[7]*z[9] + 63999999999999998080*z[1]*z[11]*z[2]*z[6]*z[7] + 256*z[1]*z[11]*z[2]*z[6]*z[8]*z[9] + 127999999999999996160*z[1]*z[11]*z[2]*z[6]*z[8] + 15999999999999999520*z[1]*z[11]*z[2]*z[6]*z[9] + 3999999999999999520000000000000007200*z[1]*z[11]*z[2]*z[6] + 512*z[1]*z[11]*z[2]*z[7]*z[8]*z[9] + 255999999999999992320*z[1]*z[11]*z[2]*z[7]*z[8] + 31999999999999999040*z[1]*z[11]*z[2]*z[7]*z[9] + 7999999999999999040000000000000014400*z[1]*z[11]*z[2]*z[7] + 63999999999999998080*z[1]*z[11]*z[2]*z[8]*z[9] + 15999999999999998080000000000000028800*z[1]*z[11]*z[2]*z[8] + 1999999999999999880000000000000002480*z[1]*z[11]*z[2]*z[9] - 59999999999999996960000000000000037200*z[1]*z[11]*z[2] + 64*z[1]*z[11]*z[3]*z[5]*z[6]*z[9] + 31999999999999999040*z[1]*z[11]*z[3]*z[5]*z[6] + 128*z[1]*z[11]*z[3]*z[5]*z[7]*z[9] + 63999999999999998080*z[1]*z[11]*z[3]*z[5]*z[7] + 256*z[1]*z[11]*z[3]*z[5]*z[8]*z[9] + 127999999999999996160*z[1]*z[11]*z[3]*z[5]*z[8] + 15999999999999999520*z[1]*z[11]*z[3]*z[5]*z[9] + 3999999999999999520000000000000007200*z[1]*z[11]*z[3]*z[5] + 256*z[1]*z[11]*z[3]*z[6]*z[7]*z[9] + 127999999999999996160*z[1]*z[11]*z[3]*z[6]*z[7] + 512*z[1]*z[11]*z[3]*z[6]*z[8]*z[9] + 255999999999999992320*z[1]*z[11]*z[3]*z[6]*z[8] + 31999999999999999040*z[1]*z[11]*z[3]*z[6]*z[9] + 7999999999999999040000000000000014400*z[1]*z[11]*z[3]*z[6] + 1024*z[1]*z[11]*z[3]*z[7]*z[8]*z[9] + 511999999999999984640*z[1]*z[11]*z[3]*z[7]*z[8] + 63999999999999998080*z[1]*z[11]*z[3]*z[7]*z[9] + 15999999999999998080000000000000028800*z[1]*z[11]*z[3]*z[7] + 127999999999999996160*z[1]*z[11]*z[3]*z[8]*z[9] + 31999999999999996160000000000000057600*z[1]*z[11]*z[3]*z[8] + 3999999999999999760000000000000004960*z[1]*z[11]*z[3]*z[9] - 119999999999999993920000000000000074400*z[1]*z[11]*z[3] + 128*z[1]*z[11]*z[4]*z[5]*z[6]*z[9] + 63999999999999998080*z[1]*z[11]*z[4]*z[5]*z[6] + 256*z[1]*z[11]*z[4]*z[5]*z[7]*z[9] + 127999999999999996160*z[1]*z[11]*z[4]*z[5]*z[7] + 512*z[1]*z[11]*z[4]*z[5]*z[8]*z[9] + 255999999999999992320*z[1]*z[11]*z[4]*z[5]*z[8] + 31999999999999999040*z[1]*z[11]*z[4]*z[5]*z[9] + 7999999999999999040000000000000014400*z[1]*z[11]*z[4]*z[5] + 512*z[1]*z[11]*z[4]*z[6]*z[7]*z[9] + 255999999999999992320*z[1]*z[11]*z[4]*z[6]*z[7] + 1024*z[1]*z[11]*z[4]*z[6]*z[8]*z[9] + 511999999999999984640*z[1]*z[11]*z[4]*z[6]*z[8] + 63999999999999998080*z[1]*z[11]*z[4]*z[6]*z[9] + 15999999999999998080000000000000028800*z[1]*z[11]*z[4]*z[6] + 2048*z[1]*z[11]*z[4]*z[7]*z[8]*z[9] + 1023999999999999969280*z[1]*z[11]*z[4]*z[7]*z[8] + 127999999999999996160*z[1]*z[11]*z[4]*z[7]*z[9] + 31999999999999996160000000000000057600*z[1]*z[11]*z[4]*z[7] + 255999999999999992320*z[1]*z[11]*z[4]*z[8]*z[9] + 63999999999999992320000000000000115200*z[1]*z[11]*z[4]*z[8] + 7999999999999999520000000000000009920*z[1]*z[11]*z[4]*z[9] - 239999999999999987840000000000000148800*z[1]*z[11]*z[4] + 7999999999999999760*z[1]*z[11]*z[5]*z[6]*z[9] + 1999999999999999760000000000000003600*z[1]*z[11]*z[5]*z[6] + 15999999999999999520*z[1]*z[11]*z[5]*z[7]*z[9] + 3999999999999999520000000000000007200*z[1]*z[11]*z[5]*z[7] + 31999999999999999040*z[1]*z[11]*z[5]*z[8]*z[9] + 7999999999999999040000000000000014400*z[1]*z[11]*z[5]*z[8] + 999999999999999880000000000000001800*z[1]*z[11]*z[5]*z[9] - 44999999999999997300000000000000027000*z[1]*z[11]*z[5] + 31999999999999999040*z[1]*z[11]*z[6]*z[7]*z[9] + 7999999999999999040000000000000014400*z[1]*z[11]*z[6]*z[7] + 63999999999999998080*z[1]*z[11]*z[6]*z[8]*z[9] + 15999999999999998080000000000000028800*z[1]*z[11]*z[6]*z[8] + 1999999999999999760000000000000003600*z[1]*z[11]*z[6]*z[9] - 89999999999999994600000000000000054000*z[1]*z[11]*z[6] + 127999999999999996160*z[1]*z[11]*z[7]*z[8]*z[9] + 31999999999999996160000000000000057600*z[1]*z[11]*z[7]*z[8] + 3999999999999999520000000000000007200*z[1]*z[11]*z[7]*z[9] - 179999999999999989200000000000000108000*z[1]*z[11]*z[7] + 7999999999999999040000000000000014400*z[1]*z[11]*z[8]*z[9] - 359999999999999978400000000000000216000*z[1]*z[11]*z[8] - 29999999999999998480000000000000018600*z[1]*z[11]*z[9] + 829999999999999967900000000000000279000*z[1]*z[11] + 64*z[1]*z[12]*z[2]*z[5]*z[6]*z[9] + 31999999999999999040*z[1]*z[12]*z[2]*z[5]*z[6] + 128*z[1]*z[12]*z[2]*z[5]*z[7]*z[9] + 63999999999999998080*z[1]*z[12]*z[2]*z[5]*z[7] + 256*z[1]*z[12]*z[2]*z[5]*z[8]*z[9] + 127999999999999996160*z[1]*z[12]*z[2]*z[5]*z[8] + 15999999999999999520*z[1]*z[12]*z[2]*z[5]*z[9] + 3999999999999999520000000000000007200*z[1]*z[12]*z[2]*z[5] + 256*z[1]*z[12]*z[2]*z[6]*z[7]*z[9] + 127999999999999996160*z[1]*z[12]*z[2]*z[6]*z[7] + 512*z[1]*z[12]*z[2]*z[6]*z[8]*z[9] + 255999999999999992320*z[1]*z[12]*z[2]*z[6]*z[8] + 31999999999999999040*z[1]*z[12]*z[2]*z[6]*z[9] + 7999999999999999040000000000000014400*z[1]*z[12]*z[2]*z[6] + 1024*z[1]*z[12]*z[2]*z[7]*z[8]*z[9] + 511999999999999984640*z[1]*z[12]*z[2]*z[7]*z[8] + 63999999999999998080*z[1]*z[12]*z[2]*z[7]*z[9] + 15999999999999998080000000000000028800*z[1]*z[12]*z[2]*z[7] + 127999999999999996160*z[1]*z[12]*z[2]*z[8]*z[9] + 31999999999999996160000000000000057600*z[1]*z[12]*z[2]*z[8] + 3999999999999999760000000000000004960*z[1]*z[12]*z[2]*z[9] - 119999999999999993920000000000000074400*z[1]*z[12]*z[2] + 128*z[1]*z[12]*z[3]*z[5]*z[6]*z[9] + 63999999999999998080*z[1]*z[12]*z[3]*z[5]*z[6] + 256*z[1]*z[12]*z[3]*z[5]*z[7]*z[9] + 127999999999999996160*z[1]*z[12]*z[3]*z[5]*z[7] + 512*z[1]*z[12]*z[3]*z[5]*z[8]*z[9] + 255999999999999992320*z[1]*z[12]*z[3]*z[5]*z[8] + 31999999999999999040*z[1]*z[12]*z[3]*z[5]*z[9] + 7999999999999999040000000000000014400*z[1]*z[12]*z[3]*z[5] + 512*z[1]*z[12]*z[3]*z[6]*z[7]*z[9] + 255999999999999992320*z[1]*z[12]*z[3]*z[6]*z[7] + 1024*z[1]*z[12]*z[3]*z[6]*z[8]*z[9] + 511999999999999984640*z[1]*z[12]*z[3]*z[6]*z[8] + 63999999999999998080*z[1]*z[12]*z[3]*z[6]*z[9] + 15999999999999998080000000000000028800*z[1]*z[12]*z[3]*z[6] + 2048*z[1]*z[12]*z[3]*z[7]*z[8]*z[9] + 1023999999999999969280*z[1]*z[12]*z[3]*z[7]*z[8] + 127999999999999996160*z[1]*z[12]*z[3]*z[7]*z[9] + 31999999999999996160000000000000057600*z[1]*z[12]*z[3]*z[7] + 255999999999999992320*z[1]*z[12]*z[3]*z[8]*z[9] + 63999999999999992320000000000000115200*z[1]*z[12]*z[3]*z[8] + 7999999999999999520000000000000009920*z[1]*z[12]*z[3]*z[9] - 239999999999999987840000000000000148800*z[1]*z[12]*z[3] + 256*z[1]*z[12]*z[4]*z[5]*z[6]*z[9] + 127999999999999996160*z[1]*z[12]*z[4]*z[5]*z[6] + 512*z[1]*z[12]*z[4]*z[5]*z[7]*z[9] + 255999999999999992320*z[1]*z[12]*z[4]*z[5]*z[7] + 1024*z[1]*z[12]*z[4]*z[5]*z[8]*z[9] + 511999999999999984640*z[1]*z[12]*z[4]*z[5]*z[8] + 63999999999999998080*z[1]*z[12]*z[4]*z[5]*z[9] + 15999999999999998080000000000000028800*z[1]*z[12]*z[4]*z[5] + 1024*z[1]*z[12]*z[4]*z[6]*z[7]*z[9] + 511999999999999984640*z[1]*z[12]*z[4]*z[6]*z[7] + 2048*z[1]*z[12]*z[4]*z[6]*z[8]*z[9] + 1023999999999999969280*z[1]*z[12]*z[4]*z[6]*z[8] + 127999999999999996160*z[1]*z[12]*z[4]*z[6]*z[9] + 31999999999999996160000000000000057600*z[1]*z[12]*z[4]*z[6] + 4096*z[1]*z[12]*z[4]*z[7]*z[8]*z[9] + 2047999999999999938560*z[1]*z[12]*z[4]*z[7]*z[8] + 255999999999999992320*z[1]*z[12]*z[4]*z[7]*z[9] + 63999999999999992320000000000000115200*z[1]*z[12]*z[4]*z[7] + 511999999999999984640*z[1]*z[12]*z[4]*z[8]*z[9] + 127999999999999984640000000000000230400*z[1]*z[12]*z[4]*z[8] + 15999999999999999040000000000000019840*z[1]*z[12]*z[4]*z[9] - 479999999999999975680000000000000297600*z[1]*z[12]*z[4] + 15999999999999999520*z[1]*z[12]*z[5]*z[6]*z[9] + 3999999999999999520000000000000007200*z[1]*z[12]*z[5]*z[6] + 31999999999999999040*z[1]*z[12]*z[5]*z[7]*z[9] + 7999999999999999040000000000000014400*z[1]*z[12]*z[5]*z[7] + 63999999999999998080*z[1]*z[12]*z[5]*z[8]*z[9] + 15999999999999998080000000000000028800*z[1]*z[12]*z[5]*z[8] + 1999999999999999760000000000000003600*z[1]*z[12]*z[5]*z[9] - 89999999999999994600000000000000054000*z[1]*z[12]*z[5] + 63999999999999998080*z[1]*z[12]*z[6]*z[7]*z[9] + 15999999999999998080000000000000028800*z[1]*z[12]*z[6]*z[7] + 127999999999999996160*z[1]*z[12]*z[6]*z[8]*z[9] + 31999999999999996160000000000000057600*z[1]*z[12]*z[6]*z[8] + 3999999999999999520000000000000007200*z[1]*z[12]*z[6]*z[9] - 179999999999999989200000000000000108000*z[1]*z[12]*z[6] + 255999999999999992320*z[1]*z[12]*z[7]*z[8]*z[9] + 63999999999999992320000000000000115200*z[1]*z[12]*z[7]*z[8] + 7999999999999999040000000000000014400*z[1]*z[12]*z[7]*z[9] - 359999999999999978400000000000000216000*z[1]*z[12]*z[7] + 15999999999999998080000000000000028800*z[1]*z[12]*z[8]*z[9] - 719999999999999956800000000000000432000*z[1]*z[12]*z[8] - 59999999999999996960000000000000037200*z[1]*z[12]*z[9] + 1659999999999999935800000000000000558000*z[1]*z[12] + 3999999999999999880*z[1]*z[2]*z[5]*z[6]*z[9] + 999999999999999940000000000000001240*z[1]*z[2]*z[5]*z[6] + 7999999999999999760*z[1]*z[2]*z[5]*z[7]*z[9] + 1999999999999999880000000000000002480*z[1]*z[2]*z[5]*z[7] + 15999999999999999520*z[1]*z[2]*z[5]*z[8]*z[9] + 3999999999999999760000000000000004960*z[1]*z[2]*z[5]*z[8] + 499999999999999940000000000000000900*z[1]*z[2]*z[5]*z[9] - 14999999999999999240000000000000009300*z[1]*z[2]*z[5] + 15999999999999999520*z[1]*z[2]*z[6]*z[7]*z[9] + 3999999999999999760000000000000004960*z[1]*z[2]*z[6]*z[7] + 31999999999999999040*z[1]*z[2]*z[6]*z[8]*z[9] + 7999999999999999520000000000000009920*z[1]*z[2]*z[6]*z[8] + 999999999999999880000000000000001800*z[1]*z[2]*z[6]*z[9] - 29999999999999998480000000000000018600*z[1]*z[2]*z[6] + 63999999999999998080*z[1]*z[2]*z[7]*z[8]*z[9] + 15999999999999999040000000000000019840*z[1]*z[2]*z[7]*z[8] + 1999999999999999760000000000000003600*z[1]*z[2]*z[7]*z[9] - 59999999999999996960000000000000037200*z[1]*z[2]*z[7] + 3999999999999999520000000000000007200*z[1]*z[2]*z[8]*z[9] - 119999999999999993920000000000000074400*z[1]*z[2]*z[8] - 14999999999999999240000000000000009300*z[1]*z[2]*z[9] + 267499999999999990700000000000000096100*z[1]*z[2] + 7999999999999999760*z[1]*z[3]*z[5]*z[6]*z[9] + 1999999999999999880000000000000002480*z[1]*z[3]*z[5]*z[6] + 15999999999999999520*z[1]*z[3]*z[5]*z[7]*z[9] + 3999999999999999760000000000000004960*z[1]*z[3]*z[5]*z[7] + 31999999999999999040*z[1]*z[3]*z[5]*z[8]*z[9] + 7999999999999999520000000000000009920*z[1]*z[3]*z[5]*z[8] + 999999999999999880000000000000001800*z[1]*z[3]*z[5]*z[9] - 29999999999999998480000000000000018600*z[1]*z[3]*z[5] + 31999999999999999040*z[1]*z[3]*z[6]*z[7]*z[9] + 7999999999999999520000000000000009920*z[1]*z[3]*z[6]*z[7] + 63999999999999998080*z[1]*z[3]*z[6]*z[8]*z[9] + 15999999999999999040000000000000019840*z[1]*z[3]*z[6]*z[8] + 1999999999999999760000000000000003600*z[1]*z[3]*z[6]*z[9] - 59999999999999996960000000000000037200*z[1]*z[3]*z[6] + 127999999999999996160*z[1]*z[3]*z[7]*z[8]*z[9] + 31999999999999998080000000000000039680*z[1]*z[3]*z[7]*z[8] + 3999999999999999520000000000000007200*z[1]*z[3]*z[7]*z[9] - 119999999999999993920000000000000074400*z[1]*z[3]*z[7] + 7999999999999999040000000000000014400*z[1]*z[3]*z[8]*z[9] - 239999999999999987840000000000000148800*z[1]*z[3]*z[8] - 29999999999999998480000000000000018600*z[1]*z[3]*z[9] + 534999999999999981400000000000000192200*z[1]*z[3] + 15999999999999999520*z[1]*z[4]*z[5]*z[6]*z[9] + 3999999999999999760000000000000004960*z[1]*z[4]*z[5]*z[6] + 31999999999999999040*z[1]*z[4]*z[5]*z[7]*z[9] + 7999999999999999520000000000000009920*z[1]*z[4]*z[5]*z[7] + 63999999999999998080*z[1]*z[4]*z[5]*z[8]*z[9] + 15999999999999999040000000000000019840*z[1]*z[4]*z[5]*z[8] + 1999999999999999760000000000000003600*z[1]*z[4]*z[5]*z[9] - 59999999999999996960000000000000037200*z[1]*z[4]*z[5] + 63999999999999998080*z[1]*z[4]*z[6]*z[7]*z[9] + 15999999999999999040000000000000019840*z[1]*z[4]*z[6]*z[7] + 127999999999999996160*z[1]*z[4]*z[6]*z[8]*z[9] + 31999999999999998080000000000000039680*z[1]*z[4]*z[6]*z[8] + 3999999999999999520000000000000007200*z[1]*z[4]*z[6]*z[9] - 119999999999999993920000000000000074400*z[1]*z[4]*z[6] + 255999999999999992320*z[1]*z[4]*z[7]*z[8]*z[9] + 63999999999999996160000000000000079360*z[1]*z[4]*z[7]*z[8] + 7999999999999999040000000000000014400*z[1]*z[4]*z[7]*z[9] - 239999999999999987840000000000000148800*z[1]*z[4]*z[7] + 15999999999999998080000000000000028800*z[1]*z[4]*z[8]*z[9] - 479999999999999975680000000000000297600*z[1]*z[4]*z[8] - 59999999999999996960000000000000037200*z[1]*z[4]*z[9] + 1069999999999999962800000000000000384400*z[1]*z[4] + 499999999999999940000000000000000900*z[1]*z[5]*z[6]*z[9] - 14999999999999999240000000000000009300*z[1]*z[5]*z[6] + 999999999999999880000000000000001800*z[1]*z[5]*z[7]*z[9] - 29999999999999998480000000000000018600*z[1]*z[5]*z[7] + 1999999999999999760000000000000003600*z[1]*z[5]*z[8]*z[9] - 59999999999999996960000000000000037200*z[1]*z[5]*z[8] - 11249999999999999325000000000000006750*z[1]*z[5]*z[9] + 207499999999999991975000000000000069750*z[1]*z[5] + 1999999999999999760000000000000003600*z[1]*z[6]*z[7]*z[9] - 59999999999999996960000000000000037200*z[1]*z[6]*z[7] + 3999999999999999520000000000000007200*z[1]*z[6]*z[8]*z[9] - 119999999999999993920000000000000074400*z[1]*z[6]*z[8] - 22499999999999998650000000000000013500*z[1]*z[6]*z[9] + 414999999999999983950000000000000139500*z[1]*z[6] + 7999999999999999040000000000000014400*z[1]*z[7]*z[8]*z[9] - 239999999999999987840000000000000148800*z[1]*z[7]*z[8] - 44999999999999997300000000000000027000*z[1]*z[7]*z[9] + 829999999999999967900000000000000279000*z[1]*z[7] - 89999999999999994600000000000000054000*z[1]*z[8]*z[9] + 1659999999999999935800000000000000558000*z[1]*z[8] + 207499999999999991975000000000000069750*z[1]*z[9] - 3168749999999999906225000000000000720750*z[1] + 256*z[10]*z[11]*z[2]*z[3]*z[5]*z[6] + 512*z[10]*z[11]*z[2]*z[3]*z[5]*z[7] + 1024*z[10]*z[11]*z[2]*z[3]*z[5]*z[8] + 63999999999999998080*z[10]*z[11]*z[2]*z[3]*z[5] + 1024*z[10]*z[11]*z[2]*z[3]*z[6]*z[7] + 2048*z[10]*z[11]*z[2]*z[3]*z[6]*z[8] + 127999999999999996160*z[10]*z[11]*z[2]*z[3]*z[6] + 4096*z[10]*z[11]*z[2]*z[3]*z[7]*z[8] + 255999999999999992320*z[10]*z[11]*z[2]*z[3]*z[7] + 511999999999999984640*z[10]*z[11]*z[2]*z[3]*z[8] + 15999999999999999040000000000000019840*z[10]*z[11]*z[2]*z[3] + 512*z[10]*z[11]*z[2]*z[4]*z[5]*z[6] + 1024*z[10]*z[11]*z[2]*z[4]*z[5]*z[7] + 2048*z[10]*z[11]*z[2]*z[4]*z[5]*z[8] + 127999999999999996160*z[10]*z[11]*z[2]*z[4]*z[5] + 2048*z[10]*z[11]*z[2]*z[4]*z[6]*z[7] + 4096*z[10]*z[11]*z[2]*z[4]*z[6]*z[8] + 255999999999999992320*z[10]*z[11]*z[2]*z[4]*z[6] + 8192*z[10]*z[11]*z[2]*z[4]*z[7]*z[8] + 511999999999999984640*z[10]*z[11]*z[2]*z[4]*z[7] + 1023999999999999969280*z[10]*z[11]*z[2]*z[4]*z[8] + 31999999999999998080000000000000039680*z[10]*z[11]*z[2]*z[4] + 31999999999999999040*z[10]*z[11]*z[2]*z[5]*z[6] + 63999999999999998080*z[10]*z[11]*z[2]*z[5]*z[7] + 127999999999999996160*z[10]*z[11]*z[2]*z[5]*z[8] + 3999999999999999520000000000000007200*z[10]*z[11]*z[2]*z[5] + 127999999999999996160*z[10]*z[11]*z[2]*z[6]*z[7] + 255999999999999992320*z[10]*z[11]*z[2]*z[6]*z[8] + 7999999999999999040000000000000014400*z[10]*z[11]*z[2]*z[6] + 511999999999999984640*z[10]*z[11]*z[2]*z[7]*z[8] + 15999999999999998080000000000000028800*z[10]*z[11]*z[2]*z[7] + 31999999999999996160000000000000057600*z[10]*z[11]*z[2]*z[8] - 119999999999999993920000000000000074400*z[10]*z[11]*z[2] + 1024*z[10]*z[11]*z[3]*z[4]*z[5]*z[6] + 2048*z[10]*z[11]*z[3]*z[4]*z[5]*z[7] + 4096*z[10]*z[11]*z[3]*z[4]*z[5]*z[8] + 255999999999999992320*z[10]*z[11]*z[3]*z[4]*z[5] + 4096*z[10]*z[11]*z[3]*z[4]*z[6]*z[7] + 8192*z[10]*z[11]*z[3]*z[4]*z[6]*z[8] + 511999999999999984640*z[10]*z[11]*z[3]*z[4]*z[6] + 16384*z[10]*z[11]*z[3]*z[4]*z[7]*z[8] + 1023999999999999969280*z[10]*z[11]*z[3]*z[4]*z[7] + 2047999999999999938560*z[10]*z[11]*z[3]*z[4]*z[8] + 63999999999999996160000000000000079360*z[10]*z[11]*z[3]*z[4] + 63999999999999998080*z[10]*z[11]*z[3]*z[5]*z[6] + 127999999999999996160*z[10]*z[11]*z[3]*z[5]*z[7] + 255999999999999992320*z[10]*z[11]*z[3]*z[5]*z[8] + 7999999999999999040000000000000014400*z[10]*z[11]*z[3]*z[5] + 255999999999999992320*z[10]*z[11]*z[3]*z[6]*z[7] + 511999999999999984640*z[10]*z[11]*z[3]*z[6]*z[8] + 15999999999999998080000000000000028800*z[10]*z[11]*z[3]*z[6] + 1023999999999999969280*z[10]*z[11]*z[3]*z[7]*z[8] + 31999999999999996160000000000000057600*z[10]*z[11]*z[3]*z[7] + 63999999999999992320000000000000115200*z[10]*z[11]*z[3]*z[8] - 239999999999999987840000000000000148800*z[10]*z[11]*z[3] + 127999999999999996160*z[10]*z[11]*z[4]*z[5]*z[6] + 255999999999999992320*z[10]*z[11]*z[4]*z[5]*z[7] + 511999999999999984640*z[10]*z[11]*z[4]*z[5]*z[8] + 15999999999999998080000000000000028800*z[10]*z[11]*z[4]*z[5] + 511999999999999984640*z[10]*z[11]*z[4]*z[6]*z[7] + 1023999999999999969280*z[10]*z[11]*z[4]*z[6]*z[8] + 31999999999999996160000000000000057600*z[10]*z[11]*z[4]*z[6] + 2047999999999999938560*z[10]*z[11]*z[4]*z[7]*z[8] + 63999999999999992320000000000000115200*z[10]*z[11]*z[4]*z[7] + 127999999999999984640000000000000230400*z[10]*z[11]*z[4]*z[8] - 479999999999999975680000000000000297600*z[10]*z[11]*z[4] + 3999999999999999760000000000000004960*z[10]*z[11]*z[5]*z[6] + 7999999999999999520000000000000009920*z[10]*z[11]*z[5]*z[7] + 15999999999999999040000000000000019840*z[10]*z[11]*z[5]*z[8] - 59999999999999996960000000000000037200*z[10]*z[11]*z[5] + 15999999999999999040000000000000019840*z[10]*z[11]*z[6]*z[7] + 31999999999999998080000000000000039680*z[10]*z[11]*z[6]*z[8] - 119999999999999993920000000000000074400*z[10]*z[11]*z[6] + 63999999999999996160000000000000079360*z[10]*z[11]*z[7]*z[8] - 239999999999999987840000000000000148800*z[10]*z[11]*z[7] - 479999999999999975680000000000000297600*z[10]*z[11]*z[8] + 1069999999999999962800000000000000384400*z[10]*z[11] + 512*z[10]*z[12]*z[2]*z[3]*z[5]*z[6] + 1024*z[10]*z[12]*z[2]*z[3]*z[5]*z[7] + 2048*z[10]*z[12]*z[2]*z[3]*z[5]*z[8] + 127999999999999996160*z[10]*z[12]*z[2]*z[3]*z[5] + 2048*z[10]*z[12]*z[2]*z[3]*z[6]*z[7] + 4096*z[10]*z[12]*z[2]*z[3]*z[6]*z[8] + 255999999999999992320*z[10]*z[12]*z[2]*z[3]*z[6] + 8192*z[10]*z[12]*z[2]*z[3]*z[7]*z[8] + 511999999999999984640*z[10]*z[12]*z[2]*z[3]*z[7] + 1023999999999999969280*z[10]*z[12]*z[2]*z[3]*z[8] + 31999999999999998080000000000000039680*z[10]*z[12]*z[2]*z[3] + 1024*z[10]*z[12]*z[2]*z[4]*z[5]*z[6] + 2048*z[10]*z[12]*z[2]*z[4]*z[5]*z[7] + 4096*z[10]*z[12]*z[2]*z[4]*z[5]*z[8] + 255999999999999992320*z[10]*z[12]*z[2]*z[4]*z[5] + 4096*z[10]*z[12]*z[2]*z[4]*z[6]*z[7] + 8192*z[10]*z[12]*z[2]*z[4]*z[6]*z[8] + 511999999999999984640*z[10]*z[12]*z[2]*z[4]*z[6] + 16384*z[10]*z[12]*z[2]*z[4]*z[7]*z[8] + 1023999999999999969280*z[10]*z[12]*z[2]*z[4]*z[7] + 2047999999999999938560*z[10]*z[12]*z[2]*z[4]*z[8] + 63999999999999996160000000000000079360*z[10]*z[12]*z[2]*z[4] + 63999999999999998080*z[10]*z[12]*z[2]*z[5]*z[6] + 127999999999999996160*z[10]*z[12]*z[2]*z[5]*z[7] + 255999999999999992320*z[10]*z[12]*z[2]*z[5]*z[8] + 7999999999999999040000000000000014400*z[10]*z[12]*z[2]*z[5] + 255999999999999992320*z[10]*z[12]*z[2]*z[6]*z[7] + 511999999999999984640*z[10]*z[12]*z[2]*z[6]*z[8] + 15999999999999998080000000000000028800*z[10]*z[12]*z[2]*z[6] + 1023999999999999969280*z[10]*z[12]*z[2]*z[7]*z[8] + 31999999999999996160000000000000057600*z[10]*z[12]*z[2]*z[7] + 63999999999999992320000000000000115200*z[10]*z[12]*z[2]*z[8] - 239999999999999987840000000000000148800*z[10]*z[12]*z[2] + 2048*z[10]*z[12]*z[3]*z[4]*z[5]*z[6] + 4096*z[10]*z[12]*z[3]*z[4]*z[5]*z[7] + 8192*z[10]*z[12]*z[3]*z[4]*z[5]*z[8] + 511999999999999984640*z[10]*z[12]*z[3]*z[4]*z[5] + 8192*z[10]*z[12]*z[3]*z[4]*z[6]*z[7] + 16384*z[10]*z[12]*z[3]*z[4]*z[6]*z[8] + 1023999999999999969280*z[10]*z[12]*z[3]*z[4]*z[6] + 32768*z[10]*z[12]*z[3]*z[4]*z[7]*z[8] + 2047999999999999938560*z[10]*z[12]*z[3]*z[4]*z[7] + 4095999999999999877120*z[10]*z[12]*z[3]*z[4]*z[8] + 127999999999999992320000000000000158720*z[10]*z[12]*z[3]*z[4] + 127999999999999996160*z[10]*z[12]*z[3]*z[5]*z[6] + 255999999999999992320*z[10]*z[12]*z[3]*z[5]*z[7] + 511999999999999984640*z[10]*z[12]*z[3]*z[5]*z[8] + 15999999999999998080000000000000028800*z[10]*z[12]*z[3]*z[5] + 511999999999999984640*z[10]*z[12]*z[3]*z[6]*z[7] + 1023999999999999969280*z[10]*z[12]*z[3]*z[6]*z[8] + 31999999999999996160000000000000057600*z[10]*z[12]*z[3]*z[6] + 2047999999999999938560*z[10]*z[12]*z[3]*z[7]*z[8] + 63999999999999992320000000000000115200*z[10]*z[12]*z[3]*z[7] + 127999999999999984640000000000000230400*z[10]*z[12]*z[3]*z[8] - 479999999999999975680000000000000297600*z[10]*z[12]*z[3] + 255999999999999992320*z[10]*z[12]*z[4]*z[5]*z[6] + 511999999999999984640*z[10]*z[12]*z[4]*z[5]*z[7] + 1023999999999999969280*z[10]*z[12]*z[4]*z[5]*z[8] + 31999999999999996160000000000000057600*z[10]*z[12]*z[4]*z[5] + 1023999999999999969280*z[10]*z[12]*z[4]*z[6]*z[7] + 2047999999999999938560*z[10]*z[12]*z[4]*z[6]*z[8] + 63999999999999992320000000000000115200*z[10]*z[12]*z[4]*z[6] + 4095999999999999877120*z[10]*z[12]*z[4]*z[7]*z[8] + 127999999999999984640000000000000230400*z[10]*z[12]*z[4]*z[7] + 255999999999999969280000000000000460800*z[10]*z[12]*z[4]*z[8] - 959999999999999951360000000000000595200*z[10]*z[12]*z[4] + 7999999999999999520000000000000009920*z[10]*z[12]*z[5]*z[6] + 15999999999999999040000000000000019840*z[10]*z[12]*z[5]*z[7] + 31999999999999998080000000000000039680*z[10]*z[12]*z[5]*z[8] - 119999999999999993920000000000000074400*z[10]*z[12]*z[5] + 31999999999999998080000000000000039680*z[10]*z[12]*z[6]*z[7] + 63999999999999996160000000000000079360*z[10]*z[12]*z[6]*z[8] - 239999999999999987840000000000000148800*z[10]*z[12]*z[6] + 127999999999999992320000000000000158720*z[10]*z[12]*z[7]*z[8] - 479999999999999975680000000000000297600*z[10]*z[12]*z[7] - 959999999999999951360000000000000595200*z[10]*z[12]*z[8] + 2139999999999999925600000000000000768800*z[10]*z[12] + 64*z[10]*z[2]*z[3]*z[5]*z[6]*z[9] + 31999999999999999040*z[10]*z[2]*z[3]*z[5]*z[6] + 128*z[10]*z[2]*z[3]*z[5]*z[7]*z[9] + 63999999999999998080*z[10]*z[2]*z[3]*z[5]*z[7] + 256*z[10]*z[2]*z[3]*z[5]*z[8]*z[9] + 127999999999999996160*z[10]*z[2]*z[3]*z[5]*z[8] + 15999999999999999520*z[10]*z[2]*z[3]*z[5]*z[9] + 3999999999999999520000000000000007200*z[10]*z[2]*z[3]*z[5] + 256*z[10]*z[2]*z[3]*z[6]*z[7]*z[9] + 127999999999999996160*z[10]*z[2]*z[3]*z[6]*z[7] + 512*z[10]*z[2]*z[3]*z[6]*z[8]*z[9] + 255999999999999992320*z[10]*z[2]*z[3]*z[6]*z[8] + 31999999999999999040*z[10]*z[2]*z[3]*z[6]*z[9] + 7999999999999999040000000000000014400*z[10]*z[2]*z[3]*z[6] + 1024*z[10]*z[2]*z[3]*z[7]*z[8]*z[9] + 511999999999999984640*z[10]*z[2]*z[3]*z[7]*z[8] + 63999999999999998080*z[10]*z[2]*z[3]*z[7]*z[9] + 15999999999999998080000000000000028800*z[10]*z[2]*z[3]*z[7] + 127999999999999996160*z[10]*z[2]*z[3]*z[8]*z[9] + 31999999999999996160000000000000057600*z[10]*z[2]*z[3]*z[8] + 3999999999999999760000000000000004960*z[10]*z[2]*z[3]*z[9] - 119999999999999993920000000000000074400*z[10]*z[2]*z[3] + 128*z[10]*z[2]*z[4]*z[5]*z[6]*z[9] + 63999999999999998080*z[10]*z[2]*z[4]*z[5]*z[6] + 256*z[10]*z[2]*z[4]*z[5]*z[7]*z[9] + 127999999999999996160*z[10]*z[2]*z[4]*z[5]*z[7] + 512*z[10]*z[2]*z[4]*z[5]*z[8]*z[9] + 255999999999999992320*z[10]*z[2]*z[4]*z[5]*z[8] + 31999999999999999040*z[10]*z[2]*z[4]*z[5]*z[9] + 7999999999999999040000000000000014400*z[10]*z[2]*z[4]*z[5] + 512*z[10]*z[2]*z[4]*z[6]*z[7]*z[9] + 255999999999999992320*z[10]*z[2]*z[4]*z[6]*z[7] + 1024*z[10]*z[2]*z[4]*z[6]*z[8]*z[9] + 511999999999999984640*z[10]*z[2]*z[4]*z[6]*z[8] + 63999999999999998080*z[10]*z[2]*z[4]*z[6]*z[9] + 15999999999999998080000000000000028800*z[10]*z[2]*z[4]*z[6] + 2048*z[10]*z[2]*z[4]*z[7]*z[8]*z[9] + 1023999999999999969280*z[10]*z[2]*z[4]*z[7]*z[8] + 127999999999999996160*z[10]*z[2]*z[4]*z[7]*z[9] + 31999999999999996160000000000000057600*z[10]*z[2]*z[4]*z[7] + 255999999999999992320*z[10]*z[2]*z[4]*z[8]*z[9] + 63999999999999992320000000000000115200*z[10]*z[2]*z[4]*z[8] + 7999999999999999520000000000000009920*z[10]*z[2]*z[4]*z[9] - 239999999999999987840000000000000148800*z[10]*z[2]*z[4] + 7999999999999999760*z[10]*z[2]*z[5]*z[6]*z[9] + 1999999999999999760000000000000003600*z[10]*z[2]*z[5]*z[6] + 15999999999999999520*z[10]*z[2]*z[5]*z[7]*z[9] + 3999999999999999520000000000000007200*z[10]*z[2]*z[5]*z[7] + 31999999999999999040*z[10]*z[2]*z[5]*z[8]*z[9] + 7999999999999999040000000000000014400*z[10]*z[2]*z[5]*z[8] + 999999999999999880000000000000001800*z[10]*z[2]*z[5]*z[9] - 44999999999999997300000000000000027000*z[10]*z[2]*z[5] + 31999999999999999040*z[10]*z[2]*z[6]*z[7]*z[9] + 7999999999999999040000000000000014400*z[10]*z[2]*z[6]*z[7] + 63999999999999998080*z[10]*z[2]*z[6]*z[8]*z[9] + 15999999999999998080000000000000028800*z[10]*z[2]*z[6]*z[8] + 1999999999999999760000000000000003600*z[10]*z[2]*z[6]*z[9] - 89999999999999994600000000000000054000*z[10]*z[2]*z[6] + 127999999999999996160*z[10]*z[2]*z[7]*z[8]*z[9] + 31999999999999996160000000000000057600*z[10]*z[2]*z[7]*z[8] + 3999999999999999520000000000000007200*z[10]*z[2]*z[7]*z[9] - 179999999999999989200000000000000108000*z[10]*z[2]*z[7] + 7999999999999999040000000000000014400*z[10]*z[2]*z[8]*z[9] - 359999999999999978400000000000000216000*z[10]*z[2]*z[8] - 29999999999999998480000000000000018600*z[10]*z[2]*z[9] + 829999999999999967900000000000000279000*z[10]*z[2] + 256*z[10]*z[3]*z[4]*z[5]*z[6]*z[9] + 127999999999999996160*z[10]*z[3]*z[4]*z[5]*z[6] + 512*z[10]*z[3]*z[4]*z[5]*z[7]*z[9] + 255999999999999992320*z[10]*z[3]*z[4]*z[5]*z[7] + 1024*z[10]*z[3]*z[4]*z[5]*z[8]*z[9] + 511999999999999984640*z[10]*z[3]*z[4]*z[5]*z[8] + 63999999999999998080*z[10]*z[3]*z[4]*z[5]*z[9] + 15999999999999998080000000000000028800*z[10]*z[3]*z[4]*z[5] + 1024*z[10]*z[3]*z[4]*z[6]*z[7]*z[9] + 511999999999999984640*z[10]*z[3]*z[4]*z[6]*z[7] + 2048*z[10]*z[3]*z[4]*z[6]*z[8]*z[9] + 1023999999999999969280*z[10]*z[3]*z[4]*z[6]*z[8] + 127999999999999996160*z[10]*z[3]*z[4]*z[6]*z[9] + 31999999999999996160000000000000057600*z[10]*z[3]*z[4]*z[6] + 4096*z[10]*z[3]*z[4]*z[7]*z[8]*z[9] + 2047999999999999938560*z[10]*z[3]*z[4]*z[7]*z[8] + 255999999999999992320*z[10]*z[3]*z[4]*z[7]*z[9] + 63999999999999992320000000000000115200*z[10]*z[3]*z[4]*z[7] + 511999999999999984640*z[10]*z[3]*z[4]*z[8]*z[9] + 127999999999999984640000000000000230400*z[10]*z[3]*z[4]*z[8] + 15999999999999999040000000000000019840*z[10]*z[3]*z[4]*z[9] - 479999999999999975680000000000000297600*z[10]*z[3]*z[4] + 15999999999999999520*z[10]*z[3]*z[5]*z[6]*z[9] + 3999999999999999520000000000000007200*z[10]*z[3]*z[5]*z[6] + 31999999999999999040*z[10]*z[3]*z[5]*z[7]*z[9] + 7999999999999999040000000000000014400*z[10]*z[3]*z[5]*z[7] + 63999999999999998080*z[10]*z[3]*z[5]*z[8]*z[9] + 15999999999999998080000000000000028800*z[10]*z[3]*z[5]*z[8] + 1999999999999999760000000000000003600*z[10]*z[3]*z[5]*z[9] - 89999999999999994600000000000000054000*z[10]*z[3]*z[5] + 63999999999999998080*z[10]*z[3]*z[6]*z[7]*z[9] + 15999999999999998080000000000000028800*z[10]*z[3]*z[6]*z[7] + 127999999999999996160*z[10]*z[3]*z[6]*z[8]*z[9] + 31999999999999996160000000000000057600*z[10]*z[3]*z[6]*z[8] + 3999999999999999520000000000000007200*z[10]*z[3]*z[6]*z[9] - 179999999999999989200000000000000108000*z[10]*z[3]*z[6] + 255999999999999992320*z[10]*z[3]*z[7]*z[8]*z[9] + 63999999999999992320000000000000115200*z[10]*z[3]*z[7]*z[8] + 7999999999999999040000000000000014400*z[10]*z[3]*z[7]*z[9] - 359999999999999978400000000000000216000*z[10]*z[3]*z[7] + 15999999999999998080000000000000028800*z[10]*z[3]*z[8]*z[9] - 719999999999999956800000000000000432000*z[10]*z[3]*z[8] - 59999999999999996960000000000000037200*z[10]*z[3]*z[9] + 1659999999999999935800000000000000558000*z[10]*z[3] + 31999999999999999040*z[10]*z[4]*z[5]*z[6]*z[9] + 7999999999999999040000000000000014400*z[10]*z[4]*z[5]*z[6] + 63999999999999998080*z[10]*z[4]*z[5]*z[7]*z[9] + 15999999999999998080000000000000028800*z[10]*z[4]*z[5]*z[7] + 127999999999999996160*z[10]*z[4]*z[5]*z[8]*z[9] + 31999999999999996160000000000000057600*z[10]*z[4]*z[5]*z[8] + 3999999999999999520000000000000007200*z[10]*z[4]*z[5]*z[9] - 179999999999999989200000000000000108000*z[10]*z[4]*z[5] + 127999999999999996160*z[10]*z[4]*z[6]*z[7]*z[9] + 31999999999999996160000000000000057600*z[10]*z[4]*z[6]*z[7] + 255999999999999992320*z[10]*z[4]*z[6]*z[8]*z[9] + 63999999999999992320000000000000115200*z[10]*z[4]*z[6]*z[8] + 7999999999999999040000000000000014400*z[10]*z[4]*z[6]*z[9] - 359999999999999978400000000000000216000*z[10]*z[4]*z[6] + 511999999999999984640*z[10]*z[4]*z[7]*z[8]*z[9] + 127999999999999984640000000000000230400*z[10]*z[4]*z[7]*z[8] + 15999999999999998080000000000000028800*z[10]*z[4]*z[7]*z[9] - 719999999999999956800000000000000432000*z[10]*z[4]*z[7] + 31999999999999996160000000000000057600*z[10]*z[4]*z[8]*z[9] - 1439999999999999913600000000000000864000*z[10]*z[4]*z[8] - 119999999999999993920000000000000074400*z[10]*z[4]*z[9] + 3319999999999999871600000000000001116000*z[10]*z[4] + 999999999999999940000000000000001240*z[10]*z[5]*z[6]*z[9] - 29999999999999998480000000000000018600*z[10]*z[5]*z[6] + 1999999999999999880000000000000002480*z[10]*z[5]*z[7]*z[9] - 59999999999999996960000000000000037200*z[10]*z[5]*z[7] + 3999999999999999760000000000000004960*z[10]*z[5]*z[8]*z[9] - 119999999999999993920000000000000074400*z[10]*z[5]*z[8] - 14999999999999999240000000000000009300*z[10]*z[5]*z[9] + 414999999999999983950000000000000139500*z[10]*z[5] + 3999999999999999760000000000000004960*z[10]*z[6]*z[7]*z[9] - 119999999999999993920000000000000074400*z[10]*z[6]*z[7] + 7999999999999999520000000000000009920*z[10]*z[6]*z[8]*z[9] - 239999999999999987840000000000000148800*z[10]*z[6]*z[8] - 29999999999999998480000000000000018600*z[10]*z[6]*z[9] + 829999999999999967900000000000000279000*z[10]*z[6] + 15999999999999999040000000000000019840*z[10]*z[7]*z[8]*z[9] - 479999999999999975680000000000000297600*z[10]*z[7]*z[8] - 59999999999999996960000000000000037200*z[10]*z[7]*z[9] + 1659999999999999935800000000000000558000*z[10]*z[7] - 119999999999999993920000000000000074400*z[10]*z[8]*z[9] + 3319999999999999871600000000000001116000*z[10]*z[8] + 267499999999999990700000000000000096100*z[10]*z[9] - 6337499999999999812450000000000001441500*z[10] + 1024*z[11]*z[12]*z[2]*z[3]*z[5]*z[6] + 2048*z[11]*z[12]*z[2]*z[3]*z[5]*z[7] + 4096*z[11]*z[12]*z[2]*z[3]*z[5]*z[8] + 255999999999999992320*z[11]*z[12]*z[2]*z[3]*z[5] + 4096*z[11]*z[12]*z[2]*z[3]*z[6]*z[7] + 8192*z[11]*z[12]*z[2]*z[3]*z[6]*z[8] + 511999999999999984640*z[11]*z[12]*z[2]*z[3]*z[6] + 16384*z[11]*z[12]*z[2]*z[3]*z[7]*z[8] + 1023999999999999969280*z[11]*z[12]*z[2]*z[3]*z[7] + 2047999999999999938560*z[11]*z[12]*z[2]*z[3]*z[8] + 63999999999999996160000000000000079360*z[11]*z[12]*z[2]*z[3] + 2048*z[11]*z[12]*z[2]*z[4]*z[5]*z[6] + 4096*z[11]*z[12]*z[2]*z[4]*z[5]*z[7] + 8192*z[11]*z[12]*z[2]*z[4]*z[5]*z[8] + 511999999999999984640*z[11]*z[12]*z[2]*z[4]*z[5] + 8192*z[11]*z[12]*z[2]*z[4]*z[6]*z[7] + 16384*z[11]*z[12]*z[2]*z[4]*z[6]*z[8] + 1023999999999999969280*z[11]*z[12]*z[2]*z[4]*z[6] + 32768*z[11]*z[12]*z[2]*z[4]*z[7]*z[8] + 2047999999999999938560*z[11]*z[12]*z[2]*z[4]*z[7] + 4095999999999999877120*z[11]*z[12]*z[2]*z[4]*z[8] + 127999999999999992320000000000000158720*z[11]*z[12]*z[2]*z[4] + 127999999999999996160*z[11]*z[12]*z[2]*z[5]*z[6] + 255999999999999992320*z[11]*z[12]*z[2]*z[5]*z[7] + 511999999999999984640*z[11]*z[12]*z[2]*z[5]*z[8] + 15999999999999998080000000000000028800*z[11]*z[12]*z[2]*z[5] + 511999999999999984640*z[11]*z[12]*z[2]*z[6]*z[7] + 1023999999999999969280*z[11]*z[12]*z[2]*z[6]*z[8] + 31999999999999996160000000000000057600*z[11]*z[12]*z[2]*z[6] + 2047999999999999938560*z[11]*z[12]*z[2]*z[7]*z[8] + 63999999999999992320000000000000115200*z[11]*z[12]*z[2]*z[7] + 127999999999999984640000000000000230400*z[11]*z[12]*z[2]*z[8] - 479999999999999975680000000000000297600*z[11]*z[12]*z[2] + 4096*z[11]*z[12]*z[3]*z[4]*z[5]*z[6] + 8192*z[11]*z[12]*z[3]*z[4]*z[5]*z[7] + 16384*z[11]*z[12]*z[3]*z[4]*z[5]*z[8] + 1023999999999999969280*z[11]*z[12]*z[3]*z[4]*z[5] + 16384*z[11]*z[12]*z[3]*z[4]*z[6]*z[7] + 32768*z[11]*z[12]*z[3]*z[4]*z[6]*z[8] + 2047999999999999938560*z[11]*z[12]*z[3]*z[4]*z[6] + 65536*z[11]*z[12]*z[3]*z[4]*z[7]*z[8] + 4095999999999999877120*z[11]*z[12]*z[3]*z[4]*z[7] + 8191999999999999754240*z[11]*z[12]*z[3]*z[4]*z[8] + 255999999999999984640000000000000317440*z[11]*z[12]*z[3]*z[4] + 255999999999999992320*z[11]*z[12]*z[3]*z[5]*z[6] + 511999999999999984640*z[11]*z[12]*z[3]*z[5]*z[7] + 1023999999999999969280*z[11]*z[12]*z[3]*z[5]*z[8] + 31999999999999996160000000000000057600*z[11]*z[12]*z[3]*z[5] + 1023999999999999969280*z[11]*z[12]*z[3]*z[6]*z[7] + 2047999999999999938560*z[11]*z[12]*z[3]*z[6]*z[8] + 63999999999999992320000000000000115200*z[11]*z[12]*z[3]*z[6] + 4095999999999999877120*z[11]*z[12]*z[3]*z[7]*z[8] + 127999999999999984640000000000000230400*z[11]*z[12]*z[3]*z[7] + 255999999999999969280000000000000460800*z[11]*z[12]*z[3]*z[8] - 959999999999999951360000000000000595200*z[11]*z[12]*z[3] + 511999999999999984640*z[11]*z[12]*z[4]*z[5]*z[6] + 1023999999999999969280*z[11]*z[12]*z[4]*z[5]*z[7] + 2047999999999999938560*z[11]*z[12]*z[4]*z[5]*z[8] + 63999999999999992320000000000000115200*z[11]*z[12]*z[4]*z[5] + 2047999999999999938560*z[11]*z[12]*z[4]*z[6]*z[7] + 4095999999999999877120*z[11]*z[12]*z[4]*z[6]*z[8] + 127999999999999984640000000000000230400*z[11]*z[12]*z[4]*z[6] + 8191999999999999754240*z[11]*z[12]*z[4]*z[7]*z[8] + 255999999999999969280000000000000460800*z[11]*z[12]*z[4]*z[7] + 511999999999999938560000000000000921600*z[11]*z[12]*z[4]*z[8] - 1919999999999999902720000000000001190400*z[11]*z[12]*z[4] + 15999999999999999040000000000000019840*z[11]*z[12]*z[5]*z[6] + 31999999999999998080000000000000039680*z[11]*z[12]*z[5]*z[7] + 63999999999999996160000000000000079360*z[11]*z[12]*z[5]*z[8] - 239999999999999987840000000000000148800*z[11]*z[12]*z[5] + 63999999999999996160000000000000079360*z[11]*z[12]*z[6]*z[7] + 127999999999999992320000000000000158720*z[11]*z[12]*z[6]*z[8] - 479999999999999975680000000000000297600*z[11]*z[12]*z[6] + 255999999999999984640000000000000317440*z[11]*z[12]*z[7]*z[8] - 959999999999999951360000000000000595200*z[11]*z[12]*z[7] - 1919999999999999902720000000000001190400*z[11]*z[12]*z[8] + 4279999999999999851200000000000001537600*z[11]*z[12] + 128*z[11]*z[2]*z[3]*z[5]*z[6]*z[9] + 63999999999999998080*z[11]*z[2]*z[3]*z[5]*z[6] + 256*z[11]*z[2]*z[3]*z[5]*z[7]*z[9] + 127999999999999996160*z[11]*z[2]*z[3]*z[5]*z[7] + 512*z[11]*z[2]*z[3]*z[5]*z[8]*z[9] + 255999999999999992320*z[11]*z[2]*z[3]*z[5]*z[8] + 31999999999999999040*z[11]*z[2]*z[3]*z[5]*z[9] + 7999999999999999040000000000000014400*z[11]*z[2]*z[3]*z[5] + 512*z[11]*z[2]*z[3]*z[6]*z[7]*z[9] + 255999999999999992320*z[11]*z[2]*z[3]*z[6]*z[7] + 1024*z[11]*z[2]*z[3]*z[6]*z[8]*z[9] + 511999999999999984640*z[11]*z[2]*z[3]*z[6]*z[8] + 63999999999999998080*z[11]*z[2]*z[3]*z[6]*z[9] + 15999999999999998080000000000000028800*z[11]*z[2]*z[3]*z[6] + 2048*z[11]*z[2]*z[3]*z[7]*z[8]*z[9] + 1023999999999999969280*z[11]*z[2]*z[3]*z[7]*z[8] + 127999999999999996160*z[11]*z[2]*z[3]*z[7]*z[9] + 31999999999999996160000000000000057600*z[11]*z[2]*z[3]*z[7] + 255999999999999992320*z[11]*z[2]*z[3]*z[8]*z[9] + 63999999999999992320000000000000115200*z[11]*z[2]*z[3]*z[8] + 7999999999999999520000000000000009920*z[11]*z[2]*z[3]*z[9] - 239999999999999987840000000000000148800*z[11]*z[2]*z[3] + 256*z[11]*z[2]*z[4]*z[5]*z[6]*z[9] + 127999999999999996160*z[11]*z[2]*z[4]*z[5]*z[6] + 512*z[11]*z[2]*z[4]*z[5]*z[7]*z[9] + 255999999999999992320*z[11]*z[2]*z[4]*z[5]*z[7] + 1024*z[11]*z[2]*z[4]*z[5]*z[8]*z[9] + 511999999999999984640*z[11]*z[2]*z[4]*z[5]*z[8] + 63999999999999998080*z[11]*z[2]*z[4]*z[5]*z[9] + 15999999999999998080000000000000028800*z[11]*z[2]*z[4]*z[5] + 1024*z[11]*z[2]*z[4]*z[6]*z[7]*z[9] + 511999999999999984640*z[11]*z[2]*z[4]*z[6]*z[7] + 2048*z[11]*z[2]*z[4]*z[6]*z[8]*z[9] + 1023999999999999969280*z[11]*z[2]*z[4]*z[6]*z[8] + 127999999999999996160*z[11]*z[2]*z[4]*z[6]*z[9] + 31999999999999996160000000000000057600*z[11]*z[2]*z[4]*z[6] + 4096*z[11]*z[2]*z[4]*z[7]*z[8]*z[9] + 2047999999999999938560*z[11]*z[2]*z[4]*z[7]*z[8] + 255999999999999992320*z[11]*z[2]*z[4]*z[7]*z[9] + 63999999999999992320000000000000115200*z[11]*z[2]*z[4]*z[7] + 511999999999999984640*z[11]*z[2]*z[4]*z[8]*z[9] + 127999999999999984640000000000000230400*z[11]*z[2]*z[4]*z[8] + 15999999999999999040000000000000019840*z[11]*z[2]*z[4]*z[9] - 479999999999999975680000000000000297600*z[11]*z[2]*z[4] + 15999999999999999520*z[11]*z[2]*z[5]*z[6]*z[9] + 3999999999999999520000000000000007200*z[11]*z[2]*z[5]*z[6] + 31999999999999999040*z[11]*z[2]*z[5]*z[7]*z[9] + 7999999999999999040000000000000014400*z[11]*z[2]*z[5]*z[7] + 63999999999999998080*z[11]*z[2]*z[5]*z[8]*z[9] + 15999999999999998080000000000000028800*z[11]*z[2]*z[5]*z[8] + 1999999999999999760000000000000003600*z[11]*z[2]*z[5]*z[9] - 89999999999999994600000000000000054000*z[11]*z[2]*z[5] + 63999999999999998080*z[11]*z[2]*z[6]*z[7]*z[9] + 15999999999999998080000000000000028800*z[11]*z[2]*z[6]*z[7] + 127999999999999996160*z[11]*z[2]*z[6]*z[8]*z[9] + 31999999999999996160000000000000057600*z[11]*z[2]*z[6]*z[8] + 3999999999999999520000000000000007200*z[11]*z[2]*z[6]*z[9] - 179999999999999989200000000000000108000*z[11]*z[2]*z[6] + 255999999999999992320*z[11]*z[2]*z[7]*z[8]*z[9] + 63999999999999992320000000000000115200*z[11]*z[2]*z[7]*z[8] + 7999999999999999040000000000000014400*z[11]*z[2]*z[7]*z[9] - 359999999999999978400000000000000216000*z[11]*z[2]*z[7] + 15999999999999998080000000000000028800*z[11]*z[2]*z[8]*z[9] - 719999999999999956800000000000000432000*z[11]*z[2]*z[8] - 59999999999999996960000000000000037200*z[11]*z[2]*z[9] + 1659999999999999935800000000000000558000*z[11]*z[2] + 512*z[11]*z[3]*z[4]*z[5]*z[6]*z[9] + 255999999999999992320*z[11]*z[3]*z[4]*z[5]*z[6] + 1024*z[11]*z[3]*z[4]*z[5]*z[7]*z[9] + 511999999999999984640*z[11]*z[3]*z[4]*z[5]*z[7] + 2048*z[11]*z[3]*z[4]*z[5]*z[8]*z[9] + 1023999999999999969280*z[11]*z[3]*z[4]*z[5]*z[8] + 127999999999999996160*z[11]*z[3]*z[4]*z[5]*z[9] + 31999999999999996160000000000000057600*z[11]*z[3]*z[4]*z[5] + 2048*z[11]*z[3]*z[4]*z[6]*z[7]*z[9] + 1023999999999999969280*z[11]*z[3]*z[4]*z[6]*z[7] + 4096*z[11]*z[3]*z[4]*z[6]*z[8]*z[9] + 2047999999999999938560*z[11]*z[3]*z[4]*z[6]*z[8] + 255999999999999992320*z[11]*z[3]*z[4]*z[6]*z[9] + 63999999999999992320000000000000115200*z[11]*z[3]*z[4]*z[6] + 8192*z[11]*z[3]*z[4]*z[7]*z[8]*z[9] + 4095999999999999877120*z[11]*z[3]*z[4]*z[7]*z[8] + 511999999999999984640*z[11]*z[3]*z[4]*z[7]*z[9] + 127999999999999984640000000000000230400*z[11]*z[3]*z[4]*z[7] + 1023999999999999969280*z[11]*z[3]*z[4]*z[8]*z[9] + 255999999999999969280000000000000460800*z[11]*z[3]*z[4]*z[8] + 31999999999999998080000000000000039680*z[11]*z[3]*z[4]*z[9] - 959999999999999951360000000000000595200*z[11]*z[3]*z[4] + 31999999999999999040*z[11]*z[3]*z[5]*z[6]*z[9] + 7999999999999999040000000000000014400*z[11]*z[3]*z[5]*z[6] + 63999999999999998080*z[11]*z[3]*z[5]*z[7]*z[9] + 15999999999999998080000000000000028800*z[11]*z[3]*z[5]*z[7] + 127999999999999996160*z[11]*z[3]*z[5]*z[8]*z[9] + 31999999999999996160000000000000057600*z[11]*z[3]*z[5]*z[8] + 3999999999999999520000000000000007200*z[11]*z[3]*z[5]*z[9] - 179999999999999989200000000000000108000*z[11]*z[3]*z[5] + 127999999999999996160*z[11]*z[3]*z[6]*z[7]*z[9] + 31999999999999996160000000000000057600*z[11]*z[3]*z[6]*z[7] + 255999999999999992320*z[11]*z[3]*z[6]*z[8]*z[9] + 63999999999999992320000000000000115200*z[11]*z[3]*z[6]*z[8] + 7999999999999999040000000000000014400*z[11]*z[3]*z[6]*z[9] - 359999999999999978400000000000000216000*z[11]*z[3]*z[6] + 511999999999999984640*z[11]*z[3]*z[7]*z[8]*z[9] + 127999999999999984640000000000000230400*z[11]*z[3]*z[7]*z[8] + 15999999999999998080000000000000028800*z[11]*z[3]*z[7]*z[9] - 719999999999999956800000000000000432000*z[11]*z[3]*z[7] + 31999999999999996160000000000000057600*z[11]*z[3]*z[8]*z[9] - 1439999999999999913600000000000000864000*z[11]*z[3]*z[8] - 119999999999999993920000000000000074400*z[11]*z[3]*z[9] + 3319999999999999871600000000000001116000*z[11]*z[3] + 63999999999999998080*z[11]*z[4]*z[5]*z[6]*z[9] + 15999999999999998080000000000000028800*z[11]*z[4]*z[5]*z[6] + 127999999999999996160*z[11]*z[4]*z[5]*z[7]*z[9] + 31999999999999996160000000000000057600*z[11]*z[4]*z[5]*z[7] + 255999999999999992320*z[11]*z[4]*z[5]*z[8]*z[9] + 63999999999999992320000000000000115200*z[11]*z[4]*z[5]*z[8] + 7999999999999999040000000000000014400*z[11]*z[4]*z[5]*z[9] - 359999999999999978400000000000000216000*z[11]*z[4]*z[5] + 255999999999999992320*z[11]*z[4]*z[6]*z[7]*z[9] + 63999999999999992320000000000000115200*z[11]*z[4]*z[6]*z[7] + 511999999999999984640*z[11]*z[4]*z[6]*z[8]*z[9] + 127999999999999984640000000000000230400*z[11]*z[4]*z[6]*z[8] + 15999999999999998080000000000000028800*z[11]*z[4]*z[6]*z[9] - 719999999999999956800000000000000432000*z[11]*z[4]*z[6] + 1023999999999999969280*z[11]*z[4]*z[7]*z[8]*z[9] + 255999999999999969280000000000000460800*z[11]*z[4]*z[7]*z[8] + 31999999999999996160000000000000057600*z[11]*z[4]*z[7]*z[9] - 1439999999999999913600000000000000864000*z[11]*z[4]*z[7] + 63999999999999992320000000000000115200*z[11]*z[4]*z[8]*z[9] - 2879999999999999827200000000000001728000*z[11]*z[4]*z[8] - 239999999999999987840000000000000148800*z[11]*z[4]*z[9] + 6639999999999999743200000000000002232000*z[11]*z[4] + 1999999999999999880000000000000002480*z[11]*z[5]*z[6]*z[9] - 59999999999999996960000000000000037200*z[11]*z[5]*z[6] + 3999999999999999760000000000000004960*z[11]*z[5]*z[7]*z[9] - 119999999999999993920000000000000074400*z[11]*z[5]*z[7] + 7999999999999999520000000000000009920*z[11]*z[5]*z[8]*z[9] - 239999999999999987840000000000000148800*z[11]*z[5]*z[8] - 29999999999999998480000000000000018600*z[11]*z[5]*z[9] + 829999999999999967900000000000000279000*z[11]*z[5] + 7999999999999999520000000000000009920*z[11]*z[6]*z[7]*z[9] - 239999999999999987840000000000000148800*z[11]*z[6]*z[7] + 15999999999999999040000000000000019840*z[11]*z[6]*z[8]*z[9] - 479999999999999975680000000000000297600*z[11]*z[6]*z[8] - 59999999999999996960000000000000037200*z[11]*z[6]*z[9] + 1659999999999999935800000000000000558000*z[11]*z[6] + 31999999999999998080000000000000039680*z[11]*z[7]*z[8]*z[9] - 959999999999999951360000000000000595200*z[11]*z[7]*z[8] - 119999999999999993920000000000000074400*z[11]*z[7]*z[9] + 3319999999999999871600000000000001116000*z[11]*z[7] - 239999999999999987840000000000000148800*z[11]*z[8]*z[9] + 6639999999999999743200000000000002232000*z[11]*z[8] + 534999999999999981400000000000000192200*z[11]*z[9] - 12674999999999999624900000000000002883000*z[11] + 256*z[12]*z[2]*z[3]*z[5]*z[6]*z[9] + 127999999999999996160*z[12]*z[2]*z[3]*z[5]*z[6] + 512*z[12]*z[2]*z[3]*z[5]*z[7]*z[9] + 255999999999999992320*z[12]*z[2]*z[3]*z[5]*z[7] + 1024*z[12]*z[2]*z[3]*z[5]*z[8]*z[9] + 511999999999999984640*z[12]*z[2]*z[3]*z[5]*z[8] + 63999999999999998080*z[12]*z[2]*z[3]*z[5]*z[9] + 15999999999999998080000000000000028800*z[12]*z[2]*z[3]*z[5] + 1024*z[12]*z[2]*z[3]*z[6]*z[7]*z[9] + 511999999999999984640*z[12]*z[2]*z[3]*z[6]*z[7] + 2048*z[12]*z[2]*z[3]*z[6]*z[8]*z[9] + 1023999999999999969280*z[12]*z[2]*z[3]*z[6]*z[8] + 127999999999999996160*z[12]*z[2]*z[3]*z[6]*z[9] + 31999999999999996160000000000000057600*z[12]*z[2]*z[3]*z[6] + 4096*z[12]*z[2]*z[3]*z[7]*z[8]*z[9] + 2047999999999999938560*z[12]*z[2]*z[3]*z[7]*z[8] + 255999999999999992320*z[12]*z[2]*z[3]*z[7]*z[9] + 63999999999999992320000000000000115200*z[12]*z[2]*z[3]*z[7] + 511999999999999984640*z[12]*z[2]*z[3]*z[8]*z[9] + 127999999999999984640000000000000230400*z[12]*z[2]*z[3]*z[8] + 15999999999999999040000000000000019840*z[12]*z[2]*z[3]*z[9] - 479999999999999975680000000000000297600*z[12]*z[2]*z[3] + 512*z[12]*z[2]*z[4]*z[5]*z[6]*z[9] + 255999999999999992320*z[12]*z[2]*z[4]*z[5]*z[6] + 1024*z[12]*z[2]*z[4]*z[5]*z[7]*z[9] + 511999999999999984640*z[12]*z[2]*z[4]*z[5]*z[7] + 2048*z[12]*z[2]*z[4]*z[5]*z[8]*z[9] + 1023999999999999969280*z[12]*z[2]*z[4]*z[5]*z[8] + 127999999999999996160*z[12]*z[2]*z[4]*z[5]*z[9] + 31999999999999996160000000000000057600*z[12]*z[2]*z[4]*z[5] + 2048*z[12]*z[2]*z[4]*z[6]*z[7]*z[9] + 1023999999999999969280*z[12]*z[2]*z[4]*z[6]*z[7] + 4096*z[12]*z[2]*z[4]*z[6]*z[8]*z[9] + 2047999999999999938560*z[12]*z[2]*z[4]*z[6]*z[8] + 255999999999999992320*z[12]*z[2]*z[4]*z[6]*z[9] + 63999999999999992320000000000000115200*z[12]*z[2]*z[4]*z[6] + 8192*z[12]*z[2]*z[4]*z[7]*z[8]*z[9] + 4095999999999999877120*z[12]*z[2]*z[4]*z[7]*z[8] + 511999999999999984640*z[12]*z[2]*z[4]*z[7]*z[9] + 127999999999999984640000000000000230400*z[12]*z[2]*z[4]*z[7] + 1023999999999999969280*z[12]*z[2]*z[4]*z[8]*z[9] + 255999999999999969280000000000000460800*z[12]*z[2]*z[4]*z[8] + 31999999999999998080000000000000039680*z[12]*z[2]*z[4]*z[9] - 959999999999999951360000000000000595200*z[12]*z[2]*z[4] + 31999999999999999040*z[12]*z[2]*z[5]*z[6]*z[9] + 7999999999999999040000000000000014400*z[12]*z[2]*z[5]*z[6] + 63999999999999998080*z[12]*z[2]*z[5]*z[7]*z[9] + 15999999999999998080000000000000028800*z[12]*z[2]*z[5]*z[7] + 127999999999999996160*z[12]*z[2]*z[5]*z[8]*z[9] + 31999999999999996160000000000000057600*z[12]*z[2]*z[5]*z[8] + 3999999999999999520000000000000007200*z[12]*z[2]*z[5]*z[9] - 179999999999999989200000000000000108000*z[12]*z[2]*z[5] + 127999999999999996160*z[12]*z[2]*z[6]*z[7]*z[9] + 31999999999999996160000000000000057600*z[12]*z[2]*z[6]*z[7] + 255999999999999992320*z[12]*z[2]*z[6]*z[8]*z[9] + 63999999999999992320000000000000115200*z[12]*z[2]*z[6]*z[8] + 7999999999999999040000000000000014400*z[12]*z[2]*z[6]*z[9] - 359999999999999978400000000000000216000*z[12]*z[2]*z[6] + 511999999999999984640*z[12]*z[2]*z[7]*z[8]*z[9] + 127999999999999984640000000000000230400*z[12]*z[2]*z[7]*z[8] + 15999999999999998080000000000000028800*z[12]*z[2]*z[7]*z[9] - 719999999999999956800000000000000432000*z[12]*z[2]*z[7] + 31999999999999996160000000000000057600*z[12]*z[2]*z[8]*z[9] - 1439999999999999913600000000000000864000*z[12]*z[2]*z[8] - 119999999999999993920000000000000074400*z[12]*z[2]*z[9] + 3319999999999999871600000000000001116000*z[12]*z[2] + 1024*z[12]*z[3]*z[4]*z[5]*z[6]*z[9] + 511999999999999984640*z[12]*z[3]*z[4]*z[5]*z[6] + 2048*z[12]*z[3]*z[4]*z[5]*z[7]*z[9] + 1023999999999999969280*z[12]*z[3]*z[4]*z[5]*z[7] + 4096*z[12]*z[3]*z[4]*z[5]*z[8]*z[9] + 2047999999999999938560*z[12]*z[3]*z[4]*z[5]*z[8] + 255999999999999992320*z[12]*z[3]*z[4]*z[5]*z[9] + 63999999999999992320000000000000115200*z[12]*z[3]*z[4]*z[5] + 4096*z[12]*z[3]*z[4]*z[6]*z[7]*z[9] + 2047999999999999938560*z[12]*z[3]*z[4]*z[6]*z[7] + 8192*z[12]*z[3]*z[4]*z[6]*z[8]*z[9] + 4095999999999999877120*z[12]*z[3]*z[4]*z[6]*z[8] + 511999999999999984640*z[12]*z[3]*z[4]*z[6]*z[9] + 127999999999999984640000000000000230400*z[12]*z[3]*z[4]*z[6] + 16384*z[12]*z[3]*z[4]*z[7]*z[8]*z[9] + 8191999999999999754240*z[12]*z[3]*z[4]*z[7]*z[8] + 1023999999999999969280*z[12]*z[3]*z[4]*z[7]*z[9] + 255999999999999969280000000000000460800*z[12]*z[3]*z[4]*z[7] + 2047999999999999938560*z[12]*z[3]*z[4]*z[8]*z[9] + 511999999999999938560000000000000921600*z[12]*z[3]*z[4]*z[8] + 63999999999999996160000000000000079360*z[12]*z[3]*z[4]*z[9] - 1919999999999999902720000000000001190400*z[12]*z[3]*z[4] + 63999999999999998080*z[12]*z[3]*z[5]*z[6]*z[9] + 15999999999999998080000000000000028800*z[12]*z[3]*z[5]*z[6] + 127999999999999996160*z[12]*z[3]*z[5]*z[7]*z[9] + 31999999999999996160000000000000057600*z[12]*z[3]*z[5]*z[7] + 255999999999999992320*z[12]*z[3]*z[5]*z[8]*z[9] + 63999999999999992320000000000000115200*z[12]*z[3]*z[5]*z[8] + 7999999999999999040000000000000014400*z[12]*z[3]*z[5]*z[9] - 359999999999999978400000000000000216000*z[12]*z[3]*z[5] + 255999999999999992320*z[12]*z[3]*z[6]*z[7]*z[9] + 63999999999999992320000000000000115200*z[12]*z[3]*z[6]*z[7] + 511999999999999984640*z[12]*z[3]*z[6]*z[8]*z[9] + 127999999999999984640000000000000230400*z[12]*z[3]*z[6]*z[8] + 15999999999999998080000000000000028800*z[12]*z[3]*z[6]*z[9] - 719999999999999956800000000000000432000*z[12]*z[3]*z[6] + 1023999999999999969280*z[12]*z[3]*z[7]*z[8]*z[9] + 255999999999999969280000000000000460800*z[12]*z[3]*z[7]*z[8] + 31999999999999996160000000000000057600*z[12]*z[3]*z[7]*z[9] - 1439999999999999913600000000000000864000*z[12]*z[3]*z[7] + 63999999999999992320000000000000115200*z[12]*z[3]*z[8]*z[9] - 2879999999999999827200000000000001728000*z[12]*z[3]*z[8] - 239999999999999987840000000000000148800*z[12]*z[3]*z[9] + 6639999999999999743200000000000002232000*z[12]*z[3] + 127999999999999996160*z[12]*z[4]*z[5]*z[6]*z[9] + 31999999999999996160000000000000057600*z[12]*z[4]*z[5]*z[6] + 255999999999999992320*z[12]*z[4]*z[5]*z[7]*z[9] + 63999999999999992320000000000000115200*z[12]*z[4]*z[5]*z[7] + 511999999999999984640*z[12]*z[4]*z[5]*z[8]*z[9] + 127999999999999984640000000000000230400*z[12]*z[4]*z[5]*z[8] + 15999999999999998080000000000000028800*z[12]*z[4]*z[5]*z[9] - 719999999999999956800000000000000432000*z[12]*z[4]*z[5] + 511999999999999984640*z[12]*z[4]*z[6]*z[7]*z[9] + 127999999999999984640000000000000230400*z[12]*z[4]*z[6]*z[7] + 1023999999999999969280*z[12]*z[4]*z[6]*z[8]*z[9] + 255999999999999969280000000000000460800*z[12]*z[4]*z[6]*z[8] + 31999999999999996160000000000000057600*z[12]*z[4]*z[6]*z[9] - 1439999999999999913600000000000000864000*z[12]*z[4]*z[6] + 2047999999999999938560*z[12]*z[4]*z[7]*z[8]*z[9] + 511999999999999938560000000000000921600*z[12]*z[4]*z[7]*z[8] + 63999999999999992320000000000000115200*z[12]*z[4]*z[7]*z[9] - 2879999999999999827200000000000001728000*z[12]*z[4]*z[7] + 127999999999999984640000000000000230400*z[12]*z[4]*z[8]*z[9] - 5759999999999999654400000000000003456000*z[12]*z[4]*z[8] - 479999999999999975680000000000000297600*z[12]*z[4]*z[9] + 13279999999999999486400000000000004464000*z[12]*z[4] + 3999999999999999760000000000000004960*z[12]*z[5]*z[6]*z[9] - 119999999999999993920000000000000074400*z[12]*z[5]*z[6] + 7999999999999999520000000000000009920*z[12]*z[5]*z[7]*z[9] - 239999999999999987840000000000000148800*z[12]*z[5]*z[7] + 15999999999999999040000000000000019840*z[12]*z[5]*z[8]*z[9] - 479999999999999975680000000000000297600*z[12]*z[5]*z[8] - 59999999999999996960000000000000037200*z[12]*z[5]*z[9] + 1659999999999999935800000000000000558000*z[12]*z[5] + 15999999999999999040000000000000019840*z[12]*z[6]*z[7]*z[9] - 479999999999999975680000000000000297600*z[12]*z[6]*z[7] + 31999999999999998080000000000000039680*z[12]*z[6]*z[8]*z[9] - 959999999999999951360000000000000595200*z[12]*z[6]*z[8] - 119999999999999993920000000000000074400*z[12]*z[6]*z[9] + 3319999999999999871600000000000001116000*z[12]*z[6] + 63999999999999996160000000000000079360*z[12]*z[7]*z[8]*z[9] - 1919999999999999902720000000000001190400*z[12]*z[7]*z[8] - 239999999999999987840000000000000148800*z[12]*z[7]*z[9] + 6639999999999999743200000000000002232000*z[12]*z[7] - 479999999999999975680000000000000297600*z[12]*z[8]*z[9] + 13279999999999999486400000000000004464000*z[12]*z[8] + 1069999999999999962800000000000000384400*z[12]*z[9] - 25349999999999999249800000000000005766000*z[12] + 15999999999999999520*z[2]*z[3]*z[5]*z[6]*z[9] + 3999999999999999760000000000000004960*z[2]*z[3]*z[5]*z[6] + 31999999999999999040*z[2]*z[3]*z[5]*z[7]*z[9] + 7999999999999999520000000000000009920*z[2]*z[3]*z[5]*z[7] + 63999999999999998080*z[2]*z[3]*z[5]*z[8]*z[9] + 15999999999999999040000000000000019840*z[2]*z[3]*z[5]*z[8] + 1999999999999999760000000000000003600*z[2]*z[3]*z[5]*z[9] - 59999999999999996960000000000000037200*z[2]*z[3]*z[5] + 63999999999999998080*z[2]*z[3]*z[6]*z[7]*z[9] + 15999999999999999040000000000000019840*z[2]*z[3]*z[6]*z[7] + 127999999999999996160*z[2]*z[3]*z[6]*z[8]*z[9] + 31999999999999998080000000000000039680*z[2]*z[3]*z[6]*z[8] + 3999999999999999520000000000000007200*z[2]*z[3]*z[6]*z[9] - 119999999999999993920000000000000074400*z[2]*z[3]*z[6] + 255999999999999992320*z[2]*z[3]*z[7]*z[8]*z[9] + 63999999999999996160000000000000079360*z[2]*z[3]*z[7]*z[8] + 7999999999999999040000000000000014400*z[2]*z[3]*z[7]*z[9] - 239999999999999987840000000000000148800*z[2]*z[3]*z[7] + 15999999999999998080000000000000028800*z[2]*z[3]*z[8]*z[9] - 479999999999999975680000000000000297600*z[2]*z[3]*z[8] - 59999999999999996960000000000000037200*z[2]*z[3]*z[9] + 1069999999999999962800000000000000384400*z[2]*z[3] + 31999999999999999040*z[2]*z[4]*z[5]*z[6]*z[9] + 7999999999999999520000000000000009920*z[2]*z[4]*z[5]*z[6] + 63999999999999998080*z[2]*z[4]*z[5]*z[7]*z[9] + 15999999999999999040000000000000019840*z[2]*z[4]*z[5]*z[7] + 127999999999999996160*z[2]*z[4]*z[5]*z[8]*z[9] + 31999999999999998080000000000000039680*z[2]*z[4]*z[5]*z[8] + 3999999999999999520000000000000007200*z[2]*z[4]*z[5]*z[9] - 119999999999999993920000000000000074400*z[2]*z[4]*z[5] + 127999999999999996160*z[2]*z[4]*z[6]*z[7]*z[9] + 31999999999999998080000000000000039680*z[2]*z[4]*z[6]*z[7] + 255999999999999992320*z[2]*z[4]*z[6]*z[8]*z[9] + 63999999999999996160000000000000079360*z[2]*z[4]*z[6]*z[8] + 7999999999999999040000000000000014400*z[2]*z[4]*z[6]*z[9] - 239999999999999987840000000000000148800*z[2]*z[4]*z[6] + 511999999999999984640*z[2]*z[4]*z[7]*z[8]*z[9] + 127999999999999992320000000000000158720*z[2]*z[4]*z[7]*z[8] + 15999999999999998080000000000000028800*z[2]*z[4]*z[7]*z[9] - 479999999999999975680000000000000297600*z[2]*z[4]*z[7] + 31999999999999996160000000000000057600*z[2]*z[4]*z[8]*z[9] - 959999999999999951360000000000000595200*z[2]*z[4]*z[8] - 119999999999999993920000000000000074400*z[2]*z[4]*z[9] + 2139999999999999925600000000000000768800*z[2]*z[4] + 999999999999999880000000000000001800*z[2]*z[5]*z[6]*z[9] - 29999999999999998480000000000000018600*z[2]*z[5]*z[6] + 1999999999999999760000000000000003600*z[2]*z[5]*z[7]*z[9] - 59999999999999996960000000000000037200*z[2]*z[5]*z[7] + 3999999999999999520000000000000007200*z[2]*z[5]*z[8]*z[9] - 119999999999999993920000000000000074400*z[2]*z[5]*z[8] - 22499999999999998650000000000000013500*z[2]*z[5]*z[9] + 414999999999999983950000000000000139500*z[2]*z[5] + 3999999999999999520000000000000007200*z[2]*z[6]*z[7]*z[9] - 119999999999999993920000000000000074400*z[2]*z[6]*z[7] + 7999999999999999040000000000000014400*z[2]*z[6]*z[8]*z[9] - 239999999999999987840000000000000148800*z[2]*z[6]*z[8] - 44999999999999997300000000000000027000*z[2]*z[6]*z[9] + 829999999999999967900000000000000279000*z[2]*z[6] + 15999999999999998080000000000000028800*z[2]*z[7]*z[8]*z[9] - 479999999999999975680000000000000297600*z[2]*z[7]*z[8] - 89999999999999994600000000000000054000*z[2]*z[7]*z[9] + 1659999999999999935800000000000000558000*z[2]*z[7] - 179999999999999989200000000000000108000*z[2]*z[8]*z[9] + 3319999999999999871600000000000001116000*z[2]*z[8] + 414999999999999983950000000000000139500*z[2]*z[9] - 6337499999999999812450000000000001441500*z[2] + 63999999999999998080*z[3]*z[4]*z[5]*z[6]*z[9] + 15999999999999999040000000000000019840*z[3]*z[4]*z[5]*z[6] + 127999999999999996160*z[3]*z[4]*z[5]*z[7]*z[9] + 31999999999999998080000000000000039680*z[3]*z[4]*z[5]*z[7] + 255999999999999992320*z[3]*z[4]*z[5]*z[8]*z[9] + 63999999999999996160000000000000079360*z[3]*z[4]*z[5]*z[8] + 7999999999999999040000000000000014400*z[3]*z[4]*z[5]*z[9] - 239999999999999987840000000000000148800*z[3]*z[4]*z[5] + 255999999999999992320*z[3]*z[4]*z[6]*z[7]*z[9] + 63999999999999996160000000000000079360*z[3]*z[4]*z[6]*z[7] + 511999999999999984640*z[3]*z[4]*z[6]*z[8]*z[9] + 127999999999999992320000000000000158720*z[3]*z[4]*z[6]*z[8] + 15999999999999998080000000000000028800*z[3]*z[4]*z[6]*z[9] - 479999999999999975680000000000000297600*z[3]*z[4]*z[6] + 1023999999999999969280*z[3]*z[4]*z[7]*z[8]*z[9] + 255999999999999984640000000000000317440*z[3]*z[4]*z[7]*z[8] + 31999999999999996160000000000000057600*z[3]*z[4]*z[7]*z[9] - 959999999999999951360000000000000595200*z[3]*z[4]*z[7] + 63999999999999992320000000000000115200*z[3]*z[4]*z[8]*z[9] - 1919999999999999902720000000000001190400*z[3]*z[4]*z[8] - 239999999999999987840000000000000148800*z[3]*z[4]*z[9] + 4279999999999999851200000000000001537600*z[3]*z[4] + 1999999999999999760000000000000003600*z[3]*z[5]*z[6]*z[9] - 59999999999999996960000000000000037200*z[3]*z[5]*z[6] + 3999999999999999520000000000000007200*z[3]*z[5]*z[7]*z[9] - 119999999999999993920000000000000074400*z[3]*z[5]*z[7] + 7999999999999999040000000000000014400*z[3]*z[5]*z[8]*z[9] - 239999999999999987840000000000000148800*z[3]*z[5]*z[8] - 44999999999999997300000000000000027000*z[3]*z[5]*z[9] + 829999999999999967900000000000000279000*z[3]*z[5] + 7999999999999999040000000000000014400*z[3]*z[6]*z[7]*z[9] - 239999999999999987840000000000000148800*z[3]*z[6]*z[7] + 15999999999999998080000000000000028800*z[3]*z[6]*z[8]*z[9] - 479999999999999975680000000000000297600*z[3]*z[6]*z[8] - 89999999999999994600000000000000054000*z[3]*z[6]*z[9] + 1659999999999999935800000000000000558000*z[3]*z[6] + 31999999999999996160000000000000057600*z[3]*z[7]*z[8]*z[9] - 959999999999999951360000000000000595200*z[3]*z[7]*z[8] - 179999999999999989200000000000000108000*z[3]*z[7]*z[9] + 3319999999999999871600000000000001116000*z[3]*z[7] - 359999999999999978400000000000000216000*z[3]*z[8]*z[9] + 6639999999999999743200000000000002232000*z[3]*z[8] + 829999999999999967900000000000000279000*z[3]*z[9] - 12674999999999999624900000000000002883000*z[3] + 3999999999999999520000000000000007200*z[4]*z[5]*z[6]*z[9] - 119999999999999993920000000000000074400*z[4]*z[5]*z[6] + 7999999999999999040000000000000014400*z[4]*z[5]*z[7]*z[9] - 239999999999999987840000000000000148800*z[4]*z[5]*z[7] + 15999999999999998080000000000000028800*z[4]*z[5]*z[8]*z[9] - 479999999999999975680000000000000297600*z[4]*z[5]*z[8] - 89999999999999994600000000000000054000*z[4]*z[5]*z[9] + 1659999999999999935800000000000000558000*z[4]*z[5] + 15999999999999998080000000000000028800*z[4]*z[6]*z[7]*z[9] - 479999999999999975680000000000000297600*z[4]*z[6]*z[7] + 31999999999999996160000000000000057600*z[4]*z[6]*z[8]*z[9] - 959999999999999951360000000000000595200*z[4]*z[6]*z[8] - 179999999999999989200000000000000108000*z[4]*z[6]*z[9] + 3319999999999999871600000000000001116000*z[4]*z[6] + 63999999999999992320000000000000115200*z[4]*z[7]*z[8]*z[9] - 1919999999999999902720000000000001190400*z[4]*z[7]*z[8] - 359999999999999978400000000000000216000*z[4]*z[7]*z[9] + 6639999999999999743200000000000002232000*z[4]*z[7] - 719999999999999956800000000000000432000*z[4]*z[8]*z[9] + 13279999999999999486400000000000004464000*z[4]*z[8] + 1659999999999999935800000000000000558000*z[4]*z[9] - 25349999999999999249800000000000005766000*z[4] - 14999999999999999240000000000000009300*z[5]*z[6]*z[9] + 267499999999999990700000000000000096100*z[5]*z[6] - 29999999999999998480000000000000018600*z[5]*z[7]*z[9] + 534999999999999981400000000000000192200*z[5]*z[7] - 59999999999999996960000000000000037200*z[5]*z[8]*z[9] + 1069999999999999962800000000000000384400*z[5]*z[8] + 207499999999999991975000000000000069750*z[5]*z[9] - 3168749999999999906225000000000000720750*z[5] - 59999999999999996960000000000000037200*z[6]*z[7]*z[9] + 1069999999999999962800000000000000384400*z[6]*z[7] - 119999999999999993920000000000000074400*z[6]*z[8]*z[9] + 2139999999999999925600000000000000768800*z[6]*z[8] + 414999999999999983950000000000000139500*z[6]*z[9] - 6337499999999999812450000000000001441500*z[6] - 239999999999999987840000000000000148800*z[7]*z[8]*z[9] + 4279999999999999851200000000000001537600*z[7]*z[8] + 829999999999999967900000000000000279000*z[7]*z[9] - 12674999999999999624900000000000002883000*z[7] + 1659999999999999935800000000000000558000*z[8]*z[9] - 25349999999999999249800000000000005766000*z[8] - 3168749999999999906225000000000000720750*z[9] + 44174999999999998918875000000000007447750)

In [93]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 12  # 
p = 1  # QAOA depth

hamiltonian_less_15_1_Erdös_2 = build_cost_hamiltonian_penalty(num_qubits, parse_hamiltonian_expr(paso3_D_Erdös_Straus_less_15_2),10**21)
final_circuit_less_15_1_Erdös_2, result_less_15_1_Erdös_2 = qaoa(num_qubits, hamiltonian_less_15_1_Erdös_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.09634841 1.10292473]
Minimum expectation value: 1.43791201171875e+40


In [105]:
# === RUN === Numbers less than 10 parse_hamiltonian_expr(paso3_D_Cat_less_10)

num_qubits = 12  # 
p = 1  # QAOA depth

hamiltonian_less_15_1_Erdös_2 = build_cost_hamiltonian_penalty(num_qubits, parse_hamiltonian_expr(paso3_D_Erdös_Straus_less_15_2),10**30)
final_circuit_less_15_1_Erdös_2, result_less_15_1_Erdös_2 = qaoa(num_qubits, hamiltonian_less_15_1_Erdös_2, p)

#print(final_circuit.draw('text'))

Optimal parameters (gammas, betas): [0.55992837 1.18689067]
Minimum expectation value: 1.5171000000000001e+40


In [106]:
top_solutions_less_15_2_Erdös = find_best_bitstrings(final_circuit_less_15_1_Erdös_2, hamiltonian_less_15_1_Erdös_2)

Top 5 bitstrings:
Bitstring: 000000100011, Cost: -88314000000000011994244379286037013200896.0000, Count: 1
Bitstring: 000001100001, Cost: -88313999999999992651431265451970217902080.0000, Count: 1
Bitstring: 000001110001, Cost: -88300999999999999130999629331388952477696.0000, Count: 1
Bitstring: 100000000001, Cost: -88286000000000008095333365641031092011008.0000, Count: 3
Bitstring: 100100000001, Cost: -88269000000000000201619360546829841203200.0000, Count: 1


In [107]:
bitstring_to_pm1(top_solutions_less_15_2_Erdös, evaluate_hamiltonian_D_Erdös_Straus_less_15_2)

["Bitstring: ('000000100011', -8.831400000000001e+40, 1), Evaluated cost: 36000000000000000000000000000000000000",
 "Bitstring: ('000000100011', -8.831400000000001e+40, 1), Evaluated cost: 36000000000000000000000000000000000000",
 "Bitstring: ('000000100011', -8.831400000000001e+40, 1), Evaluated cost: 49000000000000000000000000000000000000",
 "Bitstring: ('000000100011', -8.831400000000001e+40, 1), Evaluated cost: 64000000000000000000000000000000000000",
 "Bitstring: ('000000100011', -8.831400000000001e+40, 1), Evaluated cost: 81000000000000000000000000000000000000"]

In [108]:
bitstrings_less_63_D_Equation_2 = []
for i in range(len(top_solutions_less_15_2_Erdös)):
    bitstrings_less_63_D_Equation_2.append(top_solutions_less_15_2_Erdös[i][0])
bitstrings_less_63_D_Equation_2

['000000100011',
 '000001100001',
 '000001110001',
 '100000000001',
 '100100000001']

In [109]:
def poly_D_Erdös_Straus_less_15_2(x,y,z):
    return(expand(D_Erdös_Straus_2(x,y,z)**2))

In [110]:
group_and_convert(bitstrings_less_63_D_Equation_2,4)

[[12, 4, 0], [8, 6, 0], [8, 14, 0], [8, 0, 1], [8, 0, 9]]

In [84]:
group_and_convert(bitstrings_less_63_D_Equation_2,4)

[[13, 0, 0], [9, 0, 0], [8, 8, 0], [12, 0, 8], [8, 10, 0]]